In [ ]:
# ============================================================
# 🌿 LEAF DETECTION RESEARCH
# Notebook 0: Dataset Audit & Metadata Generation
# ============================================================

import os
import glob
import json
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter

print("Environment ready!")

# ============================================================
# REPRODUCIBILITY CONFIGURATION
# ============================================================
# Set LEAF_DATASET_ROOT to the directory containing the two source
# dataset folders. In Kaggle, the default below matches the original
# research environment. For local use, set the environment variable.

import os
from pathlib import Path

DATASET_ROOT = os.environ.get(
    "LEAF_DATASET_ROOT",
    "/kaggle/input/datasets/afsanatusa05/leaf-detection"
)
DATASET_PARENT = os.path.dirname(DATASET_ROOT.rstrip(os.sep))
OUTPUT_ROOT = os.environ.get("LEAF_OUTPUT_DIR", "./outputs")
os.makedirs(OUTPUT_ROOT, exist_ok=True)

print("Dataset root:", DATASET_ROOT)
print("Output root:", OUTPUT_ROOT)


In [ ]:
# List everything inside /kaggle/input

for root, dirs, files in os.walk(DATASET_PARENT):
    level = root.replace(DATASET_PARENT, "").count(os.sep)
    
    if level <= 2:
        print("  " * level + os.path.basename(root) + "/")


In [ ]:
# ============================================================
# Inspect the dataset directory
# ============================================================

dataset_root = DATASET_PARENT

print("Dataset root:", dataset_root)
print("\nContents:\n")

for item in os.listdir(dataset_root):
    path = os.path.join(dataset_root, item)
    
    if os.path.isdir(path):
        print(f"[DIR]  {item}")
    else:
        print(f"[FILE] {item}")

In [ ]:
# Show folder structure up to 3 levels

for root, dirs, files in os.walk(dataset_root):
    level = root.replace(dataset_root, "").count(os.sep)
    
    if level <= 3:
        indent = "    " * level
        print(f"{indent}{os.path.basename(root)}/")

In [ ]:
# ============================================================
# CELL 04: Inspect All Dataset Classes
# ============================================================

for root, dirs, files in os.walk(dataset_root):
    level = root.replace(dataset_root, "").count(os.sep)
    
    if level <= 5:
        indent = "    " * level
        print(f"{indent}{os.path.basename(root)}/")

In [ ]:
# ============================================================
# CELL 05: Count Images in Each Class Folder
# ============================================================

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

folder_stats = []

for root, dirs, files in os.walk(dataset_root):
    image_files = [
        f for f in files
        if Path(f).suffix.lower() in IMAGE_EXTENSIONS
    ]
    
    if image_files:
        folder_stats.append({
            "folder": os.path.basename(root),
            "path": root,
            "image_count": len(image_files)
        })

folder_stats_df = pd.DataFrame(folder_stats)

folder_stats_df = folder_stats_df.sort_values(
    by="image_count",
    ascending=False
).reset_index(drop=True)

print(f"Total folders containing images: {len(folder_stats_df)}")
print(f"Total images: {folder_stats_df['image_count'].sum()}")

display(folder_stats_df)

In [ ]:
# ============================================================
# CELL 06: Image Format and Empty Folder Analysis
# ============================================================

# Count image extensions
extension_counter = Counter()

for root, dirs, files in os.walk(dataset_root):
    for file in files:
        ext = Path(file).suffix.lower()
        
        if ext in IMAGE_EXTENSIONS:
            extension_counter[ext] += 1

print("Image format distribution:")
for ext, count in sorted(extension_counter.items()):
    print(f"{ext:8s} : {count}")

# Find empty folders
empty_folders = []

for root, dirs, files in os.walk(dataset_root):
    has_images = any(
        Path(f).suffix.lower() in IMAGE_EXTENSIONS
        for f in files
    )
    
    if not dirs and not has_images:
        empty_folders.append(root)

print("\nNumber of empty folders:", len(empty_folders))

if empty_folders:
    print("\nEmpty folders:")
    for folder in empty_folders:
        print(folder)

In [ ]:
# ============================================================
# CELL 07: Build Master Folder Audit Table
# ============================================================

folder_records = []

for root, dirs, files in os.walk(dataset_root):
    image_files = [
        f for f in files
        if Path(f).suffix.lower() in IMAGE_EXTENSIONS
    ]
    
    if not image_files:
        continue

    relative_path = os.path.relpath(root, dataset_root)
    path_parts = Path(relative_path).parts
    
    # Identify source dataset
    if "Our research pictures" in path_parts:
        source = "Our Research Pictures"
    elif "Medical plants" in path_parts:
        source = "Medical Plants"
    else:
        source = "Unknown Source"

    folder_name = os.path.basename(root)

    folder_records.append({
        "source": source,
        "folder_name": folder_name,
        "image_count": len(image_files),
        "folder_path": root
    })

folder_audit_df = pd.DataFrame(folder_records)

folder_audit_df = folder_audit_df.sort_values(
    by=["source", "image_count"],
    ascending=[True, False]
).reset_index(drop=True)

print("Total image-containing folders:", len(folder_audit_df))
print("Total images:", folder_audit_df["image_count"].sum())

display(folder_audit_df)

In [ ]:
# ============================================================
# CELL 08: Source-wise Dataset Statistics
# ============================================================

source_summary = (
    folder_audit_df
    .groupby("source")
    .agg(
        number_of_folders=("folder_name", "count"),
        total_images=("image_count", "sum"),
        average_images_per_folder=("image_count", "mean"),
        minimum_images=("image_count", "min"),
        maximum_images=("image_count", "max")
    )
    .reset_index()
)

display(source_summary)

In [ ]:
# ============================================================
# CELL 09: Detect Duplicate Folder Names Across Sources
# ============================================================

duplicate_folder_names = (
    folder_audit_df
    .groupby("folder_name")["source"]
    .nunique()
)

duplicate_folder_names = duplicate_folder_names[
    duplicate_folder_names > 1
]

print("Folder names appearing in multiple sources:")
print()

if len(duplicate_folder_names) == 0:
    print("No duplicate folder names found across sources.")
else:
    for folder_name in duplicate_folder_names.index:
        print(f"• {folder_name}")
        
        matching_rows = folder_audit_df[
            folder_audit_df["folder_name"] == folder_name
        ]
        
        for _, row in matching_rows.iterrows():
            print(
                f"    {row['source']}: "
                f"{row['image_count']} images"
            )


In [ ]:
# ============================================================
# CELL 10: Raw Folder Inventory
# ============================================================

raw_inventory = folder_audit_df[
    ["source", "folder_name", "image_count"]
].copy()

raw_inventory = raw_inventory.sort_values(
    by=["source", "folder_name"]
).reset_index(drop=True)

print("Complete Raw Folder Inventory")
print("=" * 70)

display(raw_inventory)

In [ ]:

# ============================================================
# CELL 11: Preliminary Health Label Detection
# ============================================================

def detect_health(folder_name):
    name = folder_name.lower().strip()
    
    if name.startswith("healthy"):
        return "Healthy"
    
    if name.startswith("disease"):
        return "Diseased"
    
    return "Unknown"


health_audit = folder_audit_df.copy()

health_audit["preliminary_health"] = (
    health_audit["folder_name"]
    .apply(detect_health)
)

health_summary = (
    health_audit
    .groupby(["source", "preliminary_health"])
    .agg(
        folders=("folder_name", "count"),
        images=("image_count", "sum")
    )
    .reset_index()
)

display(health_summary)

In [ ]:
# ============================================================
# CELL 12: Create Label Mapping Template
# ============================================================

label_mapping = folder_audit_df[
    ["source", "folder_name", "image_count"]
].copy()

# Preliminary health label
label_mapping["health"] = (
    label_mapping["folder_name"]
    .apply(detect_health)
)

# These columns will be completed after manual taxonomy verification
label_mapping["species"] = "TO_BE_MAPPED"
label_mapping["category"] = "TO_BE_MAPPED"

label_mapping = label_mapping[
    [
        "source",
        "folder_name",
        "species",
        "category",
        "health",
        "image_count"
    ]
]

display(label_mapping)


In [ ]:
# ============================================================
# CELL 13: Unique Folder Names for Taxonomy Mapping
# ============================================================

unique_folders = (
    label_mapping[
        ["source", "folder_name", "image_count"]
    ]
    .sort_values(["source", "folder_name"])
    .reset_index(drop=True)
)

display(unique_folders)

In [ ]:
# ============================================================
# CELL 14: Species Name Normalization
# ============================================================

# Normalize raw folder names first
label_mapping["folder_name_clean"] = (
    label_mapping["folder_name"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

species_mapping = {

    # -----------------------------
    # Medical Plants
    # -----------------------------
    "Amloki": "Amloki",
    "Orjun": "Arjun",
    "Patorkusi": "Patorkusi",
    "alovera": "Aloe vera",
    "আদা": "Ginger",
    "কালোজিরা": "Black cumin",
    "তুলশী গাছ": "Tulsi",
    "নিম পাতা": "Neem",
    "বাসক": "Vasaka",
    "সজনা গাছ": "Moringa",
    "হলুদ গাছ": "Turmeric",

    # -----------------------------
    # Our Research Pictures
    # -----------------------------
    "healthy kochu pataa": "Kochu",
    "Disease kosu leaf": "Kochu",

    "Healthy বড়ই (boroi)": "Boroi",
    "disease বড়ই (boroi)": "Boroi",

    "Almond Tree": "Almond",
    "Mango tree": "Mango",

    "disease sapodilla(sofeda)": "Sapodilla",
    "healthy sapodilla (sofeda)": "Sapodilla",

    "disease Jacfruits": "Jackfruit",
    "Healthy Jackfruits": "Jackfruit",
    "Jackfruit": "Jackfruit",

    "Healthy banana leaf": "Banana",
    "Disease Banana leaf": "Banana",

    "disease puishak": "Puishak",
    "healthy puishak": "Puishak",

    "disease beal leaf": "Bael",
    "healthy beal leaf": "Bael",

    "Disease Green chili": "Green chili",
    "healthy green chilli": "Green chili",

    "healthy lichi (লিচু)": "Litchi",
    "Disease lichi (লিচু )": "Litchi",

    "Chili or Chilli pepper(Morich)": "Chili",

    "Apple": "Apple",
    "Guava": "Guava",
    "পেঁপে (Papaya)": "Papaya",
    "Potato আলু": "Potato",
    "Lemon": "Lemon",

    "Pomegranate (ডালিম বেদানা)": "Pomegranate",

    # -----------------------------
    # Flower Plants
    # -----------------------------
    "Marigold": "Marigold",
    "Chrysanthemum": "Chrysanthemum",

    # -----------------------------
    # Other Trees
    # -----------------------------
    "Bakul tree": "Bakul",
    "Peepal tree": "Peepal",
}

# Apply normalized mapping
label_mapping["species"] = (
    label_mapping["folder_name_clean"]
    .map(species_mapping)
)

# Check unmapped folders
unmapped = (
    label_mapping[
        label_mapping["species"].isna()
    ]["folder_name_clean"]
    .unique()
)

print("Total folders:", len(label_mapping))
print(
    "Mapped folders:",
    label_mapping["species"].notna().sum()
)
print("Unmapped folders:", len(unmapped))

if len(unmapped) > 0:
    print("\nUnmapped folders:")
    for folder in unmapped:
        print(" -", folder)
else:
    print("\nAll folders mapped successfully! ✓")

display(
    label_mapping[
        [
            "source",
            "folder_name",
            "species",
            "category",
            "health",
            "image_count"
        ]
    ]
)


In [ ]:
# ============================================================
# CELL 15: Finalize Health Labels
# ============================================================

# Based on the original data-collection protocol,
# folders without an explicit disease label are treated as Healthy.

label_mapping["health"] = (
    label_mapping["health"]
    .replace("Unknown", "Healthy")
)

print("Final health label distribution:")
print("=" * 50)

health_counts = (
    label_mapping
    .groupby("health")["image_count"]
    .sum()
    .sort_values(ascending=False)
)

display(health_counts.to_frame("image_count"))

In [ ]:
# ============================================================
# CELL 16: Plant Category Mapping
# ============================================================

category_mapping = {

    # --------------------------------------------------------
    # MEDICINAL PLANTS
    # --------------------------------------------------------
    "Aloe vera": "Medicinal",
    "Tulsi": "Medicinal",
    "Vasaka": "Medicinal",
    "Neem": "Medicinal",
    "Moringa": "Medicinal",
    "Turmeric": "Medicinal",
    "Amloki": "Medicinal",
    "Patorkusi": "Medicinal",
    "Black cumin": "Medicinal",
    "Arjun": "Medicinal",
    "Ginger": "Medicinal",

    # --------------------------------------------------------
    # FRUIT PLANTS
    # --------------------------------------------------------
    "Boroi": "Fruit",
    "Almond": "Fruit",
    "Mango": "Fruit",
    "Sapodilla": "Fruit",
    "Jackfruit": "Fruit",
    "Banana": "Fruit",
    "Litchi": "Fruit",
    "Apple": "Fruit",
    "Guava": "Fruit",
    "Papaya": "Fruit",
    "Lemon": "Fruit",
    "Pomegranate": "Fruit",
    "Bael": "Fruit",

    # --------------------------------------------------------
    # VEGETABLE / CROP PLANTS
    # --------------------------------------------------------
    "Kochu": "Vegetable/Crop",
    "Puishak": "Vegetable/Crop",
    "Chili": "Vegetable/Crop",
    "Green chili": "Vegetable/Crop",
    "Potato": "Vegetable/Crop",

    # --------------------------------------------------------
    # FLOWER / ORNAMENTAL PLANTS
    # --------------------------------------------------------
    "Marigold": "Flower/Ornamental",
    "Chrysanthemum": "Flower/Ornamental",

    # --------------------------------------------------------
    # OTHER TREES
    # --------------------------------------------------------
    "Bakul": "Flower/Ornamental",
    "Peepal": "Medicinal",
}


# Apply category mapping
label_mapping["category"] = (
    label_mapping["species"].map(category_mapping)
)


# Check mapping completeness
unmapped_categories = (
    label_mapping[
        label_mapping["category"].isna()
    ]["species"]
    .unique()
)


print("Total folders:", len(label_mapping))
print(
    "Mapped categories:",
    label_mapping["category"].notna().sum()
)
print(
    "Unmapped categories:",
    len(unmapped_categories)
)

if len(unmapped_categories) > 0:
    print("\nSpecies without category:")
    for species in unmapped_categories:
        print(" -", species)
else:
    print("\nAll species categorized successfully! ✓")


# Display final label table
display(
    label_mapping[
        [
            "source",
            "folder_name",
            "species",
            "category",
            "health",
            "image_count"
        ]
    ]
)

In [ ]:
# ============================================================
# CELL 17: Category Distribution
# ============================================================

category_counts = (
    label_mapping
    .groupby("category")["image_count"]
    .sum()
    .sort_values(ascending=False)
)

print("Final category distribution:")
print("=" * 50)

display(
    category_counts.to_frame(name="image_count")
)

print("\nTotal images:", category_counts.sum())
print("Total categories:", len(category_counts))

In [ ]:
# ============================================================
# CELL 18: Species-wise Dataset Distribution
# ============================================================

species_counts = (
    label_mapping
    .groupby("species")["image_count"]
    .sum()
    .sort_values(ascending=False)
)

print("Species-wise image distribution:")
print("=" * 50)

display(
    species_counts.to_frame(name="image_count")
)

print("\nTotal species:", len(species_counts))
print("Total images:", species_counts.sum())

print("\nSmallest species classes:")
display(
    species_counts
    .sort_values()
    .head(10)
    .to_frame(name="image_count")
)

print("\nLargest species classes:")
display(
    species_counts
    .sort_values(ascending=False)
    .head(10)
    .to_frame(name="image_count")
)

In [ ]:
# ============================================================
# CELL 19: Species-wise Health Distribution
# ============================================================

species_health_distribution = (
    label_mapping
    .pivot_table(
        index="species",
        columns="health",
        values="image_count",
        aggfunc="sum",
        fill_value=0
    )
)

# Ensure both health columns exist
for col in ["Healthy", "Diseased"]:
    if col not in species_health_distribution.columns:
        species_health_distribution[col] = 0

# Keep a consistent column order
species_health_distribution = species_health_distribution[
    ["Healthy", "Diseased"]
]

# Add total images per species
species_health_distribution["Total"] = (
    species_health_distribution["Healthy"]
    + species_health_distribution["Diseased"]
)

# Sort by total images
species_health_distribution = (
    species_health_distribution
    .sort_values("Total", ascending=False)
)

print("Species-wise health distribution:")
print("=" * 60)

display(species_health_distribution)

print("\nTotal Healthy:",
      species_health_distribution["Healthy"].sum())

print("Total Diseased:",
      species_health_distribution["Diseased"].sum())

print("Total Images:",
      species_health_distribution["Total"].sum())

In [ ]:
# ============================================================
# CELL 20: Image Quality Audit
# ============================================================

from PIL import Image
import os
import pandas as pd

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp",
    ".tif", ".tiff", ".webp"
}

BASE_PATH = DATASET_ROOT

quality_records = []
corrupted_images = []
missing_folders = []


# ------------------------------------------------------------
# Scan all mapped folders
# ------------------------------------------------------------

for _, row in label_mapping.iterrows():

    source = row["source"]
    folder_name = row["folder_name"]

    # Find source directory
    source_candidates = [
        os.path.join(BASE_PATH, source),
        os.path.join(
            BASE_PATH,
            "Our Research Pictures-20260712T062632Z-2-001",
            "Our Research Pictures"
        ) if source == "Our Research Pictures" else None,
        os.path.join(
            BASE_PATH,
            "Medical plants-20260712T062633Z-2-001",
            "Medical plants"
        ) if source == "Medical Plants" else None
    ]

    source_path = None

    for candidate in source_candidates:

        if candidate is not None and os.path.isdir(candidate):
            source_path = candidate
            break

    if source_path is None:
        missing_folders.append(
            f"Source not found: {source}"
        )
        continue


    # Actual image folder
    folder_path = os.path.join(
        source_path,
        folder_name
    )

    if not os.path.isdir(folder_path):

        missing_folders.append(
            folder_path
        )

        continue


    # --------------------------------------------------------
    # Scan images
    # --------------------------------------------------------

    for file_name in os.listdir(folder_path):

        file_path = os.path.join(
            folder_path,
            file_name
        )

        ext = os.path.splitext(
            file_name
        )[1].lower()

        if ext not in IMAGE_EXTENSIONS:
            continue

        try:

            # Check image integrity
            with Image.open(file_path) as img:
                img.verify()


            # Read image properties
            with Image.open(file_path) as img:

                width, height = img.size
                mode = img.mode


            quality_records.append({

                "file_path": file_path,
                "source": source,
                "folder_name": folder_name,
                "species": row["species"],
                "category": row["category"],
                "health": row["health"],
                "width": width,
                "height": height,
                "mode": mode,
                "extension": ext

            })


        except Exception as e:

            corrupted_images.append({

                "file_path": file_path,
                "error": str(e)

            })


# ------------------------------------------------------------
# Create DataFrames
# ------------------------------------------------------------

quality_df = pd.DataFrame(
    quality_records
)

corrupted_df = pd.DataFrame(
    corrupted_images
)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("IMAGE QUALITY AUDIT")
print("=" * 60)

print(
    "Expected images :",
    label_mapping["image_count"].sum()
)

print(
    "Readable images :",
    len(quality_df)
)

print(
    "Corrupted images:",
    len(corrupted_df)
)

print(
    "Missing folders :",
    len(missing_folders)
)


# ------------------------------------------------------------
# Image modes
# ------------------------------------------------------------

if len(quality_df) > 0:

    print("\nImage modes:")
    print("-" * 40)

    print(
        quality_df["mode"].value_counts()
    )


# ------------------------------------------------------------
# File extensions
# ------------------------------------------------------------

if len(quality_df) > 0:

    print("\nFile extensions:")
    print("-" * 40)

    print(
        quality_df["extension"].value_counts()
    )


# ------------------------------------------------------------
# Image dimensions
# ------------------------------------------------------------

if len(quality_df) > 0:

    print("\nImage width statistics:")
    print("-" * 40)

    print(
        quality_df["width"].describe()
    )

    print("\nImage height statistics:")
    print("-" * 40)

    print(
        quality_df["height"].describe()
    )


# ------------------------------------------------------------
# Missing folders
# ------------------------------------------------------------

if len(missing_folders) > 0:

    print("\nMissing folders:")
    print("-" * 40)

    for folder in missing_folders:
        print(folder)


# ------------------------------------------------------------
# Corrupted images
# ------------------------------------------------------------

if len(corrupted_df) > 0:

    print("\nCorrupted images:")
    print("-" * 40)

    display(corrupted_df)

else:

    print("\nNo corrupted images detected! ✓")

In [ ]:
# ============================================================
# CELL 20A: Inspect Label Mapping Structure
# ============================================================

print("label_mapping columns:")
print("=" * 60)

print(label_mapping.columns.tolist())

print("\nFirst 5 rows:")
display(label_mapping.head())

In [ ]:
# ============================================================
# CELL 20B: Inspect Actual Dataset Structure
# ============================================================

import os

BASE_PATH = DATASET_ROOT

print("BASE PATH:")
print(BASE_PATH)

print("\nTop-level contents:")
print("=" * 60)

for item in os.listdir(BASE_PATH):
    full_path = os.path.join(BASE_PATH, item)

    if os.path.isdir(full_path):
        print("📁", repr(item))
    else:
        print("📄", repr(item))

In [ ]:
# ============================================================
# CELL 20C: Inspect Dataset Subdirectories
# ============================================================

print("DATASET SUBDIRECTORIES")
print("=" * 60)

for item in os.listdir(BASE_PATH):

    item_path = os.path.join(BASE_PATH, item)

    if os.path.isdir(item_path):

        print(f"\n📁 {item}")

        for subitem in os.listdir(item_path):

            sub_path = os.path.join(item_path, subitem)

            if os.path.isdir(sub_path):
                print(f"   📁 {subitem}")
            else:
                print(f"   📄 {subitem}")

In [ ]:
# ============================================================
# CELL 20D: Final Image Quality Audit
# ============================================================

from PIL import Image
import os
import pandas as pd

# ------------------------------------------------------------
# Dataset paths
# ------------------------------------------------------------

BASE_PATH = DATASET_ROOT

SOURCE_PATHS = {
    "Our Research Pictures": os.path.join(
        BASE_PATH,
        "Our research pictures-20260712T062632Z-2-001",
        "Our research pictures"
    ),

    "Medical Plants": os.path.join(
        BASE_PATH,
        "Medical plants-20260712T062633Z-2-001",
        "Medical plants"
    )
}


# ------------------------------------------------------------
# Supported image formats
# ------------------------------------------------------------

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff",
    ".webp"
}


# ------------------------------------------------------------
# Storage
# ------------------------------------------------------------

quality_records = []
corrupted_images = []
missing_folders = []


# ------------------------------------------------------------
# Scan all 44 mapped folders
# ------------------------------------------------------------

for _, row in label_mapping.iterrows():

    source = row["source"]
    folder_name = row["folder_name"]


    # Get correct parent directory
    source_path = SOURCE_PATHS.get(source)


    if source_path is None:

        missing_folders.append(
            f"Unknown source: {source}"
        )

        continue


    # Build exact folder path
    folder_path = os.path.join(
        source_path,
        folder_name
    )


    # Check folder existence
    if not os.path.isdir(folder_path):

        missing_folders.append(
            folder_path
        )

        continue


    # --------------------------------------------------------
    # Scan images
    # --------------------------------------------------------

    for file_name in os.listdir(folder_path):

        file_path = os.path.join(
            folder_path,
            file_name
        )

        ext = os.path.splitext(
            file_name
        )[1].lower()


        if ext not in IMAGE_EXTENSIONS:
            continue


        try:

            # Verify image integrity
            with Image.open(file_path) as img:
                img.verify()


            # Read image properties
            with Image.open(file_path) as img:

                width, height = img.size
                mode = img.mode


            quality_records.append({

                "file_path": file_path,
                "source": source,
                "folder_name": folder_name,
                "species": row["species"],
                "category": row["category"],
                "health": row["health"],
                "width": width,
                "height": height,
                "mode": mode,
                "extension": ext

            })


        except Exception as e:

            corrupted_images.append({

                "file_path": file_path,
                "error": str(e)

            })


# ------------------------------------------------------------
# Create DataFrames
# ------------------------------------------------------------

quality_df = pd.DataFrame(
    quality_records
)

corrupted_df = pd.DataFrame(
    corrupted_images
)


# ------------------------------------------------------------
# Audit summary
# ------------------------------------------------------------

expected_images = int(
    label_mapping["image_count"].sum()
)

readable_images = len(
    quality_df
)

corrupted_count = len(
    corrupted_df
)

missing_count = len(
    missing_folders
)


print("IMAGE QUALITY AUDIT")
print("=" * 60)

print(
    f"Expected images : {expected_images}"
)

print(
    f"Readable images : {readable_images}"
)

print(
    f"Corrupted images: {corrupted_count}"
)

print(
    f"Missing folders : {missing_count}"
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("\nAUDIT STATUS")
print("=" * 60)

if (
    expected_images == readable_images
    and corrupted_count == 0
    and missing_count == 0
):

    print("✓ All 1299 images verified successfully!")

else:

    print("⚠ Dataset audit requires further inspection.")


# ------------------------------------------------------------
# Image modes
# ------------------------------------------------------------

if len(quality_df) > 0:

    print("\nImage modes:")
    print("-" * 40)

    print(
        quality_df["mode"].value_counts()
    )


# ------------------------------------------------------------
# File extensions
# ------------------------------------------------------------

if len(quality_df) > 0:

    print("\nFile extensions:")
    print("-" * 40)

    print(
        quality_df["extension"].value_counts()
    )


# ------------------------------------------------------------
# Image dimensions
# ------------------------------------------------------------

if len(quality_df) > 0:

    print("\nImage width statistics:")
    print("-" * 40)

    print(
        quality_df["width"].describe()
    )


    print("\nImage height statistics:")
    print("-" * 40)

    print(
        quality_df["height"].describe()
    )


# ------------------------------------------------------------
# Missing folders
# ------------------------------------------------------------

if missing_count > 0:

    print("\nMissing folders:")
    print("-" * 40)

    for folder in missing_folders:
        print(folder)


# ------------------------------------------------------------
# Corrupted images
# ------------------------------------------------------------

if corrupted_count > 0:

    print("\nCorrupted images:")
    print("-" * 40)

    display(corrupted_df)

In [ ]:
# ============================================================
# CELL 21: Duplicate & Near-Duplicate Image Detection
# ============================================================

import hashlib
from PIL import Image
import numpy as np
from collections import defaultdict


# ------------------------------------------------------------
# Exact Duplicate Detection using MD5
# ------------------------------------------------------------

print("EXACT DUPLICATE IMAGE CHECK")
print("=" * 60)

md5_groups = defaultdict(list)

for file_path in quality_df["file_path"]:

    try:
        with open(file_path, "rb") as f:
            file_hash = hashlib.md5(f.read()).hexdigest()

        md5_groups[file_hash].append(file_path)

    except Exception as e:
        print(f"Error reading: {file_path}")


exact_duplicate_groups = {
    h: paths
    for h, paths in md5_groups.items()
    if len(paths) > 1
}


print(
    "Unique image files :",
    len(md5_groups)
)

print(
    "Duplicate groups    :",
    len(exact_duplicate_groups)
)

print(
    "Images involved     :",
    sum(len(v) for v in exact_duplicate_groups.values())
)


# ------------------------------------------------------------
# Show exact duplicates
# ------------------------------------------------------------

if len(exact_duplicate_groups) > 0:

    print("\nExact duplicate groups:")
    print("-" * 60)

    for i, paths in enumerate(
        exact_duplicate_groups.values(),
        start=1
    ):

        print(f"\nGroup {i}:")

        for path in paths:
            print("  ", path)

else:

    print("\nNo exact duplicate images detected! ✓")


# ------------------------------------------------------------
# Perceptual Hash for Near-Duplicate Detection
# ------------------------------------------------------------

print("\n\nNEAR-DUPLICATE IMAGE CHECK")
print("=" * 60)

def average_hash(image_path, hash_size=16):

    with Image.open(image_path) as img:

        img = img.convert("L")
        img = img.resize(
            (hash_size, hash_size)
        )

        pixels = np.asarray(img)

    avg = pixels.mean()

    return (
        pixels > avg
    ).flatten()


def hamming_distance(hash1, hash2):

    return np.count_nonzero(
        hash1 != hash2
    )


# ------------------------------------------------------------
# Generate perceptual hashes
# ------------------------------------------------------------

perceptual_hashes = {}

for file_path in quality_df["file_path"]:

    try:

        perceptual_hashes[file_path] = (
            average_hash(file_path)
        )

    except Exception as e:

        print(
            f"Hashing error: {file_path}"
        )


# ------------------------------------------------------------
# Compare images
# ------------------------------------------------------------

near_duplicate_pairs = []

paths = list(
    perceptual_hashes.keys()
)

for i in range(len(paths)):

    hash1 = perceptual_hashes[
        paths[i]
    ]

    for j in range(i + 1, len(paths)):

        hash2 = perceptual_hashes[
            paths[j]
        ]

        distance = hamming_distance(
            hash1,
            hash2
        )

        # Threshold for possible near duplicates
        if distance <= 8:

            near_duplicate_pairs.append({

                "image_1": paths[i],
                "image_2": paths[j],
                "hamming_distance": distance

            })


near_duplicate_df = pd.DataFrame(
    near_duplicate_pairs
)


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print(
    "Images hashed:",
    len(perceptual_hashes)
)

print(
    "Potential near-duplicate pairs:",
    len(near_duplicate_df)
)


if len(near_duplicate_df) > 0:

    print("\nPotential near-duplicates:")
    print("-" * 60)

    display(
        near_duplicate_df.sort_values(
            "hamming_distance"
        ).head(50)
    )

else:

    print(
        "\nNo potential near-duplicates detected! ✓"
    )

In [ ]:
# ============================================================
# CELL 22: CREATE FINAL IMAGE-LEVEL DATASET
# ============================================================

import os
import pandas as pd

# Dataset base path
# DATASET_ROOT is defined in the configuration cell above.

records = []

for _, row in label_mapping.iterrows():

    source = row["source"]
    folder_name = row["folder_name"]

    # --------------------------------------------------------
    # Construct actual folder path
    # --------------------------------------------------------

    if source == "Our Research Pictures":

        folder_path = os.path.join(
            DATASET_ROOT,
            "Our research pictures-20260712T062632Z-2-001",
            "Our research pictures",
            folder_name
        )

    elif source == "Medical Plants":

        folder_path = os.path.join(
            DATASET_ROOT,
            "Medical plants-20260712T062633Z-2-001",
            "Medical plants",
            folder_name
        )

    else:
        continue

    # --------------------------------------------------------
    # Check folder
    # --------------------------------------------------------

    if not os.path.isdir(folder_path):
        print(f"Warning: Folder not found -> {folder_path}")
        continue

    # --------------------------------------------------------
    # Collect images
    # --------------------------------------------------------

    for filename in os.listdir(folder_path):

        if filename.lower().endswith((".jpg", ".jpeg", ".png")):

            image_path = os.path.join(folder_path, filename)

            records.append({
                "image_path": image_path,
                "species": row["species"],
                "category": row["category"],
                "health": row["health"],
                "source": row["source"],
                "folder_name": row["folder_name"]
            })


# ============================================================
# CREATE FINAL DATAFRAME
# ============================================================

image_df = pd.DataFrame(records)

image_df = image_df.reset_index(drop=True)


# ============================================================
# DATASET SUMMARY
# ============================================================

print("FINAL IMAGE DATASET")
print("=" * 60)

print(f"Total images    : {len(image_df)}")
print(f"Total species   : {image_df['species'].nunique()}")
print(f"Total categories: {image_df['category'].nunique()}")
print(f"Total health    : {image_df['health'].nunique()}")

print("\nColumns:")
print(image_df.columns.tolist())

print("\nFirst 5 records:")
display(image_df.head())


In [ ]:
# ============================================================
# CELL 23: EXACT DUPLICATE IMAGE CHECK
# ============================================================

import hashlib
import os
import pandas as pd


def calculate_md5(file_path):
    """Calculate MD5 hash of an image file."""
    
    hash_md5 = hashlib.md5()

    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            hash_md5.update(chunk)

    return hash_md5.hexdigest()


# Calculate hash for every image
image_df["image_hash"] = image_df["image_path"].apply(calculate_md5)


# ------------------------------------------------------------
# Find duplicate images
# ------------------------------------------------------------

duplicate_mask = image_df["image_hash"].duplicated(keep=False)

duplicate_images = image_df[duplicate_mask].copy()


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("EXACT DUPLICATE IMAGE CHECK")
print("=" * 60)

print(f"Total images checked : {len(image_df)}")
print(f"Unique image hashes  : {image_df['image_hash'].nunique()}")
print(f"Duplicate image rows : {duplicate_mask.sum()}")
print(f"Duplicate groups     : "
      f"{duplicate_images['image_hash'].nunique()}")


if len(duplicate_images) == 0:

    print("\n✓ No exact duplicate images detected!")

else:

    print("\n⚠ Duplicate images detected!")

    print("\nDuplicate groups:")
    
    display(
        duplicate_images[
            [
                "image_path",
                "species",
                "category",
                "health",
                "image_hash"
            ]
        ].sort_values("image_hash")
    )

In [ ]:
# ============================================================
# CELL 24: DUPLICATE AND LABEL-CONFLICT ANALYSIS
# ============================================================

# Group images by their exact hash
duplicate_analysis = (
    image_df
    .groupby("image_hash")
    .agg(
        occurrences=("image_path", "count"),
        species_count=("species", "nunique"),
        category_count=("category", "nunique"),
        health_count=("health", "nunique"),
        species=("species", lambda x: " | ".join(sorted(x.unique()))),
        category=("category", lambda x: " | ".join(sorted(x.unique()))),
        health=("health", lambda x: " | ".join(sorted(x.unique())))
    )
    .reset_index()
)

# Keep only actual duplicates
duplicate_analysis = duplicate_analysis[
    duplicate_analysis["occurrences"] > 1
].copy()


# ------------------------------------------------------------
# Categorize duplicate groups
# ------------------------------------------------------------

duplicate_analysis["conflict_type"] = "Same label"

duplicate_analysis.loc[
    duplicate_analysis["health_count"] > 1,
    "conflict_type"
] = "HEALTH LABEL CONFLICT"

duplicate_analysis.loc[
    duplicate_analysis["species_count"] > 1,
    "conflict_type"
] = "SPECIES LABEL CONFLICT"

duplicate_analysis.loc[
    (duplicate_analysis["health_count"] > 1) &
    (duplicate_analysis["species_count"] > 1),
    "conflict_type"
] = "MULTIPLE LABEL CONFLICT"


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("DUPLICATE & LABEL-CONFLICT ANALYSIS")
print("=" * 60)

print(f"Duplicate groups              : {len(duplicate_analysis)}")
print(
    f"Same-label duplicate groups   : "
    f"{(duplicate_analysis['conflict_type'] == 'Same label').sum()}"
)
print(
    f"Health label conflicts        : "
    f"{(duplicate_analysis['conflict_type'] == 'HEALTH LABEL CONFLICT').sum()}"
)
print(
    f"Species label conflicts       : "
    f"{(duplicate_analysis['conflict_type'] == 'SPECIES LABEL CONFLICT').sum()}"
)
print(
    f"Multiple-label conflicts      : "
    f"{(duplicate_analysis['conflict_type'] == 'MULTIPLE LABEL CONFLICT').sum()}"
)


# ------------------------------------------------------------
# Show label conflicts
# ------------------------------------------------------------

label_conflicts = duplicate_analysis[
    duplicate_analysis["conflict_type"] != "Same label"
].copy()

print("\nLABEL CONFLICTS")
print("=" * 60)

if len(label_conflicts) > 0:
    display(
        label_conflicts[
            [
                "image_hash",
                "occurrences",
                "species",
                "category",
                "health",
                "conflict_type"
            ]
        ].sort_values("conflict_type")
    )
else:
    print("✓ No label conflicts detected.")

In [ ]:
# ============================================================
# CELL 25: LOCATE ALL LABEL-CONFLICTING IMAGES
# ============================================================

# Get hashes involved in label conflicts
conflict_hashes = label_conflicts["image_hash"].tolist()

# Extract every image record belonging to those hashes
conflict_images = image_df[
    image_df["image_hash"].isin(conflict_hashes)
].copy()

# Add filename for easier inspection
conflict_images["filename"] = conflict_images["image_path"].apply(
    os.path.basename
)

# Select useful columns
conflict_images = conflict_images[
    [
        "filename",
        "image_path",
        "species",
        "category",
        "health",
        "source",
        "folder_name",
        "image_hash"
    ]
].sort_values(
    ["species", "image_hash", "folder_name"]
).reset_index(drop=True)


print("LABEL-CONFLICTING IMAGE RECORDS")
print("=" * 70)

print(f"Conflict groups : {len(conflict_hashes)}")
print(f"Image records   : {len(conflict_images)}")

display(conflict_images)

In [ ]:
# ============================================================
# CONFLICTING DUPLICATE IMAGES: FILE PATH INSPECTION
# ============================================================

import pandas as pd

# Find duplicate hashes
hash_counts = image_df["image_hash"].value_counts()
duplicate_hashes = hash_counts[hash_counts > 1].index

duplicate_df = image_df[
    image_df["image_hash"].isin(duplicate_hashes)
].copy()

# ------------------------------------------------------------
# 1. Health-label conflicts
# ------------------------------------------------------------

health_conflicts = (
    duplicate_df.groupby("image_hash")["health"]
    .nunique()
)

health_conflict_hashes = health_conflicts[
    health_conflicts > 1
].index

health_conflict_images = duplicate_df[
    duplicate_df["image_hash"].isin(health_conflict_hashes)
].sort_values("image_hash")

print("=" * 70)
print("HEALTH LABEL CONFLICTS")
print("=" * 70)

if len(health_conflict_images) == 0:
    print("No health-label conflicts found.")
else:
    display(
        health_conflict_images[
            ["image_path", "species", "category", "health", "image_hash"]
        ]
    )


# ------------------------------------------------------------
# 2. Species-label conflicts
# ------------------------------------------------------------

species_conflicts = (
    duplicate_df.groupby("image_hash")["species"]
    .nunique()
)

species_conflict_hashes = species_conflicts[
    species_conflicts > 1
].index

species_conflict_images = duplicate_df[
    duplicate_df["image_hash"].isin(species_conflict_hashes)
].sort_values("image_hash")

print("\n" + "=" * 70)
print("SPECIES LABEL CONFLICTS")
print("=" * 70)

if len(species_conflict_images) == 0:
    print("No species-label conflicts found.")
else:
    display(
        species_conflict_images[
            ["image_path", "species", "category", "health", "image_hash"]
        ]
    )


# ------------------------------------------------------------
# 3. Summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CONFLICT SUMMARY")
print("=" * 70)

print("Health conflict groups  :", len(health_conflict_hashes))
print("Species conflict groups :", len(species_conflict_hashes))
print("Total conflicting groups:",
      len(set(health_conflict_hashes) | set(species_conflict_hashes)))

In [ ]:
# ============================================================
# VISUAL INSPECTION OF LABEL-CONFLICTING DUPLICATES
# ============================================================

import os
import math
import matplotlib.pyplot as plt
from PIL import Image

# Combine all conflicting hashes
conflict_hashes = list(
    set(health_conflict_hashes) |
    set(species_conflict_hashes)
)

conflict_images = duplicate_df[
    duplicate_df["image_hash"].isin(conflict_hashes)
].copy()

print("=" * 70)
print("CONFLICTING IMAGES FOR VISUAL INSPECTION")
print("=" * 70)
print(f"Conflict groups : {len(conflict_hashes)}")
print(f"Image rows      : {len(conflict_images)}")


# ------------------------------------------------------------
# Show each conflicting image pair/group
# ------------------------------------------------------------

for image_hash in conflict_hashes:

    group = conflict_images[
        conflict_images["image_hash"] == image_hash
    ]

    print("\n" + "-" * 70)
    print("HASH:", image_hash)

    for idx, row in group.iterrows():
        print(
            f"Row {idx} | "
            f"Species: {row['species']} | "
            f"Health: {row['health']} | "
            f"Path: {row['image_path']}"
        )

    # Display images
    n = len(group)
    fig, axes = plt.subplots(
        1, n,
        figsize=(5 * n, 5)
    )

    if n == 1:
        axes = [axes]

    for ax, (_, row) in zip(axes, group.iterrows()):

        try:
            img = Image.open(row["image_path"]).convert("RGB")
            ax.imshow(img)

            ax.set_title(
                f"Species: {row['species']}\n"
                f"Health: {row['health']}\n"
                f"Row: {row.name}",
                fontsize=10
            )

            ax.axis("off")

        except Exception as e:
            ax.set_title(f"ERROR\n{e}")
            ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# STEP: CLEAN LABEL-CONFLICTING DUPLICATES
# ============================================================

import pandas as pd

# image_df contains image-level records + image_hash
clean_df = image_df.copy()

before_count = len(clean_df)

print("=" * 70)
print("CLEANING LABEL-CONFLICTING DUPLICATES")
print("=" * 70)


# ------------------------------------------------------------
# 1. Almond ↔ Chrysanthemum conflicts
# ------------------------------------------------------------

species_conflict_hashes = (
    clean_df.groupby("image_hash")["species"]
    .nunique()
)

species_conflict_hashes = species_conflict_hashes[
    species_conflict_hashes > 1
].index

# Based on visual inspection:
# conflicting images belong to Chrysanthemum,
# so remove the Almond copies.

almond_conflict_rows = clean_df[
    (clean_df["image_hash"].isin(species_conflict_hashes)) &
    (clean_df["species"] == "Almond")
].index

print(f"Almond conflict rows removed : {len(almond_conflict_rows)}")

clean_df = clean_df.drop(index=almond_conflict_rows)


# ------------------------------------------------------------
# 2. Puishak Healthy ↔ Diseased conflicts
# ------------------------------------------------------------

health_conflict_hashes = (
    clean_df.groupby("image_hash")["health"]
    .nunique()
)

health_conflict_hashes = health_conflict_hashes[
    health_conflict_hashes > 1
].index

# Keep Diseased copy and remove Healthy duplicate.

puishak_healthy_conflict_rows = clean_df[
    (clean_df["image_hash"].isin(health_conflict_hashes)) &
    (clean_df["species"] == "Puishak") &
    (clean_df["health"] == "Healthy")
].index

print(
    f"Puishak Healthy conflict rows removed : "
    f"{len(puishak_healthy_conflict_rows)}"
)

clean_df = clean_df.drop(index=puishak_healthy_conflict_rows)


# ------------------------------------------------------------
# Reset index
# ------------------------------------------------------------

clean_df = clean_df.reset_index(drop=True)

after_count = len(clean_df)


# ------------------------------------------------------------
# Cleaning summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CLEANING SUMMARY")
print("=" * 70)

print(f"Original images : {before_count}")
print(f"Removed images  : {before_count - after_count}")
print(f"Cleaned images  : {after_count}")


# ------------------------------------------------------------
# Verify remaining conflicts
# ------------------------------------------------------------

print("\nRemaining health conflicts:")

remaining_health_conflicts = (
    clean_df.groupby("image_hash")["health"].nunique()
)

print((remaining_health_conflicts > 1).sum())


print("\nRemaining species conflicts:")

remaining_species_conflicts = (
    clean_df.groupby("image_hash")["species"].nunique()
)

print((remaining_species_conflicts > 1).sum())

In [ ]:
# Check available DataFrame variables

import pandas as pd

df_names = [
    name for name, obj in globals().items()
    if isinstance(obj, pd.DataFrame)
]

print("Available DataFrames:")
print("=" * 50)

for name in df_names:
    print(f"{name:25s} -> {obj.shape if False else globals()[name].shape}")

In [ ]:
print("quality_df columns:")
print(quality_df.columns.tolist())

print("\nShape:", quality_df.shape)

In [ ]:
print("image_df columns:")
print(image_df.columns.tolist())

print("\nShape:", image_df.shape)

In [ ]:
# ============================================================
# CELL 26: REMAINING EXACT DUPLICATE ANALYSIS
# ============================================================

print("=" * 70)
print("REMAINING EXACT DUPLICATE ANALYSIS")
print("=" * 70)

# Count occurrences of each image hash
hash_counts = clean_df["image_hash"].value_counts()

# Keep only hashes appearing more than once
duplicate_hashes = hash_counts[hash_counts > 1]

print(f"Total cleaned images : {len(clean_df)}")
print(f"Unique image hashes  : {clean_df['image_hash'].nunique()}")
print(f"Duplicate groups     : {len(duplicate_hashes)}")
print(f"Duplicate rows       : {duplicate_hashes.sum()}")
print(
    f"Extra duplicate rows : "
    f"{duplicate_hashes.sum() - len(duplicate_hashes)}"
)

print("\nTop remaining duplicate groups:")
print(duplicate_hashes.head(20))

In [ ]:
# ============================================================
# CELL 27: DUPLICATE CONSISTENCY CHECK
# ============================================================

print("=" * 70)
print("DUPLICATE LABEL CONSISTENCY CHECK")
print("=" * 70)

duplicate_hash_list = duplicate_hashes.index

duplicate_records = clean_df[
    clean_df["image_hash"].isin(duplicate_hash_list)
].copy()

# Check species consistency
species_per_hash = (
    duplicate_records
    .groupby("image_hash")["species"]
    .nunique()
)

# Check health consistency
health_per_hash = (
    duplicate_records
    .groupby("image_hash")["health"]
    .nunique()
)

# Check category consistency
category_per_hash = (
    duplicate_records
    .groupby("image_hash")["category"]
    .nunique()
)

print(f"Duplicate groups checked : {len(duplicate_hash_list)}")

print("\nSpecies conflicts among duplicates:")
print((species_per_hash > 1).sum())

print("\nHealth conflicts among duplicates:")
print((health_per_hash > 1).sum())

print("\nCategory conflicts among duplicates:")
print((category_per_hash > 1).sum())


# ------------------------------------------------------------
# Show any remaining conflicts
# ------------------------------------------------------------

remaining_conflict_hashes = set(
    species_per_hash[species_per_hash > 1].index
) | set(
    health_per_hash[health_per_hash > 1].index
) | set(
    category_per_hash[category_per_hash > 1].index
)

print("\n" + "=" * 70)
print("REMAINING DUPLICATE CONFLICTS")
print("=" * 70)

print(
    f"Conflicting duplicate groups : "
    f"{len(remaining_conflict_hashes)}"
)

if remaining_conflict_hashes:
    print("\nConflict records:")
    print(
        duplicate_records[
            duplicate_records["image_hash"].isin(
                remaining_conflict_hashes
            )
        ][
            [
                "image_path",
                "species",
                "category",
                "health",
                "image_hash"
            ]
        ].to_string(index=False)
    )
else:
    print("✓ All remaining duplicates have consistent labels.")

In [ ]:
# ============================================================
# CELL 28: REMOVE EXACT DUPLICATE COPIES
# ============================================================

print("=" * 70)
print("REMOVING EXACT DUPLICATE COPIES")
print("=" * 70)

before_count = len(clean_df)

# Keep only one copy of each exact image
clean_df = clean_df.drop_duplicates(
    subset="image_hash",
    keep="first"
).reset_index(drop=True)

after_count = len(clean_df)

print(f"Images before duplicate removal : {before_count}")
print(f"Duplicate copies removed        : {before_count - after_count}")
print(f"Final unique images             : {after_count}")

print("\nUnique image hashes:")
print(clean_df["image_hash"].nunique())

print("\nRemaining exact duplicate groups:")

remaining_duplicates = (
    clean_df["image_hash"].value_counts()
)

remaining_duplicates = remaining_duplicates[
    remaining_duplicates > 1
]

print(len(remaining_duplicates))

if len(remaining_duplicates) == 0:
    print("✓ No exact duplicate images remain.")

In [ ]:
# ============================================================
# CELL 23: FINAL CLEAN DATASET DISTRIBUTION
# ============================================================

print("=" * 70)
print("FINAL CLEAN DATASET DISTRIBUTION")
print("=" * 70)

print(f"Total images    : {len(clean_df)}")
print(f"Total species   : {clean_df['species'].nunique()}")
print(f"Total categories: {clean_df['category'].nunique()}")
print(f"Total health    : {clean_df['health'].nunique()}")

# ------------------------------------------------------------
# 1. Species distribution
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SPECIES-WISE DISTRIBUTION")
print("=" * 70)

species_dist = (
    clean_df["species"]
    .value_counts()
    .sort_values(ascending=False)
)

display(species_dist.to_frame("image_count"))


# ------------------------------------------------------------
# 2. Health distribution
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("HEALTH-WISE DISTRIBUTION")
print("=" * 70)

health_dist = (
    clean_df["health"]
    .value_counts()
)

display(health_dist.to_frame("image_count"))


# ------------------------------------------------------------
# 3. Species × Health distribution
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SPECIES × HEALTH DISTRIBUTION")
print("=" * 70)

species_health_dist = pd.crosstab(
    clean_df["species"],
    clean_df["health"]
)

display(species_health_dist)


# ------------------------------------------------------------
# 4. Category distribution
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CATEGORY DISTRIBUTION")
print("=" * 70)

category_dist = (
    clean_df["category"]
    .value_counts()
)

display(category_dist.to_frame("image_count"))


# ------------------------------------------------------------
# 5. Smallest classes
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SMALLEST SPECIES CLASSES")
print("=" * 70)

display(
    species_dist
    .sort_values()
    .head(10)
    .to_frame("image_count")
)


# ------------------------------------------------------------
# 6. Largest classes
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("LARGEST SPECIES CLASSES")
print("=" * 70)

display(
    species_dist
    .head(10)
    .to_frame("image_count")
)

print("\n" + "=" * 70)
print("✓ FINAL DATASET CHECK COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 24: FINAL 70/15/15 STRATIFIED SPLIT
# ============================================================

from sklearn.model_selection import train_test_split
import pandas as pd

print("=" * 70)
print("CREATING FINAL STRATIFIED 70/15/15 SPLIT")
print("=" * 70)

df = clean_df.copy()

# ------------------------------------------------------------
# Combined species + health label
# ------------------------------------------------------------

df["stratify_label"] = (
    df["species"].astype(str)
    + "__"
    + df["health"].astype(str)
)

# ============================================================
# STEP 1: 70% TRAIN / 30% TEMP
# ============================================================

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["stratify_label"]
)

print(f"\nTrain      : {len(train_df)}")
print(f"Temporary  : {len(temp_df)}")


# ============================================================
# STEP 2: TEMP → VALIDATION + TEST
#
# Some tiny groups have only 1 sample inside temp_df.
# Therefore, use species-level stratification where possible.
# For species that still have only 1 sample, don't stratify them.
# ============================================================

temp_species_counts = temp_df["species"].value_counts()

# Species having at least 2 samples can participate in stratification
valid_species = temp_species_counts[
    temp_species_counts >= 2
].index

stratified_temp = temp_df[
    temp_df["species"].isin(valid_species)
].copy()

unstratified_temp = temp_df[
    ~temp_df["species"].isin(valid_species)
].copy()

print("\nTemporary set:")
print(f"Stratified portion   : {len(stratified_temp)}")
print(f"Unstratified portion : {len(unstratified_temp)}")


# ------------------------------------------------------------
# Split the stratifiable portion
# ------------------------------------------------------------

val_part, test_part = train_test_split(
    stratified_temp,
    test_size=0.50,
    random_state=42,
    stratify=stratified_temp["species"]
)


# ------------------------------------------------------------
# Split the tiny/unstratifiable samples manually
# ------------------------------------------------------------

# Deterministic split so results remain reproducible
unstratified_temp = unstratified_temp.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

n_test_extra = len(unstratified_temp) // 2

test_extra = unstratified_temp.iloc[:n_test_extra]
val_extra = unstratified_temp.iloc[n_test_extra:]


# ------------------------------------------------------------
# Combine
# ------------------------------------------------------------

val_df = pd.concat(
    [val_part, val_extra],
    ignore_index=True
)

test_df = pd.concat(
    [test_part, test_extra],
    ignore_index=True
)


# ============================================================
# RESET INDEX
# ============================================================

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)


# Remove helper column
train_df = train_df.drop(columns=["stratify_label"])
val_df = val_df.drop(columns=["stratify_label"])
test_df = test_df.drop(columns=["stratify_label"])


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("FINAL SPLIT SUMMARY")
print("=" * 70)

total = len(df)

print(f"Total images      : {total}")
print(f"Training images   : {len(train_df)} ({len(train_df)/total*100:.2f}%)")
print(f"Validation images : {len(val_df)} ({len(val_df)/total*100:.2f}%)")
print(f"Test images       : {len(test_df)} ({len(test_df)/total*100:.2f}%)")


# ============================================================
# SPECIES DISTRIBUTION
# ============================================================

print("\n" + "=" * 70)
print("SPECIES DISTRIBUTION")
print("=" * 70)

species_split = pd.DataFrame({
    "Total": df["species"].value_counts(),
    "Train": train_df["species"].value_counts(),
    "Validation": val_df["species"].value_counts(),
    "Test": test_df["species"].value_counts()
}).fillna(0).astype(int)

display(species_split)


# ============================================================
# HEALTH DISTRIBUTION
# ============================================================

print("\n" + "=" * 70)
print("HEALTH DISTRIBUTION")
print("=" * 70)

health_split = pd.DataFrame({
    "Total": df["health"].value_counts(),
    "Train": train_df["health"].value_counts(),
    "Validation": val_df["health"].value_counts(),
    "Test": test_df["health"].value_counts()
}).fillna(0).astype(int)

display(health_split)


# ============================================================
# CATEGORY DISTRIBUTION
# ============================================================

print("\n" + "=" * 70)
print("CATEGORY DISTRIBUTION")
print("=" * 70)

category_split = pd.DataFrame({
    "Total": df["category"].value_counts(),
    "Train": train_df["category"].value_counts(),
    "Validation": val_df["category"].value_counts(),
    "Test": test_df["category"].value_counts()
}).fillna(0).astype(int)

display(category_split)


print("\n" + "=" * 70)
print("✓ FINAL 70/15/15 SPLIT COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 25: DATA LEAKAGE CHECK
# ============================================================

print("=" * 70)
print("DATA LEAKAGE CHECK")
print("=" * 70)

# ------------------------------------------------------------
# 1. Image path overlap
# ------------------------------------------------------------

train_paths = set(train_df["image_path"])
val_paths   = set(val_df["image_path"])
test_paths  = set(test_df["image_path"])

train_val_overlap = train_paths & val_paths
train_test_overlap = train_paths & test_paths
val_test_overlap = val_paths & test_paths

print("\nIMAGE PATH OVERLAP")
print("-" * 70)

print(f"Train ∩ Validation : {len(train_val_overlap)}")
print(f"Train ∩ Test       : {len(train_test_overlap)}")
print(f"Validation ∩ Test  : {len(val_test_overlap)}")


# ------------------------------------------------------------
# 2. Exact image hash overlap
# ------------------------------------------------------------

# image_hash may not be present if it was removed earlier.
# Recover it from the original image_df if necessary.

if "image_hash" not in train_df.columns:

    hash_lookup = image_df[
        ["image_path", "image_hash"]
    ].drop_duplicates("image_path")

    train_check = train_df.merge(
        hash_lookup,
        on="image_path",
        how="left"
    )

    val_check = val_df.merge(
        hash_lookup,
        on="image_path",
        how="left"
    )

    test_check = test_df.merge(
        hash_lookup,
        on="image_path",
        how="left"
    )

else:

    train_check = train_df.copy()
    val_check = val_df.copy()
    test_check = test_df.copy()


train_hashes = set(
    train_check["image_hash"].dropna()
)

val_hashes = set(
    val_check["image_hash"].dropna()
)

test_hashes = set(
    test_check["image_hash"].dropna()
)

train_val_hash_overlap = train_hashes & val_hashes
train_test_hash_overlap = train_hashes & test_hashes
val_test_hash_overlap = val_hashes & test_hashes


print("\nEXACT IMAGE HASH OVERLAP")
print("-" * 70)

print(f"Train ∩ Validation : {len(train_val_hash_overlap)}")
print(f"Train ∩ Test       : {len(train_test_hash_overlap)}")
print(f"Validation ∩ Test  : {len(val_test_hash_overlap)}")


# ------------------------------------------------------------
# 3. Label consistency
# ------------------------------------------------------------

print("\nLABEL CONSISTENCY")
print("-" * 70)

print(
    "Train species labels   :",
    train_df["species"].nunique()
)

print(
    "Validation species     :",
    val_df["species"].nunique()
)

print(
    "Test species           :",
    test_df["species"].nunique()
)


# ------------------------------------------------------------
# 4. Final verdict
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("LEAKAGE CHECK RESULT")
print("=" * 70)

total_overlap = (
    len(train_val_overlap)
    + len(train_test_overlap)
    + len(val_test_overlap)
    + len(train_val_hash_overlap)
    + len(train_test_hash_overlap)
    + len(val_test_hash_overlap)
)

if total_overlap == 0:

    print("✓ No image-path overlap detected.")
    print("✓ No exact image-hash overlap detected.")
    print("✓ Train / Validation / Test are leakage-free.")

else:

    print("⚠ Potential data leakage detected!")
    print("Please inspect the overlap results above.")

print("=" * 70)

In [ ]:
# ============================================================
# CELL 22: SPECIES LABEL ENCODING
# ============================================================

from sklearn.preprocessing import LabelEncoder
import pandas as pd
import numpy as np

print("=" * 70)
print("SPECIES LABEL ENCODING")
print("=" * 70)

# ------------------------------------------------------------
# Create label encoder
# ------------------------------------------------------------

label_encoder = LabelEncoder()

# Fit ONLY on the complete cleaned dataset's species names
# This guarantees one consistent mapping across all splits.
label_encoder.fit(clean_df["species"])

# ------------------------------------------------------------
# Encode labels
# ------------------------------------------------------------

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["label"] = label_encoder.transform(train_df["species"])
val_df["label"] = label_encoder.transform(val_df["species"])
test_df["label"] = label_encoder.transform(test_df["species"])

# ------------------------------------------------------------
# Number of classes
# ------------------------------------------------------------

num_classes = len(label_encoder.classes_)

print(f"Number of species/classes : {num_classes}")

print("\nSpecies → Label mapping:")
print("-" * 70)

for idx, species in enumerate(label_encoder.classes_):
    print(f"{idx:2d} -> {species}")

# ------------------------------------------------------------
# Verify labels
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("LABEL DISTRIBUTION")
print("=" * 70)

print(f"Train labels      : {train_df['label'].nunique()}")
print(f"Validation labels : {val_df['label'].nunique()}")
print(f"Test labels       : {test_df['label'].nunique()}")

print("\nLabel encoding complete ✓")

In [ ]:
# ============================================================
# CELL 23: IMAGE PREPROCESSING + AUGMENTATION
# ============================================================

import torch
from torchvision import transforms

print("=" * 70)
print("IMAGE PREPROCESSING & AUGMENTATION")
print("=" * 70)

# ------------------------------------------------------------
# Image configuration
# ------------------------------------------------------------

IMG_SIZE = 224

print(f"Image size : {IMG_SIZE} x {IMG_SIZE}")

# ------------------------------------------------------------
# Training transformations
# ------------------------------------------------------------
# Augmentation is applied ONLY to training images.

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),

    transforms.RandomRotation(
        degrees=20
    ),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05
    ),

    transforms.RandomResizedCrop(
        IMG_SIZE,
        scale=(0.80, 1.0),
        ratio=(0.90, 1.10)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ------------------------------------------------------------
# Validation transformations
# ------------------------------------------------------------
# NO random augmentation.

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ------------------------------------------------------------
# Test transformations
# ------------------------------------------------------------
# Test set must remain completely deterministic.

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\nTraining augmentation:")
print("  ✓ Horizontal flip")
print("  ✓ Vertical flip")
print("  ✓ Rotation")
print("  ✓ Color jitter")
print("  ✓ Random resized crop")

print("\nValidation:")
print("  ✓ Resize")
print("  ✓ Normalize")
print("  ✗ Random augmentation")

print("\nTest:")
print("  ✓ Resize")
print("  ✓ Normalize")
print("  ✗ Random augmentation")

print("\n" + "=" * 70)
print("TRANSFORMS READY ✓")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 24: PYTORCH DATASET
# ============================================================

from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torch
import os

print("=" * 70)
print("CREATING PYTORCH DATASET")
print("=" * 70)


# ------------------------------------------------------------
# Custom Dataset class
# ------------------------------------------------------------

class LeafDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):

        row = self.dataframe.iloc[idx]

        image_path = row["image_path"]
        label = int(row["label"])

        # Load image
        image = Image.open(image_path).convert("RGB")

        # Apply transformation
        if self.transform is not None:
            image = self.transform(image)

        return image, label


# ------------------------------------------------------------
# Create datasets
# ------------------------------------------------------------

train_dataset = LeafDataset(
    train_df,
    transform=train_transform
)

val_dataset = LeafDataset(
    val_df,
    transform=val_transform
)

test_dataset = LeafDataset(
    test_df,
    transform=test_transform
)


# ------------------------------------------------------------
# Dataset sizes
# ------------------------------------------------------------

print(f"Training dataset   : {len(train_dataset)}")
print(f"Validation dataset : {len(val_dataset)}")
print(f"Test dataset       : {len(test_dataset)}")


# ------------------------------------------------------------
# DataLoader configuration
# ------------------------------------------------------------

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# ------------------------------------------------------------
# Verify one batch
# ------------------------------------------------------------

images, labels = next(iter(train_loader))

print("\n" + "=" * 70)
print("BATCH VERIFICATION")
print("=" * 70)

print(f"Image batch shape : {images.shape}")
print(f"Label batch shape : {labels.shape}")
print(f"Image dtype       : {images.dtype}")
print(f"Label dtype       : {labels.dtype}")

print("\nExpected:")
print(f"  Batch size : {BATCH_SIZE}")
print(f"  Channels   : 3")
print(f"  Height     : {IMG_SIZE}")
print(f"  Width      : {IMG_SIZE}")

print("\n" + "=" * 70)
print("PYTORCH DATASET & DATALOADERS READY ✓")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 25: CLASS / LABEL MAPPING VERIFICATION
# ============================================================

print("=" * 70)
print("CLASS / LABEL MAPPING VERIFICATION")
print("=" * 70)

# Create class names from the final cleaned dataset
class_names = sorted(image_df["species"].unique().tolist())

# Create class-to-index mapping
class_to_idx = {
    name: idx
    for idx, name in enumerate(class_names)
}

idx_to_class = {
    idx: name
    for name, idx in class_to_idx.items()
}

print(f"Total classes : {len(class_names)}")

print("\nClass index mapping:")
print("-" * 70)

for idx, name in enumerate(class_names):
    print(f"{idx:2d} -> {name}")

print("\n" + "=" * 70)
print("LABEL RANGE CHECK")
print("=" * 70)

print(f"Minimum label in batch : {labels.min().item()}")
print(f"Maximum label in batch : {labels.max().item()}")

assert labels.min().item() >= 0
assert labels.max().item() < len(class_names)

print("\n✓ All labels are within valid class range.")
print("✓ Class mapping verified.")

print("=" * 70)

In [ ]:
# ============================================================
# CELL 26: VERIFY DATASET LABEL ENCODING
# ============================================================

print("=" * 70)
print("DATASET LABEL ENCODING VERIFICATION")
print("=" * 70)

print(f"Number of classes : {len(class_names)}")
print(f"Number of mappings: {len(class_to_idx)}")

# Check that every class has exactly one index
assert len(class_names) == 33
assert len(class_to_idx) == 33
assert set(class_names) == set(class_to_idx.keys())

print("\nClass-to-index mapping:")
print("-" * 70)

for class_name in class_names:
    print(f"{class_to_idx[class_name]:2d} -> {class_name}")

print("\n" + "=" * 70)
print("✓ Dataset label encoding verified.")
print("✓ 33 classes are mapped uniquely.")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 27: EFFICIENTNET-B0 MODEL SETUP
# ============================================================

import torch
import torch.nn as nn
from torchvision import models

print("=" * 70)
print("EFFICIENTNET-B0 MODEL SETUP")
print("=" * 70)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

num_classes = len(class_names)

print(f"Device           : {device}")
print(f"Number of classes: {num_classes}")


# ------------------------------------------------------------
# Load EfficientNet-B0
# ------------------------------------------------------------

print("\nLoading EfficientNet-B0...")

weights = models.EfficientNet_B0_Weights.DEFAULT

try:
    model = models.efficientnet_b0(
        weights=weights
    )

    pretrained_loaded = True

    print("✓ ImageNet pretrained weights loaded.")

except Exception as e:

    pretrained_loaded = False

    print("⚠ Pretrained weights could not be loaded.")
    print(f"Reason: {type(e).__name__}")
    print("STOP: Do not train from scratch yet.")
    print("We need the pretrained weights available first.")

    model = None


# ------------------------------------------------------------
# Replace classifier for our 33 classes
# ------------------------------------------------------------

if pretrained_loaded:

    in_features = model.classifier[1].in_features

    model.classifier[1] = nn.Linear(
        in_features,
        num_classes
    )

    model = model.to(device)

    print("\nClassifier updated:")
    print(f"Input features : {in_features}")
    print(f"Output classes : {num_classes}")

    print("\n" + "=" * 70)
    print("EFFICIENTNET-B0 MODEL READY ✓")
    print("=" * 70)

In [ ]:
# ============================================================
# CELL 28: TRAINING CONFIGURATION
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim

print("=" * 70)
print("TRAINING CONFIGURATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Verify model
# ------------------------------------------------------------

assert model is not None, "Model is not available."
assert pretrained_loaded, "Pretrained weights are not loaded."

print(f"Device           : {device}")
print(f"Number of classes: {num_classes}")


# ------------------------------------------------------------
# 2. Freeze EfficientNet backbone
# ------------------------------------------------------------
# Initially train only the new classification layer.
# This is safer for our relatively small dataset.

for param in model.features.parameters():
    param.requires_grad = False

# Keep classifier trainable
for param in model.classifier.parameters():
    param.requires_grad = True


# ------------------------------------------------------------
# 3. Loss function
# ------------------------------------------------------------

criterion = nn.CrossEntropyLoss()


# ------------------------------------------------------------
# 4. Optimizer
# ------------------------------------------------------------

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3,
    weight_decay=1e-4
)


# ------------------------------------------------------------
# 5. Learning-rate scheduler
# ------------------------------------------------------------

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)


# ------------------------------------------------------------
# 6. Training parameters
# ------------------------------------------------------------

NUM_EPOCHS = 15

BEST_MODEL_PATH = "best_efficientnet_b0.pth"

print("\nTraining parameters:")
print(f"Epochs           : {NUM_EPOCHS}")
print(f"Initial LR       : {optimizer.param_groups[0]['lr']}")
print(f"Weight decay     : 1e-4")
print(f"Optimizer        : AdamW")
print(f"Loss             : CrossEntropyLoss")
print(f"Scheduler        : ReduceLROnPlateau")


# ------------------------------------------------------------
# 7. Trainable parameter check
# ------------------------------------------------------------

total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("\nParameter summary:")
print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")
print(f"Frozen parameters    : {total_params - trainable_params:,}")


# ------------------------------------------------------------
# Final verification
# ------------------------------------------------------------

assert trainable_params > 0

print("\n" + "=" * 70)
print("TRAINING CONFIGURATION READY ✓")
print("=" * 70)


In [ ]:
# ============================================================
# CELL 29: STRATIFIED 4-FOLD TRAINING + EARLY STOPPING
# ============================================================

import copy
import time
import numpy as np
import torch

from sklearn.model_selection import StratifiedKFold

print("=" * 70)
print("STRATIFIED 4-FOLD CROSS-VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

N_SPLITS = 4
MAX_EPOCHS = 150
EARLY_STOPPING_PATIENCE = 5

print(f"Device                 : {device}")
print(f"Number of folds        : {N_SPLITS}")
print(f"Maximum epochs         : {MAX_EPOCHS}")
print(f"Early stopping patience: {EARLY_STOPPING_PATIENCE}")

# ------------------------------------------------------------
# Use the 70% training dataframe for K-Fold
# ------------------------------------------------------------

kf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=42
)

# Stratification label
train_stratify_label = (
    train_df["species"].astype(str)
    + "__"
    + train_df["health"].astype(str)
)

# ------------------------------------------------------------
# Store results from every fold
# ------------------------------------------------------------

fold_results = []
fold_histories = []

best_fold_models = []

print("\n" + "=" * 70)
print("STARTING CROSS-VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# K-FOLD LOOP
# ------------------------------------------------------------

for fold, (train_idx, val_idx) in enumerate(
    kf.split(train_df, train_stratify_label),
    start=1
):

    print("\n" + "=" * 70)
    print(f"FOLD {fold}/{N_SPLITS}")
    print("=" * 70)

    print(f"Fold training images   : {len(train_idx)}")
    print(f"Fold validation images : {len(val_idx)}")

    # --------------------------------------------------------
    # Create fold datasets
    # --------------------------------------------------------

    fold_train_df = train_df.iloc[train_idx].reset_index(drop=True)
    fold_val_df = train_df.iloc[val_idx].reset_index(drop=True)

    # --------------------------------------------------------
    # Create DataLoaders
    # --------------------------------------------------------

    fold_train_dataset = LeafDataset(
        fold_train_df,
        transform=train_transform
    )

    fold_val_dataset = LeafDataset(
        fold_val_df,
        transform=val_transform
    )

    fold_train_loader = torch.utils.data.DataLoader(
        fold_train_dataset,
        batch_size=32,
        shuffle=True,
        num_workers=2,
        pin_memory=False
    )

    fold_val_loader = torch.utils.data.DataLoader(
        fold_val_dataset,
        batch_size=32,
        shuffle=False,
        num_workers=2,
        pin_memory=False
    )

    # --------------------------------------------------------
    # Create a fresh EfficientNet-B0 for this fold
    # --------------------------------------------------------

    fold_model = models.efficientnet_b0(
        weights=weights
    )

    in_features = fold_model.classifier[1].in_features

    fold_model.classifier[1] = nn.Linear(
        in_features,
        num_classes
    )

    # --------------------------------------------------------
    # Move model to device
    # --------------------------------------------------------

    fold_model = fold_model.to(device)

    # --------------------------------------------------------
    # Optimizer
    # --------------------------------------------------------

    fold_optimizer = torch.optim.AdamW(
        fold_model.parameters(),
        lr=0.001,
        weight_decay=1e-4
    )

    # --------------------------------------------------------
    # Loss
    # --------------------------------------------------------

    fold_criterion = nn.CrossEntropyLoss()

    # --------------------------------------------------------
    # Scheduler
    # --------------------------------------------------------

    fold_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        fold_optimizer,
        mode="min",
        factor=0.5,
        patience=2
    )

    # --------------------------------------------------------
    # Fold tracking
    # --------------------------------------------------------

    best_val_acc = 0.0
    best_val_loss = float("inf")
    best_epoch = 0

    best_model_state = copy.deepcopy(
        fold_model.state_dict()
    )

    patience_counter = 0

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
        "lr": []
    }

    # ========================================================
    # EPOCH LOOP
    # ========================================================

    for epoch in range(MAX_EPOCHS):

        start_time = time.time()

        # ====================================================
        # TRAINING
        # ====================================================

        fold_model.train()

        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in fold_train_loader:

            images = images.to(device)
            labels = labels.to(device)

            fold_optimizer.zero_grad()

            outputs = fold_model(images)

            loss = fold_criterion(
                outputs,
                labels
            )

            loss.backward()

            fold_optimizer.step()

            running_loss += (
                loss.item() * images.size(0)
            )

            _, predicted = torch.max(
                outputs,
                1
            )

            total += labels.size(0)

            correct += (
                predicted == labels
            ).sum().item()

        train_loss = running_loss / total

        train_acc = (
            100.0 * correct / total
        )

        # ====================================================
        # VALIDATION
        # ====================================================

        fold_model.eval()

        val_running_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():

            for images, labels in fold_val_loader:

                images = images.to(device)
                labels = labels.to(device)

                outputs = fold_model(images)

                loss = fold_criterion(
                    outputs,
                    labels
                )

                val_running_loss += (
                    loss.item() * images.size(0)
                )

                _, predicted = torch.max(
                    outputs,
                    1
                )

                val_total += labels.size(0)

                val_correct += (
                    predicted == labels
                ).sum().item()

        val_loss = (
            val_running_loss / val_total
        )

        val_acc = (
            100.0 * val_correct / val_total
        )

        # ====================================================
        # SCHEDULER
        # ====================================================

        fold_scheduler.step(val_loss)

        current_lr = (
            fold_optimizer
            .param_groups[0]["lr"]
        )

        # ====================================================
        # SAVE HISTORY
        # ====================================================

        history["train_loss"].append(
            train_loss
        )

        history["train_acc"].append(
            train_acc
        )

        history["val_loss"].append(
            val_loss
        )

        history["val_acc"].append(
            val_acc
        )

        history["lr"].append(
            current_lr
        )

        # ====================================================
        # EARLY STOPPING
        # ====================================================

        if val_acc > best_val_acc:

            best_val_acc = val_acc
            best_val_loss = val_loss
            best_epoch = epoch + 1

            best_model_state = copy.deepcopy(
                fold_model.state_dict()
            )

            patience_counter = 0

            best_marker = " ★ BEST"

        else:

            patience_counter += 1

            best_marker = ""

        # ====================================================
        # EPOCH OUTPUT
        # ====================================================

        epoch_time = (
            time.time() - start_time
        )

        print(
            f"Fold {fold} | "
            f"Epoch [{epoch + 1:02d}/{MAX_EPOCHS}] | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.2f}% | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.2f}% | "
            f"LR: {current_lr:.6f} | "
            f"Time: {epoch_time:.1f}s"
            f"{best_marker}"
        )

        # ====================================================
        # CHECK EARLY STOPPING
        # ====================================================

        if patience_counter >= EARLY_STOPPING_PATIENCE:

            print(
                f"\n⚠ Early stopping triggered "
                f"at epoch {epoch + 1}."
            )

            print(
                f"Best epoch: {best_epoch}"
            )

            break

    # --------------------------------------------------------
    # Restore best model for this fold
    # --------------------------------------------------------

    fold_model.load_state_dict(
        best_model_state
    )

    # --------------------------------------------------------
    # Save fold model
    # --------------------------------------------------------

    fold_model_path = (
        f"efficientnet_b0_fold_{fold}.pth"
    )

    torch.save(
        best_model_state,
        fold_model_path
    )

    # --------------------------------------------------------
    # Store fold result
    # --------------------------------------------------------

    fold_results.append({
        "fold": fold,
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "best_val_acc": best_val_acc
    })

    fold_histories.append(history)

    best_fold_models.append(
        best_model_state
    )

    print("\nFold summary:")
    print(f"Best epoch      : {best_epoch}")
    print(f"Best Val Loss   : {best_val_loss:.4f}")
    print(f"Best Val Acc    : {best_val_acc:.2f}%")
    print(f"Model saved     : {fold_model_path}")


# ============================================================
# CROSS-VALIDATION SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("CROSS-VALIDATION COMPLETE")
print("=" * 70)

fold_accuracies = [
    result["best_val_acc"]
    for result in fold_results
]

mean_cv_accuracy = np.mean(
    fold_accuracies
)

std_cv_accuracy = np.std(
    fold_accuracies
)

print("\nFold validation accuracies:")

for result in fold_results:

    print(
        f"Fold {result['fold']} : "
        f"{result['best_val_acc']:.2f}% "
        f"(Epoch {result['best_epoch']})"
    )

print("\n" + "-" * 70)

print(
    f"Mean CV Accuracy : "
    f"{mean_cv_accuracy:.2f}%"
)

print(
    f"Std CV Accuracy  : "
    f"{std_cv_accuracy:.2f}%"
)

print("=" * 70)
print("✓ STRATIFIED 4-FOLD + EARLY STOPPING COMPLETE")
print("=" * 70)


In [ ]:
# ============================================================
# CELL 30: CROSS-VALIDATION RESULTS VISUALIZATION
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

print("=" * 70)
print("CROSS-VALIDATION RESULTS")
print("=" * 70)

# ------------------------------------------------------------
# Fold results
# ------------------------------------------------------------

fold_numbers = np.arange(1, len(fold_results) + 1)

fold_accuracies = [
    result["best_val_acc"]
    for result in fold_results
]

fold_epochs = [
    result["best_epoch"]
    for result in fold_results
]

mean_accuracy = np.mean(fold_accuracies)
std_accuracy = np.std(fold_accuracies)

# ------------------------------------------------------------
# Print summary
# ------------------------------------------------------------

print("\nFold Validation Accuracy:")
for i, acc in enumerate(fold_accuracies, start=1):
    print(f"Fold {i}: {acc:.2f}%")

print("\n" + "-" * 70)

print(f"Mean CV Accuracy : {mean_accuracy:.2f}%")
print(f"Std CV Accuracy  : {std_accuracy:.2f}%")

print("\nBest fold:")
best_fold_idx = np.argmax(fold_accuracies)

print(
    f"Fold {best_fold_idx + 1} "
    f"-> {fold_accuracies[best_fold_idx]:.2f}%"
)

print(
    f"Best epoch: {fold_epochs[best_fold_idx]}"
)

# ------------------------------------------------------------
# Accuracy plot
# ------------------------------------------------------------

plt.figure(figsize=(8, 5))

plt.bar(
    fold_numbers,
    fold_accuracies
)

plt.axhline(
    mean_accuracy,
    linestyle="--",
    label=f"Mean CV Accuracy: {mean_accuracy:.2f}%"
)

plt.xlabel("Fold")
plt.ylabel("Validation Accuracy (%)")
plt.title("EfficientNet-B0 Stratified 4-Fold Cross-Validation")
plt.xticks(fold_numbers)
plt.legend()
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Best epoch plot
# ------------------------------------------------------------

plt.figure(figsize=(8, 5))

plt.bar(
    fold_numbers,
    fold_epochs
)

plt.xlabel("Fold")
plt.ylabel("Best Epoch")
plt.title("Best Epoch Selected by Early Stopping")
plt.xticks(fold_numbers)
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "=" * 70)
print("✓ CROSS-VALIDATION VISUALIZATION COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 31: TRAINING HISTORY ANALYSIS
# ============================================================

import matplotlib.pyplot as plt

print("=" * 70)
print("TRAINING HISTORY ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# Check available history
# ------------------------------------------------------------

print("\nAvailable history keys:")
print(list(history.keys()))

print(f"\nTotal epochs recorded: {len(history['train_loss'])}")

# ------------------------------------------------------------
# Best validation epoch
# ------------------------------------------------------------

best_history_epoch = (
    history["val_acc"].index(max(history["val_acc"])) + 1
)

best_history_val_acc = max(history["val_acc"])

print(f"\nBest validation accuracy : {best_history_val_acc:.2f}%")
print(f"Best validation epoch    : {best_history_epoch}")

# ------------------------------------------------------------
# Loss curve
# ------------------------------------------------------------

plt.figure(figsize=(8, 5))

plt.plot(
    history["train_loss"],
    label="Training Loss"
)

plt.plot(
    history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Accuracy curve
# ------------------------------------------------------------

plt.figure(figsize=(8, 5))

plt.plot(
    history["train_acc"],
    label="Training Accuracy"
)

plt.plot(
    history["val_acc"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Learning rate curve
# ------------------------------------------------------------

plt.figure(figsize=(8, 5))

plt.plot(
    history["lr"],
    marker="o"
)

plt.xlabel("Epoch")
plt.ylabel("Learning Rate")
plt.title("Learning Rate Schedule")
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "=" * 70)
print("✓ TRAINING HISTORY ANALYSIS COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 32: FINAL TEST SET EVALUATION
# ============================================================

import torch
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
import matplotlib.pyplot as plt

print("=" * 70)
print("FINAL TEST SET EVALUATION")
print("=" * 70)

# ------------------------------------------------------------
# Load best model
# ------------------------------------------------------------

model.load_state_dict(best_model_state)
model = model.to(device)
model.eval()

# ------------------------------------------------------------
# Collect predictions
# ------------------------------------------------------------

all_predictions = []
all_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predictions = torch.max(outputs, 1)

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

# ------------------------------------------------------------
# Convert to numpy
# ------------------------------------------------------------

all_predictions = np.array(all_predictions)
all_labels = np.array(all_labels)

# ------------------------------------------------------------
# Test accuracy
# ------------------------------------------------------------

test_accuracy = accuracy_score(
    all_labels,
    all_predictions
) * 100

print(f"\nTest images   : {len(all_labels)}")
print(f"Test accuracy : {test_accuracy:.2f}%")

# ------------------------------------------------------------
# All 33 class labels
# ------------------------------------------------------------

all_class_indices = np.arange(
    len(class_names)
)

# ------------------------------------------------------------
# Classification report
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)

report = classification_report(
    all_labels,
    all_predictions,
    labels=all_class_indices,
    target_names=class_names,
    digits=4,
    zero_division=0
)

print(report)

# ------------------------------------------------------------
# Confusion matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    all_labels,
    all_predictions,
    labels=all_class_indices
)

print("=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)

print(f"Confusion matrix shape: {cm.shape}")

# ------------------------------------------------------------
# Check classes absent from test set
# ------------------------------------------------------------

test_classes = set(all_labels.tolist())

missing_test_classes = [
    class_names[i]
    for i in all_class_indices
    if i not in test_classes
]

print("\nClasses absent from test set:")

if missing_test_classes:
    for name in missing_test_classes:
        print(f" - {name}")
else:
    print("None")

# ------------------------------------------------------------
# Plot confusion matrix
# ------------------------------------------------------------

plt.figure(figsize=(16, 14))

plt.imshow(cm)

plt.title(
    "EfficientNet-B0 Test Set Confusion Matrix"
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")

plt.xticks(
    all_class_indices,
    class_names,
    rotation=90
)

plt.yticks(
    all_class_indices,
    class_names
)

plt.colorbar()

plt.tight_layout()
plt.show()

print("\n" + "=" * 70)
print("✓ FINAL TEST EVALUATION COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 33: PER-CLASS PERFORMANCE ANALYSIS
# ============================================================

from sklearn.metrics import classification_report
import pandas as pd
import numpy as np

print("=" * 70)
print("PER-CLASS PERFORMANCE ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# Generate classification report as dictionary
# ------------------------------------------------------------

report_dict = classification_report(
    all_labels,
    all_predictions,
    labels=np.arange(len(class_names)),
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

# ------------------------------------------------------------
# Create DataFrame
# ------------------------------------------------------------

class_metrics = []

for class_name in class_names:

    class_metrics.append({
        "Species": class_name,
        "Precision": report_dict[class_name]["precision"],
        "Recall": report_dict[class_name]["recall"],
        "F1-Score": report_dict[class_name]["f1-score"],
        "Support": int(report_dict[class_name]["support"])
    })

class_metrics_df = pd.DataFrame(class_metrics)

# ------------------------------------------------------------
# Sort by F1-score
# ------------------------------------------------------------

class_metrics_df = class_metrics_df.sort_values(
    by="F1-Score",
    ascending=False
).reset_index(drop=True)

# ------------------------------------------------------------
# Display all classes
# ------------------------------------------------------------

print("\nAll Class Performance:")
print(
    class_metrics_df.to_string(
        index=False,
        formatters={
            "Precision": "{:.4f}".format,
            "Recall": "{:.4f}".format,
            "F1-Score": "{:.4f}".format
        }
    )
)

# ------------------------------------------------------------
# Best performing classes
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TOP 5 PERFORMING CLASSES")
print("=" * 70)

print(
    class_metrics_df.head(5).to_string(
        index=False,
        formatters={
            "Precision": "{:.4f}".format,
            "Recall": "{:.4f}".format,
            "F1-Score": "{:.4f}".format
        }
    )
)

# ------------------------------------------------------------
# Weakest performing classes
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("LOWEST 5 PERFORMING CLASSES")
print("=" * 70)

print(
    class_metrics_df.tail(5).to_string(
        index=False,
        formatters={
            "Precision": "{:.4f}".format,
            "Recall": "{:.4f}".format,
            "F1-Score": "{:.4f}".format
        }
    )
)

# ------------------------------------------------------------
# Save metrics
# ------------------------------------------------------------

class_metrics_df.to_csv(
    "per_class_test_metrics.csv",
    index=False
)

print("\n✓ Per-class metrics saved:")
print("  per_class_test_metrics.csv")

print("\n" + "=" * 70)
print("✓ PER-CLASS PERFORMANCE ANALYSIS COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 34: CONFUSION MATRIX
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

print("=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)

# ------------------------------------------------------------
# Generate confusion matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    all_labels,
    all_predictions,
    labels=np.arange(len(class_names))
)

# ------------------------------------------------------------
# Display confusion matrix
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(18, 18))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

disp.plot(
    ax=ax,
    xticks_rotation=90,
    cmap="Blues",
    colorbar=True
)

ax.set_title(
    "EfficientNet-B0 Confusion Matrix",
    fontsize=18,
    pad=20
)

ax.set_xlabel("Predicted Label", fontsize=13)
ax.set_ylabel("True Label", fontsize=13)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Save figure
# ------------------------------------------------------------

fig.savefig(
    "confusion_matrix_efficientnet_b0.png",
    dpi=300,
    bbox_inches="tight"
)

print("\n✓ Confusion matrix generated.")
print("✓ Saved as: confusion_matrix_efficientnet_b0.png")

print("\n" + "=" * 70)
print("✓ CONFUSION MATRIX ANALYSIS COMPLETE")
print("=" * 70)


In [ ]:
# ============================================================
# CELL 35: TRAINING HISTORY VISUALIZATION
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

print("=" * 70)
print("TRAINING HISTORY VISUALIZATION")
print("=" * 70)

# ------------------------------------------------------------
# Get epoch numbers
# ------------------------------------------------------------

epochs_range = np.arange(1, len(history["train_loss"]) + 1)

# ------------------------------------------------------------
# 1. LOSS CURVE
# ------------------------------------------------------------

plt.figure(figsize=(10, 6))

plt.plot(
    epochs_range,
    history["train_loss"],
    marker="o",
    label="Training Loss"
)

plt.plot(
    epochs_range,
    history["val_loss"],
    marker="o",
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

plt.savefig(
    "training_validation_loss.png",
    dpi=300,
    bbox_inches="tight"
)

# ------------------------------------------------------------
# 2. ACCURACY CURVE
# ------------------------------------------------------------

plt.figure(figsize=(10, 6))

plt.plot(
    epochs_range,
    history["train_acc"],
    marker="o",
    label="Training Accuracy"
)

plt.plot(
    epochs_range,
    history["val_acc"],
    marker="o",
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

plt.savefig(
    "training_validation_accuracy.png",
    dpi=300,
    bbox_inches="tight"
)

# ------------------------------------------------------------
# Best epoch information
# ------------------------------------------------------------

best_epoch_index = np.argmax(history["val_acc"])

print("\nBest Validation Performance")
print("-" * 70)

print(
    f"Best Epoch              : "
    f"{best_epoch_index + 1}"
)

print(
    f"Training Accuracy       : "
    f"{history['train_acc'][best_epoch_index]:.2f}%"
)

print(
    f"Validation Accuracy     : "
    f"{history['val_acc'][best_epoch_index]:.2f}%"
)

print(
    f"Training Loss           : "
    f"{history['train_loss'][best_epoch_index]:.4f}"
)

print(
    f"Validation Loss         : "
    f"{history['val_loss'][best_epoch_index]:.4f}"
)

print(
    f"Learning Rate            : "
    f"{history['lr'][best_epoch_index]:.6f}"
)

print("\n✓ Training/validation curves generated.")
print("✓ Saved:")
print("  training_validation_loss.png")
print("  training_validation_accuracy.png")

print("\n" + "=" * 70)
print("✓ TRAINING HISTORY VISUALIZATION COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 36: MISCLASSIFIED TEST IMAGES VISUALIZATION
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
import torch

print("=" * 70)
print("MISCLASSIFIED TEST IMAGES")
print("=" * 70)

# ------------------------------------------------------------
# Collect misclassified images
# ------------------------------------------------------------

model.eval()

misclassified_images = []
misclassified_labels = []
misclassified_predictions = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        predictions = torch.argmax(outputs, dim=1)

        for i in range(len(labels)):

            if predictions[i] != labels[i]:

                misclassified_images.append(
                    images[i].cpu()
                )

                misclassified_labels.append(
                    labels[i].cpu().item()
                )

                misclassified_predictions.append(
                    predictions[i].cpu().item()
                )

print(f"Total test images       : {len(test_dataset)}")
print(f"Misclassified images    : {len(misclassified_images)}")
print(
    f"Correctly classified    : "
    f"{len(test_dataset) - len(misclassified_images)}"
)

# ------------------------------------------------------------
# Display misclassified images
# ------------------------------------------------------------

if len(misclassified_images) > 0:

    num_display = min(12, len(misclassified_images))

    fig, axes = plt.subplots(
        3,
        4,
        figsize=(14, 10)
    )

    axes = axes.flatten()

    for i in range(num_display):

        image = misclassified_images[i]

        # Convert CHW -> HWC
        image = image.permute(1, 2, 0).numpy()

        # Normalize for display
        image = image - image.min()

        if image.max() > 0:
            image = image / image.max()

        axes[i].imshow(image)

        true_label = class_names[
            misclassified_labels[i]
        ]

        predicted_label = class_names[
            misclassified_predictions[i]
        ]

        axes[i].set_title(
            f"True: {true_label}\n"
            f"Pred: {predicted_label}"
        )

        axes[i].axis("off")

    # Hide unused axes
    for i in range(num_display, len(axes)):
        axes[i].axis("off")

    plt.suptitle(
        "Misclassified Test Images",
        fontsize=16
    )

    plt.tight_layout()

    plt.savefig(
        "misclassified_test_images.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    print("\n✓ Misclassified image visualization generated.")
    print("✓ Saved: misclassified_test_images.png")

else:

    print("\n✓ No misclassified images found.")

print("\n" + "=" * 70)
print("✓ MISCLASSIFICATION ANALYSIS COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 37: CONFUSION MATRIX
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix

print("=" * 70)
print("CONFUSION MATRIX ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# Generate confusion matrix
# ------------------------------------------------------------

labels_range = np.arange(len(class_names))

cm = confusion_matrix(
    all_labels,
    all_predictions,
    labels=labels_range
)

print(f"\nConfusion matrix shape : {cm.shape}")
print(f"Number of classes     : {len(class_names)}")

# ------------------------------------------------------------
# Plot confusion matrix
# ------------------------------------------------------------

plt.figure(figsize=(18, 16))

plt.imshow(
    cm,
    interpolation="nearest"
)

plt.title("Confusion Matrix - EfficientNet-B0")
plt.colorbar()

plt.xticks(
    labels_range,
    class_names,
    rotation=90
)

plt.yticks(
    labels_range,
    class_names
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")

# ------------------------------------------------------------
# Add values inside cells
# ------------------------------------------------------------

threshold = cm.max() / 2.0

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):

        if cm[i, j] > 0:

            plt.text(
                j,
                i,
                str(cm[i, j]),
                ha="center",
                va="center",
                fontsize=8,
                color="white" if cm[i, j] > threshold else "black"
            )

plt.tight_layout()

plt.savefig(
    "confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# Identify major confusion pairs
# ------------------------------------------------------------

confusion_pairs = []

for i in range(len(class_names)):
    for j in range(len(class_names)):

        if i != j and cm[i, j] > 0:

            confusion_pairs.append({
                "True Species": class_names[i],
                "Predicted Species": class_names[j],
                "Count": cm[i, j]
            })

confusion_pairs_df = (
    __import__("pandas")
    .DataFrame(confusion_pairs)
    .sort_values(
        by="Count",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 70)
print("TOP CONFUSION PAIRS")
print("=" * 70)

if len(confusion_pairs_df) > 0:

    print(
        confusion_pairs_df.head(10).to_string(
            index=False
        )
    )

else:

    print("✓ No misclassification pairs found.")

# ------------------------------------------------------------
# Save confusion data
# ------------------------------------------------------------

confusion_pairs_df.to_csv(
    "confusion_pairs.csv",
    index=False
)

np.save(
    "confusion_matrix.npy",
    cm
)

print("\n✓ Confusion matrix saved:")
print("  confusion_matrix.png")
print("  confusion_matrix.npy")
print("  confusion_pairs.csv")

print("\n" + "=" * 70)
print("✓ CONFUSION MATRIX ANALYSIS COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 38: PREDICTION CONFIDENCE ANALYSIS
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
import torch

print("=" * 70)
print("PREDICTION CONFIDENCE ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# Collect prediction confidence
# ------------------------------------------------------------

model.eval()

correct_confidences = []
incorrect_confidences = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        probabilities = torch.softmax(outputs, dim=1)

        confidences, predictions = torch.max(
            probabilities,
            dim=1
        )

        for i in range(len(labels)):

            confidence = confidences[i].item() * 100

            if predictions[i] == labels[i]:
                correct_confidences.append(confidence)
            else:
                incorrect_confidences.append(confidence)

# ------------------------------------------------------------
# Convert to NumPy
# ------------------------------------------------------------

correct_confidences = np.array(correct_confidences)
incorrect_confidences = np.array(incorrect_confidences)

# ------------------------------------------------------------
# Print summary
# ------------------------------------------------------------

print(f"\nCorrect predictions   : {len(correct_confidences)}")
print(f"Incorrect predictions : {len(incorrect_confidences)}")

if len(correct_confidences) > 0:

    print(
        f"\nAverage confidence (correct)   : "
        f"{correct_confidences.mean():.2f}%"
    )

if len(incorrect_confidences) > 0:

    print(
        f"Average confidence (incorrect) : "
        f"{incorrect_confidences.mean():.2f}%"
    )

# ------------------------------------------------------------
# Confidence distribution
# ------------------------------------------------------------

plt.figure(figsize=(10, 6))

plt.hist(
    correct_confidences,
    bins=15,
    alpha=0.7,
    label="Correct Predictions"
)

if len(incorrect_confidences) > 0:

    plt.hist(
        incorrect_confidences,
        bins=15,
        alpha=0.7,
        label="Incorrect Predictions"
    )

plt.xlabel("Prediction Confidence (%)")
plt.ylabel("Number of Images")

plt.title(
    "Prediction Confidence Distribution - EfficientNet-B0"
)

plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()

plt.savefig(
    "prediction_confidence_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\n✓ Confidence visualization saved:")
print("  prediction_confidence_distribution.png")

print("\n" + "=" * 70)
print("✓ PREDICTION CONFIDENCE ANALYSIS COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 40: CLASS-WISE F1-SCORE VISUALIZATION
# ============================================================

import matplotlib.pyplot as plt

print("=" * 70)
print("CLASS-WISE F1-SCORE ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# Sort classes by F1-score
# ------------------------------------------------------------

f1_df = class_metrics_df.sort_values(
    by="F1-Score",
    ascending=True
).reset_index(drop=True)

# ------------------------------------------------------------
# Display summary
# ------------------------------------------------------------

print("\nClass-wise F1-Score:")
print(
    f1_df[
        ["Species", "F1-Score", "Support"]
    ].to_string(
        index=False,
        formatters={
            "F1-Score": "{:.4f}".format
        }
    )
)

# ------------------------------------------------------------
# Create visualization
# ------------------------------------------------------------

plt.figure(figsize=(12, 10))

plt.barh(
    f1_df["Species"],
    f1_df["F1-Score"]
)

plt.xlabel("F1-Score")
plt.ylabel("Species")

plt.title(
    "Class-wise F1-Score - EfficientNet-B0"
)

plt.xlim(0, 1.05)

plt.grid(
    axis="x",
    alpha=0.3
)

# ------------------------------------------------------------
# Add F1 values
# ------------------------------------------------------------

for i, value in enumerate(f1_df["F1-Score"]):

    plt.text(
        value + 0.01,
        i,
        f"{value:.2f}",
        va="center",
        fontsize=8
    )

plt.tight_layout()

plt.savefig(
    "classwise_f1_score.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\n✓ Class-wise F1 visualization saved:")
print("  classwise_f1_score.png")

print("\n" + "=" * 70)
print("✓ CLASS-WISE F1-SCORE ANALYSIS COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 41: CLASS SUPPORT vs F1-SCORE ANALYSIS
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

print("=" * 70)
print("CLASS SUPPORT vs F1-SCORE ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# Prepare data
# ------------------------------------------------------------

support = class_metrics_df["Support"].values
f1_scores = class_metrics_df["F1-Score"].values
species = class_metrics_df["Species"].values

# ------------------------------------------------------------
# Print support summary
# ------------------------------------------------------------

print("\nClass Support Statistics:")
print(f"Total classes        : {len(species)}")
print(f"Total test images    : {support.sum()}")
print(f"Minimum class support: {support.min()}")
print(f"Maximum class support: {support.max()}")
print(f"Average class support: {support.mean():.2f}")

# ------------------------------------------------------------
# Create scatter plot
# ------------------------------------------------------------

plt.figure(figsize=(10, 7))

plt.scatter(
    support,
    f1_scores,
    alpha=0.75
)

plt.xlabel("Test Set Support (Number of Images)")
plt.ylabel("F1-Score")

plt.title(
    "Class Support vs F1-Score - EfficientNet-B0"
)

plt.ylim(-0.05, 1.05)
plt.grid(alpha=0.3)

# ------------------------------------------------------------
# Add class labels
# ------------------------------------------------------------

for i in range(len(species)):

    plt.annotate(
        species[i],
        (support[i], f1_scores[i]),
        xytext=(5, 4),
        textcoords="offset points",
        fontsize=7
    )

plt.tight_layout()

plt.savefig(
    "class_support_vs_f1.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\n✓ Support vs F1 visualization saved:")
print("  class_support_vs_f1.png")

print("\n" + "=" * 70)
print("✓ CLASS SUPPORT vs F1-SCORE ANALYSIS COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 42: FINAL MODEL PERFORMANCE SUMMARY
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
import pandas as pd

print("=" * 70)
print("FINAL MODEL PERFORMANCE SUMMARY")
print("=" * 70)

# ------------------------------------------------------------
# Calculate overall metrics
# ------------------------------------------------------------

accuracy = accuracy_score(
    all_labels,
    all_predictions
)

macro_precision = precision_score(
    all_labels,
    all_predictions,
    labels=np.arange(len(class_names)),
    average="macro",
    zero_division=0
)

macro_recall = recall_score(
    all_labels,
    all_predictions,
    labels=np.arange(len(class_names)),
    average="macro",
    zero_division=0
)

macro_f1 = f1_score(
    all_labels,
    all_predictions,
    labels=np.arange(len(class_names)),
    average="macro",
    zero_division=0
)

weighted_precision = precision_score(
    all_labels,
    all_predictions,
    average="weighted",
    zero_division=0
)

weighted_recall = recall_score(
    all_labels,
    all_predictions,
    average="weighted",
    zero_division=0
)

weighted_f1 = f1_score(
    all_labels,
    all_predictions,
    average="weighted",
    zero_division=0
)

# ------------------------------------------------------------
# Create summary table
# ------------------------------------------------------------

final_metrics = pd.DataFrame({
    "Metric": [
        "Test Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1-Score",
        "Weighted Precision",
        "Weighted Recall",
        "Weighted F1-Score"
    ],
    "Score": [
        accuracy,
        macro_precision,
        macro_recall,
        macro_f1,
        weighted_precision,
        weighted_recall,
        weighted_f1
    ]
})

# ------------------------------------------------------------
# Display metrics
# ------------------------------------------------------------

print("\nOverall Test Performance:")

print(
    final_metrics.to_string(
        index=False,
        formatters={
            "Score": "{:.4f}".format
        }
    )
)

print("\nPercentage Format:")

for _, row in final_metrics.iterrows():

    print(
        f"{row['Metric']:25s}: "
        f"{row['Score'] * 100:.2f}%"
    )

# ------------------------------------------------------------
# Save summary
# ------------------------------------------------------------

final_metrics.to_csv(
    "final_model_performance.csv",
    index=False
)

print("\n✓ Final performance metrics saved:")
print("  final_model_performance.csv")

print("\n" + "=" * 70)
print("✓ FINAL MODEL PERFORMANCE SUMMARY COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 43: FINAL MODEL EXPORT & METADATA
# ============================================================

import torch
import json
import os

print("=" * 70)
print("FINAL MODEL EXPORT")
print("=" * 70)

# ------------------------------------------------------------
# Make sure the best model is loaded
# ------------------------------------------------------------

model.load_state_dict(best_model_state)
model.eval()

# ------------------------------------------------------------
# Save complete model checkpoint
# ------------------------------------------------------------

final_checkpoint = {
    "model_state_dict": model.state_dict(),
    "class_names": class_names,
    "num_classes": len(class_names),
    "model_name": "EfficientNet-B0",
    "best_validation_accuracy": best_val_acc,
    "best_epoch": best_epoch,
    "test_accuracy": accuracy,
    "macro_f1": macro_f1,
    "weighted_f1": weighted_f1
}

torch.save(
    final_checkpoint,
    "final_efficientnet_b0_leaf_classifier.pth"
)

# ------------------------------------------------------------
# Save class names separately
# ------------------------------------------------------------

with open(
    "class_names.json",
    "w"
) as f:

    json.dump(
        class_names,
        f,
        indent=4
    )

# ------------------------------------------------------------
# Save model summary
# ------------------------------------------------------------

model_info = {
    "Model": "EfficientNet-B0",
    "Number of Classes": len(class_names),
    "Best Validation Accuracy": round(best_val_acc, 4),
    "Best Epoch": best_epoch,
    "Test Accuracy": round(accuracy, 4),
    "Macro Precision": round(macro_precision, 4),
    "Macro Recall": round(macro_recall, 4),
    "Macro F1-Score": round(macro_f1, 4),
    "Weighted Precision": round(weighted_precision, 4),
    "Weighted Recall": round(weighted_recall, 4),
    "Weighted F1-Score": round(weighted_f1, 4)
}

with open(
    "model_summary.json",
    "w"
) as f:

    json.dump(
        model_info,
        f,
        indent=4
    )

# ------------------------------------------------------------
# Check saved files
# ------------------------------------------------------------

files_to_check = [
    "final_efficientnet_b0_leaf_classifier.pth",
    "class_names.json",
    "model_summary.json",
    "best_efficientnet_b0.pth"
]

print("\nSaved Model Files:")

for file_name in files_to_check:

    if os.path.exists(file_name):

        size_mb = os.path.getsize(file_name) / (1024 ** 2)

        print(
            f"✓ {file_name} "
            f"({size_mb:.2f} MB)"
        )

    else:

        print(f"⚠ {file_name} not found")

print("\n" + "=" * 70)
print("✓ FINAL MODEL EXPORT COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 44: RESNET50 MODEL SETUP
# ============================================================

import torch
import torch.nn as nn
from torchvision import models

print("=" * 70)
print("RESNET50 MODEL SETUP")
print("=" * 70)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

num_classes = len(class_names)

print(f"Device           : {device}")
print(f"Number of classes: {num_classes}")

# ------------------------------------------------------------
# Load ImageNet pretrained ResNet50
# ------------------------------------------------------------

print("\nLoading ResNet50...")

try:

    weights = models.ResNet50_Weights.DEFAULT

    model = models.resnet50(weights=weights)

    print("✓ ImageNet pretrained weights loaded.")

except Exception as e:

    print("⚠ Pretrained weights could not be loaded.")
    print(f"Reason: {type(e).__name__}")
    print("STOP: Do not train from scratch.")

    raise e

# ------------------------------------------------------------
# Freeze pretrained backbone
# ------------------------------------------------------------

for param in model.parameters():
    param.requires_grad = False

# ------------------------------------------------------------
# Replace final classifier
# ------------------------------------------------------------

in_features = model.fc.in_features

model.fc = nn.Linear(
    in_features,
    num_classes
)

# Make only classifier trainable
for param in model.fc.parameters():
    param.requires_grad = True

# ------------------------------------------------------------
# Move model to device
# ------------------------------------------------------------

model = model.to(device)

# ------------------------------------------------------------
# Parameter summary
# ------------------------------------------------------------

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

frozen_params = total_params - trainable_params

print("\nResNet50 classifier updated:")
print(f"Input features      : {in_features}")
print(f"Output classes      : {num_classes}")

print("\nParameter summary:")
print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters   : {frozen_params:,}")

print("\n" + "=" * 70)
print("RESNET50 MODEL READY ✓")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 45: RESNET50 STRATIFIED 4-FOLD CROSS-VALIDATION
# ============================================================

import copy
import time
import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import StratifiedKFold


print("=" * 70)
print("RESNET50 STRATIFIED 4-FOLD CROSS-VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

NUM_FOLDS = 4
MAX_EPOCHS = 150
EARLY_STOPPING_PATIENCE = 10

INITIAL_LR = 0.001
WEIGHT_DECAY = 0.0001
BATCH_SIZE = 32

print(f"\nNumber of folds        : {NUM_FOLDS}")
print(f"Maximum epochs         : {MAX_EPOCHS}")
print(f"Early stopping patience: {EARLY_STOPPING_PATIENCE}")
print(f"Initial learning rate  : {INITIAL_LR}")
print(f"Weight decay           : {WEIGHT_DECAY}")
print(f"Batch size             : {BATCH_SIZE}")
print(f"Device                 : {device}")

# ------------------------------------------------------------
# Extract labels from custom LeafDataset
# ------------------------------------------------------------

print("\nExtracting dataset labels...")

all_dataset_labels = []

for i in range(len(train_dataset)):

    label = train_dataset[i][1]

    if torch.is_tensor(label):
        label = label.item()

    all_dataset_labels.append(int(label))

all_dataset_labels = np.array(all_dataset_labels)

print(f"Training images        : {len(train_dataset)}")
print(f"Number of classes      : {len(np.unique(all_dataset_labels))}")

# ------------------------------------------------------------
# Check class distribution
# ------------------------------------------------------------

unique_labels, label_counts = np.unique(
    all_dataset_labels,
    return_counts=True
)

print(
    f"Minimum class samples  : "
    f"{label_counts.min()}"
)

print(
    f"Maximum class samples  : "
    f"{label_counts.max()}"
)

# ------------------------------------------------------------
# Stratified K-Fold
# ------------------------------------------------------------

skf = StratifiedKFold(
    n_splits=NUM_FOLDS,
    shuffle=True,
    random_state=42
)

# ------------------------------------------------------------
# Save initial ResNet50 weights
# ------------------------------------------------------------

initial_model_state = copy.deepcopy(
    model.state_dict()
)

# ------------------------------------------------------------
# Cross-validation storage
# ------------------------------------------------------------

fold_results = []
fold_histories = []

best_overall_val_acc = 0.0
best_overall_fold = 0
best_overall_epoch = 0
best_overall_model_state = None

# ============================================================
# START CROSS-VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("STARTING CROSS-VALIDATION")
print("=" * 70)

for fold, (train_indices, val_indices) in enumerate(
    skf.split(
        np.zeros(len(all_dataset_labels)),
        all_dataset_labels
    ),
    start=1
):

    print("\n" + "=" * 70)
    print(f"FOLD {fold}/{NUM_FOLDS}")
    print("=" * 70)

    print(
        f"Fold training images   : "
        f"{len(train_indices)}"
    )

    print(
        f"Fold validation images : "
        f"{len(val_indices)}"
    )

    # --------------------------------------------------------
    # Create fold datasets
    # --------------------------------------------------------

    fold_train_dataset = Subset(
        train_dataset,
        train_indices
    )

    fold_val_dataset = Subset(
        train_dataset,
        val_indices
    )

    # --------------------------------------------------------
    # Create fold DataLoaders
    # --------------------------------------------------------

    fold_train_loader = DataLoader(
        fold_train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=torch.cuda.is_available()
    )

    fold_val_loader = DataLoader(
        fold_val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=torch.cuda.is_available()
    )

    # --------------------------------------------------------
    # Reset model to original pretrained state
    # --------------------------------------------------------

    model.load_state_dict(
        initial_model_state
    )

    model = model.to(device)

    # --------------------------------------------------------
    # Make sure only classifier is trainable
    # --------------------------------------------------------

    for param in model.parameters():
        param.requires_grad = False

    for param in model.fc.parameters():
        param.requires_grad = True

    # --------------------------------------------------------
    # Loss function
    # --------------------------------------------------------

    criterion = nn.CrossEntropyLoss()

    # --------------------------------------------------------
    # Optimizer
    # --------------------------------------------------------

    optimizer = torch.optim.AdamW(
        filter(
            lambda p: p.requires_grad,
            model.parameters()
        ),
        lr=INITIAL_LR,
        weight_decay=WEIGHT_DECAY
    )

    # --------------------------------------------------------
    # Learning rate scheduler
    # --------------------------------------------------------

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=3
    )

    # --------------------------------------------------------
    # Fold history
    # --------------------------------------------------------

    fold_history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
        "lr": []
    }

    # --------------------------------------------------------
    # Early stopping variables
    # --------------------------------------------------------

    best_fold_val_acc = 0.0
    best_fold_val_loss = float("inf")
    best_fold_epoch = 0

    best_fold_model_state = copy.deepcopy(
        model.state_dict()
    )

    epochs_without_improvement = 0

    # ========================================================
    # TRAINING LOOP
    # ========================================================

    for epoch in range(1, MAX_EPOCHS + 1):

        start_time = time.time()

        # ====================================================
        # TRAINING
        # ====================================================

        model.train()

        running_train_loss = 0.0
        train_correct = 0
        train_total = 0

        for images, labels in fold_train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            loss.backward()

            optimizer.step()

            running_train_loss += (
                loss.item() *
                images.size(0)
            )

            _, predicted = torch.max(
                outputs,
                1
            )

            train_total += labels.size(0)

            train_correct += (
                predicted == labels
            ).sum().item()

        train_loss = (
            running_train_loss /
            train_total
        )

        train_acc = (
            100.0 *
            train_correct /
            train_total
        )

        # ====================================================
        # VALIDATION
        # ====================================================

        model.eval()

        running_val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():

            for images, labels in fold_val_loader:

                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)

                loss = criterion(
                    outputs,
                    labels
                )

                running_val_loss += (
                    loss.item() *
                    images.size(0)
                )

                _, predicted = torch.max(
                    outputs,
                    1
                )

                val_total += labels.size(0)

                val_correct += (
                    predicted == labels
                ).sum().item()

        val_loss = (
            running_val_loss /
            val_total
        )

        val_acc = (
            100.0 *
            val_correct /
            val_total
        )

        # ====================================================
        # LEARNING RATE SCHEDULER
        # ====================================================

        scheduler.step(val_loss)

        current_lr = (
            optimizer.param_groups[0]["lr"]
        )

        # ====================================================
        # SAVE HISTORY
        # ====================================================

        fold_history["train_loss"].append(
            train_loss
        )

        fold_history["train_acc"].append(
            train_acc
        )

        fold_history["val_loss"].append(
            val_loss
        )

        fold_history["val_acc"].append(
            val_acc
        )

        fold_history["lr"].append(
            current_lr
        )

        # ====================================================
        # CHECK IMPROVEMENT
        # ====================================================

        if val_acc > best_fold_val_acc:

            best_fold_val_acc = val_acc
            best_fold_val_loss = val_loss
            best_fold_epoch = epoch

            best_fold_model_state = copy.deepcopy(
                model.state_dict()
            )

            epochs_without_improvement = 0

            best_marker = " ★ BEST"

        else:

            epochs_without_improvement += 1

            best_marker = ""

        # ====================================================
        # EPOCH RESULT
        # ====================================================

        epoch_time = (
            time.time() -
            start_time
        )

        print(
            f"Fold {fold} | "
            f"Epoch [{epoch:03d}/{MAX_EPOCHS}] | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.2f}% | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.2f}% | "
            f"LR: {current_lr:.6f} | "
            f"Time: {epoch_time:.1f}s"
            f"{best_marker}"
        )

        # ====================================================
        # EARLY STOPPING
        # ====================================================

        if (
            epochs_without_improvement
            >= EARLY_STOPPING_PATIENCE
        ):

            print(
                f"\n⚠ Early stopping triggered "
                f"at epoch {epoch}."
            )

            print(
                f"Best epoch: "
                f"{best_fold_epoch}"
            )

            break

    # ========================================================
    # RESTORE BEST MODEL FOR THIS FOLD
    # ========================================================

    model.load_state_dict(
        best_fold_model_state
    )

    # --------------------------------------------------------
    # Save fold model
    # --------------------------------------------------------

    fold_model_filename = (
        f"resnet50_fold_{fold}.pth"
    )

    torch.save(
        best_fold_model_state,
        fold_model_filename
    )

    # --------------------------------------------------------
    # Save fold results
    # --------------------------------------------------------

    fold_results.append({
        "fold": fold,
        "best_epoch": best_fold_epoch,
        "best_val_loss": best_fold_val_loss,
        "best_val_acc": best_fold_val_acc
    })

    fold_histories.append(
        fold_history
    )

    # --------------------------------------------------------
    # Overall best model
    # --------------------------------------------------------

    if (
        best_fold_val_acc >
        best_overall_val_acc
    ):

        best_overall_val_acc = (
            best_fold_val_acc
        )

        best_overall_fold = fold

        best_overall_epoch = (
            best_fold_epoch
        )

        best_overall_model_state = (
            copy.deepcopy(
                best_fold_model_state
            )
        )

    # --------------------------------------------------------
    # Fold summary
    # --------------------------------------------------------

    print("\nFold summary:")
    print(
        f"Best epoch      : "
        f"{best_fold_epoch}"
    )

    print(
        f"Best Val Loss   : "
        f"{best_fold_val_loss:.4f}"
    )

    print(
        f"Best Val Acc    : "
        f"{best_fold_val_acc:.2f}%"
    )

    print(
        f"Model saved     : "
        f"{fold_model_filename}"
    )


# ============================================================
# CROSS-VALIDATION SUMMARY
# ============================================================

fold_accuracies = np.array([
    result["best_val_acc"]
    for result in fold_results
])

mean_cv_accuracy = (
    fold_accuracies.mean()
)

std_cv_accuracy = (
    fold_accuracies.std()
)

print("\n" + "=" * 70)
print("CROSS-VALIDATION COMPLETE")
print("=" * 70)

print("\nFold validation accuracies:")

for result in fold_results:

    print(
        f"Fold {result['fold']} : "
        f"{result['best_val_acc']:.2f}% "
        f"(Epoch {result['best_epoch']})"
    )

print(
    "\n" + "-" * 70
)

print(
    f"Mean CV Accuracy : "
    f"{mean_cv_accuracy:.2f}%"
)

print(
    f"Std CV Accuracy  : "
    f"{std_cv_accuracy:.2f}%"
)

print(
    f"\nBest overall fold : "
    f"Fold {best_overall_fold}"
)

print(
    f"Best overall epoch: "
    f"{best_overall_epoch}"
)

print(
    f"Best CV accuracy  : "
    f"{best_overall_val_acc:.2f}%"
)

# ------------------------------------------------------------
# Save overall best ResNet50 model
# ------------------------------------------------------------

if best_overall_model_state is not None:

    torch.save(
        best_overall_model_state,
        "best_resnet50.pth"
    )

    model.load_state_dict(
        best_overall_model_state
    )

print(
    "\n✓ Best ResNet50 model saved:"
)

print(
    "  best_resnet50.pth"
)

print("\n" + "=" * 70)
print(
    "✓ RESNET50 STRATIFIED 4-FOLD "
    "+ EARLY STOPPING COMPLETE"
)
print("=" * 70)

In [ ]:
# ============================================================
# CELL 46: RESNET50 CROSS-VALIDATION RESULTS
#          & VISUALIZATION
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 70)
print("RESNET50 CROSS-VALIDATION RESULTS")
print("=" * 70)

# ------------------------------------------------------------
# Create fold results DataFrame
# ------------------------------------------------------------

fold_results_df = pd.DataFrame(fold_results)

# ------------------------------------------------------------
# Display fold-wise results
# ------------------------------------------------------------

print("\nFold-wise Validation Performance:")
print("-" * 70)

print(
    fold_results_df.to_string(
        index=False,
        formatters={
            "best_val_loss": "{:.4f}".format,
            "best_val_acc": "{:.2f}%".format
        }
    )
)

# ------------------------------------------------------------
# Calculate summary statistics
# ------------------------------------------------------------

fold_accuracies = (
    fold_results_df["best_val_acc"].values
)

mean_accuracy = fold_accuracies.mean()
std_accuracy = fold_accuracies.std()

min_accuracy = fold_accuracies.min()
max_accuracy = fold_accuracies.max()

print("\n" + "=" * 70)
print("CROSS-VALIDATION SUMMARY")
print("=" * 70)

print(
    f"Mean CV Accuracy : "
    f"{mean_accuracy:.2f}%"
)

print(
    f"Std CV Accuracy  : "
    f"{std_accuracy:.2f}%"
)

print(
    f"Minimum Accuracy  : "
    f"{min_accuracy:.2f}%"
)

print(
    f"Maximum Accuracy  : "
    f"{max_accuracy:.2f}%"
)

# ------------------------------------------------------------
# Best fold
# ------------------------------------------------------------

best_fold_index = fold_results_df[
    "best_val_acc"
].idxmax()

best_fold_row = fold_results_df.loc[
    best_fold_index
]

print("\nBest Fold:")
print(
    f"Fold          : "
    f"{int(best_fold_row['fold'])}"
)

print(
    f"Best Epoch    : "
    f"{int(best_fold_row['best_epoch'])}"
)

print(
    f"Validation Acc: "
    f"{best_fold_row['best_val_acc']:.2f}%"
)

# ------------------------------------------------------------
# Save fold results
# ------------------------------------------------------------

fold_results_df.to_csv(
    "resnet50_cv_results.csv",
    index=False
)

print("\n✓ Fold results saved:")
print("  resnet50_cv_results.csv")

# ============================================================
# VISUALIZATION 1: FOLD ACCURACY
# ============================================================

plt.figure(figsize=(10, 6))

plt.bar(
    fold_results_df["fold"].astype(str),
    fold_results_df["best_val_acc"]
)

plt.axhline(
    mean_accuracy,
    linestyle="--",
    label=f"Mean CV Accuracy ({mean_accuracy:.2f}%)"
)

plt.xlabel("Fold")
plt.ylabel("Validation Accuracy (%)")

plt.title(
    "ResNet50 4-Fold Cross-Validation Accuracy"
)

plt.legend()

plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "resnet50_cv_accuracy.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# VISUALIZATION 2: BEST EPOCH PER FOLD
# ============================================================

plt.figure(figsize=(10, 6))

plt.bar(
    fold_results_df["fold"].astype(str),
    fold_results_df["best_epoch"]
)

plt.xlabel("Fold")
plt.ylabel("Best Epoch")

plt.title(
    "ResNet50 Best Epoch per Fold"
)

plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "resnet50_best_epoch_per_fold.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\n✓ Visualizations saved:")
print("  resnet50_cv_accuracy.png")
print("  resnet50_best_epoch_per_fold.png")

print("\n" + "=" * 70)
print("✓ RESNET50 CV RESULTS & VISUALIZATION COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 47: RESNET50 TRAINING HISTORY
#          TRAIN vs VALIDATION LOSS & ACCURACY
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

print("=" * 70)
print("RESNET50 TRAINING HISTORY")
print("=" * 70)

# ------------------------------------------------------------
# Check available history
# ------------------------------------------------------------

print(f"\nNumber of folds with history: {len(fold_histories)}")

# ------------------------------------------------------------
# Plot Loss for each fold
# ------------------------------------------------------------

plt.figure(figsize=(12, 7))

for fold_idx, history in enumerate(fold_histories, start=1):

    epochs = range(1, len(history["train_loss"]) + 1)

    plt.plot(
        epochs,
        history["train_loss"],
        label=f"Fold {fold_idx} Train Loss"
    )

    plt.plot(
        epochs,
        history["val_loss"],
        linestyle="--",
        label=f"Fold {fold_idx} Val Loss"
    )

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("ResNet50 4-Fold Cross-Validation: Training vs Validation Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    "resnet50_training_validation_loss.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# Plot Accuracy for each fold
# ------------------------------------------------------------

plt.figure(figsize=(12, 7))

for fold_idx, history in enumerate(fold_histories, start=1):

    epochs = range(1, len(history["train_acc"]) + 1)

    plt.plot(
        epochs,
        history["train_acc"],
        label=f"Fold {fold_idx} Train Accuracy"
    )

    plt.plot(
        epochs,
        history["val_acc"],
        linestyle="--",
        label=f"Fold {fold_idx} Val Accuracy"
    )

plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("ResNet50 4-Fold Cross-Validation: Training vs Validation Accuracy")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    "resnet50_training_validation_accuracy.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# Print best epoch information
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BEST EPOCH INFORMATION")
print("=" * 70)

for fold_idx, history in enumerate(fold_histories, start=1):

    best_epoch = np.argmax(history["val_acc"]) + 1
    best_val_acc = max(history["val_acc"])
    best_val_loss = history["val_loss"][best_epoch - 1]

    print(
        f"Fold {fold_idx}: "
        f"Best Epoch = {best_epoch}, "
        f"Val Accuracy = {best_val_acc:.2f}%, "
        f"Val Loss = {best_val_loss:.4f}"
    )

# ------------------------------------------------------------
# Save history
# ------------------------------------------------------------

print("\n✓ Training history plots saved:")
print("  resnet50_training_validation_loss.png")
print("  resnet50_training_validation_accuracy.png")

print("\n" + "=" * 70)
print("✓ RESNET50 TRAINING HISTORY COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 48: RESNET50 FINAL TEST EVALUATION
# ============================================================

import torch
import numpy as np
from sklearn.metrics import accuracy_score

print("=" * 70)
print("RESNET50 FINAL TEST EVALUATION")
print("=" * 70)

# ------------------------------------------------------------
# Load Best ResNet50 Model
# ------------------------------------------------------------

best_model_path = "best_resnet50.pth"

checkpoint = torch.load(
    best_model_path,
    map_location=device
)

# Handle checkpoint format
if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])
else:
    model.load_state_dict(checkpoint)

model = model.to(device)
model.eval()

print(f"\n✓ Loaded best ResNet50 model from: {best_model_path}")

# ------------------------------------------------------------
# Test Evaluation
# ------------------------------------------------------------

test_predictions = []
test_true_labels = []
test_probabilities = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        probabilities = torch.softmax(outputs, dim=1)
        predictions = torch.argmax(outputs, dim=1)

        test_predictions.extend(
            predictions.cpu().numpy()
        )

        test_true_labels.extend(
            labels.cpu().numpy()
        )

        test_probabilities.extend(
            probabilities.cpu().numpy()
        )

# Convert to NumPy arrays
test_predictions = np.array(test_predictions)
test_true_labels = np.array(test_true_labels)
test_probabilities = np.array(test_probabilities)

# ------------------------------------------------------------
# Calculate Accuracy
# ------------------------------------------------------------

test_accuracy = accuracy_score(
    test_true_labels,
    test_predictions
) * 100

print("\n" + "=" * 70)
print("FINAL TEST RESULT")
print("=" * 70)

print(f"Test Samples  : {len(test_true_labels)}")
print(f"Test Accuracy  : {test_accuracy:.2f}%")

# ------------------------------------------------------------
# Prediction Statistics
# ------------------------------------------------------------

correct_predictions = (
    test_predictions == test_true_labels
).sum()

incorrect_predictions = (
    test_predictions != test_true_labels
).sum()

print(f"\nCorrect Predictions   : {correct_predictions}")
print(f"Incorrect Predictions : {incorrect_predictions}")

print("\n" + "=" * 70)
print("✓ RESNET50 FINAL TEST EVALUATION COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 49: RESNET50 CLASSIFICATION REPORT
# ============================================================

from sklearn.metrics import (
    classification_report,
    precision_score,
    recall_score,
    f1_score
)
import pandas as pd
import numpy as np

print("=" * 70)
print("RESNET50 CLASSIFICATION REPORT")
print("=" * 70)

# ------------------------------------------------------------
# Classification Report
# ------------------------------------------------------------

report_dict = classification_report(
    test_true_labels,
    test_predictions,
    labels=np.arange(len(class_names)),
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report_dict).transpose()

# ------------------------------------------------------------
# Display Class-wise Results
# ------------------------------------------------------------

class_report_df = report_df.loc[
    class_names,
    ["precision", "recall", "f1-score", "support"]
].copy()

class_report_df["support"] = class_report_df["support"].astype(int)

print("\nClass-wise Performance:")
print("-" * 70)

print(
    class_report_df.to_string(
        formatters={
            "precision": "{:.4f}".format,
            "recall": "{:.4f}".format,
            "f1-score": "{:.4f}".format
        }
    )
)

# ------------------------------------------------------------
# Overall Metrics
# ------------------------------------------------------------

macro_precision = report_dict["macro avg"]["precision"]
macro_recall = report_dict["macro avg"]["recall"]
macro_f1 = report_dict["macro avg"]["f1-score"]

weighted_precision = report_dict["weighted avg"]["precision"]
weighted_recall = report_dict["weighted avg"]["recall"]
weighted_f1 = report_dict["weighted avg"]["f1-score"]

print("\n" + "=" * 70)
print("OVERALL CLASSIFICATION METRICS")
print("=" * 70)

print(f"Macro Precision   : {macro_precision * 100:.2f}%")
print(f"Macro Recall      : {macro_recall * 100:.2f}%")
print(f"Macro F1-score    : {macro_f1 * 100:.2f}%")

print(f"\nWeighted Precision: {weighted_precision * 100:.2f}%")
print(f"Weighted Recall   : {weighted_recall * 100:.2f}%")
print(f"Weighted F1-score : {weighted_f1 * 100:.2f}%")

# ------------------------------------------------------------
# Save Classification Report
# ------------------------------------------------------------

class_report_df.to_csv(
    "resnet50_classification_report.csv"
)

print("\n✓ Classification report saved:")
print("  resnet50_classification_report.csv")

print("\n" + "=" * 70)
print("✓ RESNET50 CLASSIFICATION REPORT COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 50: RESNET50 CONFUSION MATRIX
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

print("=" * 70)
print("RESNET50 CONFUSION MATRIX")
print("=" * 70)

# ------------------------------------------------------------
# Generate Confusion Matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    test_true_labels,
    test_predictions,
    labels=np.arange(len(class_names))
)

print(f"\nConfusion Matrix Shape: {cm.shape}")
print(f"Number of Classes      : {len(class_names)}")

# ------------------------------------------------------------
# Plot Confusion Matrix
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(18, 16))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

disp.plot(
    ax=ax,
    cmap="Blues",
    xticks_rotation=90,
    values_format="d",
    colorbar=True
)

ax.set_title(
    "ResNet50 Confusion Matrix",
    fontsize=16,
    pad=20
)

ax.set_xlabel(
    "Predicted Label",
    fontsize=12
)

ax.set_ylabel(
    "True Label",
    fontsize=12
)

plt.tight_layout()

plt.savefig(
    "resnet50_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# Analyze Misclassifications
# ------------------------------------------------------------

misclassified_pairs = []

for true_idx in range(len(class_names)):
    for pred_idx in range(len(class_names)):

        if true_idx != pred_idx and cm[true_idx, pred_idx] > 0:

            misclassified_pairs.append({
                "True Class": class_names[true_idx],
                "Predicted Class": class_names[pred_idx],
                "Count": cm[true_idx, pred_idx]
            })

misclassified_pairs_df = pd.DataFrame(
    misclassified_pairs
)

# ------------------------------------------------------------
# Display Most Common Confusions
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MOST COMMON MISCLASSIFICATIONS")
print("=" * 70)

if len(misclassified_pairs_df) > 0:

    misclassified_pairs_df = (
        misclassified_pairs_df
        .sort_values("Count", ascending=False)
        .reset_index(drop=True)
    )

    print(
        misclassified_pairs_df.head(15).to_string(
            index=False
        )
    )

    misclassified_pairs_df.to_csv(
        "resnet50_misclassification_pairs.csv",
        index=False
    )

else:
    print("No misclassified class pairs found.")

print("\n✓ Confusion matrix saved:")
print("  resnet50_confusion_matrix.png")

print("\n" + "=" * 70)
print("✓ RESNET50 CONFUSION MATRIX COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 51: RESNET50 MISCLASSIFIED IMAGES
#          ACTUAL vs PREDICTED
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

print("=" * 70)
print("RESNET50 MISCLASSIFIED IMAGES")
print("=" * 70)

# ------------------------------------------------------------
# Find Misclassified Samples
# ------------------------------------------------------------

misclassified_indices = np.where(
    test_predictions != test_true_labels
)[0]

print(f"\nTotal Test Images          : {len(test_true_labels)}")
print(f"Correct Predictions        : {len(test_true_labels) - len(misclassified_indices)}")
print(f"Misclassified Predictions  : {len(misclassified_indices)}")

# ------------------------------------------------------------
# Get confidence for each prediction
# ------------------------------------------------------------

misclassified_data = []

for idx in misclassified_indices:

    true_label = test_true_labels[idx]
    pred_label = test_predictions[idx]

    confidence = test_probabilities[idx][pred_label] * 100

    misclassified_data.append({
        "index": idx,
        "true_label": true_label,
        "pred_label": pred_label,
        "true_class": class_names[true_label],
        "pred_class": class_names[pred_label],
        "confidence": confidence
    })

# Sort by confidence, highest first
misclassified_data = sorted(
    misclassified_data,
    key=lambda x: x["confidence"],
    reverse=True
)

# ------------------------------------------------------------
# Display Misclassified Images
# ------------------------------------------------------------

num_to_show = min(12, len(misclassified_data))

if num_to_show > 0:

    fig, axes = plt.subplots(
        3, 4,
        figsize=(16, 13)
    )

    axes = axes.flatten()

    for plot_idx in range(num_to_show):

        item = misclassified_data[plot_idx]

        dataset_idx = item["index"]

        # Get image and label from test dataset
        image, _ = test_dataset[dataset_idx]

        # Convert tensor to image
        if torch.is_tensor(image):

            image = image.cpu()

            # CHW -> HWC
            image = image.permute(1, 2, 0).numpy()

            # Normalize to 0-1 for display
            image = image - image.min()

            if image.max() > 0:
                image = image / image.max()

        axes[plot_idx].imshow(image)

        axes[plot_idx].set_title(
            f"Actual: {item['true_class']}\n"
            f"Predicted: {item['pred_class']}\n"
            f"Confidence: {item['confidence']:.1f}%",
            fontsize=10
        )

        axes[plot_idx].axis("off")

    # Hide unused axes
    for idx in range(num_to_show, len(axes)):
        axes[idx].axis("off")

    plt.suptitle(
        "ResNet50 Misclassified Test Images",
        fontsize=16
    )

    plt.tight_layout()

    plt.savefig(
        "resnet50_misclassified_images.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

else:
    print("\n✓ No misclassified images found!")

# ------------------------------------------------------------
# Misclassification Table
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MISCLASSIFICATION DETAILS")
print("=" * 70)

if len(misclassified_data) > 0:

    misclassified_df = pd.DataFrame(misclassified_data)

    display(
        misclassified_df[
            [
                "index",
                "true_class",
                "pred_class",
                "confidence"
            ]
        ].style.format({
            "confidence": "{:.2f}%"
        })
    )

    misclassified_df.to_csv(
        "resnet50_misclassified_images.csv",
        index=False
    )

    print("\n✓ Misclassification details saved:")
    print("  resnet50_misclassified_images.csv")

print("\n✓ Visualization saved:")
print("  resnet50_misclassified_images.png")

print("\n" + "=" * 70)
print("✓ RESNET50 MISCLASSIFIED IMAGE ANALYSIS COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 52: RESNET50 PREDICTION CONFIDENCE ANALYSIS
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 70)
print("RESNET50 PREDICTION CONFIDENCE ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# Prediction Confidence
# ------------------------------------------------------------

predicted_confidences = (
    np.max(test_probabilities, axis=1) * 100
)

correct_mask = (
    test_predictions == test_true_labels
)

correct_confidences = predicted_confidences[correct_mask]
incorrect_confidences = predicted_confidences[~correct_mask]

# ------------------------------------------------------------
# Basic Statistics
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CONFIDENCE STATISTICS")
print("=" * 70)

print(f"\nCorrect Predictions   : {len(correct_confidences)}")
print(f"Incorrect Predictions : {len(incorrect_confidences)}")

if len(correct_confidences) > 0:
    print(
        f"\nAverage Correct Confidence   : "
        f"{correct_confidences.mean():.2f}%"
    )

    print(
        f"Minimum Correct Confidence   : "
        f"{correct_confidences.min():.2f}%"
    )

    print(
        f"Maximum Correct Confidence   : "
        f"{correct_confidences.max():.2f}%"
    )

if len(incorrect_confidences) > 0:
    print(
        f"\nAverage Incorrect Confidence : "
        f"{incorrect_confidences.mean():.2f}%"
    )

    print(
        f"Minimum Incorrect Confidence : "
        f"{incorrect_confidences.min():.2f}%"
    )

    print(
        f"Maximum Incorrect Confidence : "
        f"{incorrect_confidences.max():.2f}%"
    )

# ------------------------------------------------------------
# Confidence Distribution
# ------------------------------------------------------------

plt.figure(figsize=(11, 6))

if len(correct_confidences) > 0:
    plt.hist(
        correct_confidences,
        bins=15,
        alpha=0.6,
        label="Correct Predictions"
    )

if len(incorrect_confidences) > 0:
    plt.hist(
        incorrect_confidences,
        bins=15,
        alpha=0.6,
        label="Incorrect Predictions"
    )

plt.xlabel("Prediction Confidence (%)")
plt.ylabel("Number of Images")
plt.title("ResNet50 Prediction Confidence Distribution")
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

plt.savefig(
    "resnet50_confidence_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# High-Confidence Wrong Predictions
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("HIGH-CONFIDENCE WRONG PREDICTIONS")
print("=" * 70)

wrong_indices = np.where(~correct_mask)[0]

high_conf_wrong = []

for idx in wrong_indices:

    pred_class = test_predictions[idx]
    true_class = test_true_labels[idx]
    confidence = predicted_confidences[idx]

    high_conf_wrong.append({
        "index": idx,
        "actual": class_names[true_class],
        "predicted": class_names[pred_class],
        "confidence": confidence
    })

if len(high_conf_wrong) > 0:

    high_conf_wrong_df = pd.DataFrame(
        high_conf_wrong
    ).sort_values(
        "confidence",
        ascending=False
    )

    print(
        high_conf_wrong_df.head(10).to_string(
            index=False,
            formatters={
                "confidence": "{:.2f}%".format
            }
        )
    )

    high_conf_wrong_df.to_csv(
        "resnet50_high_confidence_wrong_predictions.csv",
        index=False
    )

else:
    print("\n✓ No incorrect predictions found.")

# ------------------------------------------------------------
# Confidence Summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CONFIDENCE ANALYSIS SUMMARY")
print("=" * 70)

if len(incorrect_confidences) > 0:

    high_conf_count = np.sum(
        incorrect_confidences >= 80
    )

    print(
        f"Incorrect predictions with ≥80% confidence: "
        f"{high_conf_count}"
    )

    print(
        f"Percentage of wrong predictions with ≥80% confidence: "
        f"{(high_conf_count / len(incorrect_confidences)) * 100:.2f}%"
    )

print("\n✓ Confidence visualization saved:")
print("  resnet50_confidence_distribution.png")

print("\n" + "=" * 70)
print("✓ RESNET50 CONFIDENCE ANALYSIS COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 53: RESNET50 CLASS-WISE F1-SCORE VISUALIZATION
# ============================================================

from sklearn.metrics import classification_report
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

print("=" * 70)
print("RESNET50 CLASS-WISE F1-SCORE ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# Classification report dictionary
# ------------------------------------------------------------

report_dict = classification_report(
    test_true_labels,
    test_predictions,
    labels=np.arange(len(class_names)),
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

# ------------------------------------------------------------
# Create DataFrame
# ------------------------------------------------------------

class_metrics = []

for class_name in class_names:

    class_metrics.append({
        "Species": class_name,
        "Precision": report_dict[class_name]["precision"],
        "Recall": report_dict[class_name]["recall"],
        "F1-Score": report_dict[class_name]["f1-score"],
        "Support": int(report_dict[class_name]["support"])
    })

class_metrics_df = pd.DataFrame(class_metrics)

# ------------------------------------------------------------
# Sort by F1-score
# ------------------------------------------------------------

class_metrics_df = class_metrics_df.sort_values(
    by="F1-Score",
    ascending=True
).reset_index(drop=True)

# ------------------------------------------------------------
# Display metrics
# ------------------------------------------------------------

print("\nClass-wise Performance:")
print(
    class_metrics_df.to_string(
        index=False,
        formatters={
            "Precision": "{:.4f}".format,
            "Recall": "{:.4f}".format,
            "F1-Score": "{:.4f}".format
        }
    )
)

# ------------------------------------------------------------
# Plot Horizontal Bar Chart
# ------------------------------------------------------------

plt.figure(figsize=(12, 10))

plt.barh(
    class_metrics_df["Species"],
    class_metrics_df["F1-Score"]
)

plt.xlabel("F1-Score")
plt.ylabel("Species")

plt.title(
    "ResNet50 Class-wise F1 Score"
)

plt.xlim(0, 1.05)

plt.grid(axis="x", alpha=0.3)

# Add F1 labels
for i, value in enumerate(class_metrics_df["F1-Score"]):

    plt.text(
        value + 0.01,
        i,
        f"{value:.2f}",
        va="center",
        fontsize=8
    )

plt.tight_layout()

plt.savefig(
    "resnet50_classwise_f1_score.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# Top / Bottom Classes
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TOP 5 PERFORMING CLASSES")
print("=" * 70)

print(
    class_metrics_df.sort_values(
        by="F1-Score",
        ascending=False
    ).head(5).to_string(
        index=False,
        formatters={
            "F1-Score": "{:.4f}".format
        }
    )
)

print("\n" + "=" * 70)
print("LOWEST 5 PERFORMING CLASSES")
print("=" * 70)

print(
    class_metrics_df.head(5).to_string(
        index=False,
        formatters={
            "F1-Score": "{:.4f}".format
        }
    )
)

# ------------------------------------------------------------
# Save CSV
# ------------------------------------------------------------

class_metrics_df.to_csv(
    "resnet50_per_class_metrics.csv",
    index=False
)

print("\n✓ Saved files:")
print("  resnet50_classwise_f1_score.png")
print("  resnet50_per_class_metrics.csv")

print("\n" + "=" * 70)
print("✓ RESNET50 CLASS-WISE F1 ANALYSIS COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 54: RESNET50 SUPPORT vs F1-SCORE ANALYSIS
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

print("=" * 70)
print("RESNET50 SUPPORT vs F1-SCORE ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# Prepare Data
# ------------------------------------------------------------

support = class_metrics_df["Support"].values
f1_scores = class_metrics_df["F1-Score"].values
species = class_metrics_df["Species"].values

# ------------------------------------------------------------
# Calculate Correlation
# ------------------------------------------------------------

if len(support) > 1 and np.std(support) > 0 and np.std(f1_scores) > 0:
    correlation = np.corrcoef(
        support,
        f1_scores
    )[0, 1]
else:
    correlation = np.nan

print(f"\nPearson Correlation (Support vs F1): {correlation:.4f}")

# ------------------------------------------------------------
# Scatter Plot
# ------------------------------------------------------------

plt.figure(figsize=(12, 8))

plt.scatter(
    support,
    f1_scores,
    s=70,
    alpha=0.75
)

# Add species labels
for i in range(len(species)):

    plt.annotate(
        species[i],
        (
            support[i],
            f1_scores[i]
        ),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8
    )

plt.xlabel("Test Set Support (Number of Images)")
plt.ylabel("F1-Score")
plt.title(
    "ResNet50: Test Set Support vs Class-wise F1-Score"
)

plt.ylim(-0.05, 1.05)

plt.grid(alpha=0.3)

plt.tight_layout()

plt.savefig(
    "resnet50_support_vs_f1.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# Low-Support Classes
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("LOW-SUPPORT CLASSES")
print("=" * 70)

low_support_df = class_metrics_df[
    class_metrics_df["Support"] <= 3
].sort_values(
    by="Support",
    ascending=True
)

if len(low_support_df) > 0:

    print(
        low_support_df.to_string(
            index=False,
            formatters={
                "Precision": "{:.4f}".format,
                "Recall": "{:.4f}".format,
                "F1-Score": "{:.4f}".format
            }
        )
    )

else:
    print("No classes with support ≤ 3.")

# ------------------------------------------------------------
# Save Analysis
# ------------------------------------------------------------

class_metrics_df[
    ["Species", "Support", "F1-Score"]
].to_csv(
    "resnet50_support_vs_f1_data.csv",
    index=False
)

print("\n✓ Saved files:")
print("  resnet50_support_vs_f1.png")
print("  resnet50_support_vs_f1_data.csv")

print("\n" + "=" * 70)
print("✓ RESNET50 SUPPORT vs F1 ANALYSIS COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 55: RESNET50 FINAL PERFORMANCE SUMMARY
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("RESNET50 FINAL PERFORMANCE SUMMARY")
print("=" * 70)

# ------------------------------------------------------------
# Cross-Validation Metrics
# ------------------------------------------------------------

cv_accuracy = fold_results_df["best_val_acc"].mean()
cv_std = fold_results_df["best_val_acc"].std()

# ------------------------------------------------------------
# Test Metrics
# ------------------------------------------------------------

test_acc = test_accuracy

# ------------------------------------------------------------
# Classification Metrics
# ------------------------------------------------------------

summary_data = {
    "Metric": [
        "Mean CV Accuracy",
        "CV Accuracy Std",
        "Test Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1-Score",
        "Weighted Precision",
        "Weighted Recall",
        "Weighted F1-Score",
        "Test Samples",
        "Number of Classes"
    ],
    
    "Value": [
        f"{cv_accuracy:.2f}%",
        f"{cv_std:.2f}%",
        f"{test_acc:.2f}%",
        f"{macro_precision * 100:.2f}%",
        f"{macro_recall * 100:.2f}%",
        f"{macro_f1 * 100:.2f}%",
        f"{weighted_precision * 100:.2f}%",
        f"{weighted_recall * 100:.2f}%",
        f"{weighted_f1 * 100:.2f}%",
        len(test_true_labels),
        len(class_names)
    ]
}

resnet50_summary_df = pd.DataFrame(summary_data)

# ------------------------------------------------------------
# Display Summary
# ------------------------------------------------------------

print("\nFinal ResNet50 Metrics:")
print("-" * 70)

print(
    resnet50_summary_df.to_string(
        index=False
    )
)

# ------------------------------------------------------------
# Important Metrics Individually
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("KEY RESULTS")
print("=" * 70)

print(f"Mean CV Accuracy : {cv_accuracy:.2f}%")
print(f"Test Accuracy    : {test_acc:.2f}%")
print(f"Macro F1-Score   : {macro_f1 * 100:.2f}%")
print(f"Weighted F1      : {weighted_f1 * 100:.2f}%")

# ------------------------------------------------------------
# Generalization Gap
# ------------------------------------------------------------

generalization_gap = cv_accuracy - test_acc

print(
    f"\nCV Accuracy - Test Accuracy: "
    f"{generalization_gap:.2f} percentage points"
)

# ------------------------------------------------------------
# Save Summary
# ------------------------------------------------------------

resnet50_summary_df.to_csv(
    "resnet50_final_performance_summary.csv",
    index=False
)

print("\n✓ Final summary saved:")
print("  resnet50_final_performance_summary.csv")

print("\n" + "=" * 70)
print("✓ RESNET50 FINAL PERFORMANCE SUMMARY COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 56: RESNET50 FINAL MODEL EXPORT
# ============================================================

import torch
import json
import os

print("=" * 70)
print("RESNET50 FINAL MODEL EXPORT")
print("=" * 70)

# ------------------------------------------------------------
# 1. Save Best Model
# ------------------------------------------------------------

final_model_path = "final_resnet50_leaf_classifier.pth"

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "num_classes": len(class_names),
        "class_names": class_names,
        "test_accuracy": float(test_accuracy),
        "cv_accuracy": float(cv_accuracy),
        "cv_std": float(cv_std),
    },
    final_model_path
)

print(f"\n✓ Final model saved:")
print(f"  {final_model_path}")

# ------------------------------------------------------------
# 2. Save Class Names
# ------------------------------------------------------------

class_names_path = "resnet50_class_names.json"

with open(
    class_names_path,
    "w"
) as f:

    json.dump(
        class_names,
        f,
        indent=4
    )

print(f"✓ Class names saved:")
print(f"  {class_names_path}")

# ------------------------------------------------------------
# 3. Model Configuration
# ------------------------------------------------------------

model_config = {
    "model": "ResNet50",
    "pretrained": True,
    "num_classes": len(class_names),
    "input_size": [224, 224],
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "initial_learning_rate": INITIAL_LR,
    "weight_decay": WEIGHT_DECAY,
    "optimizer": "AdamW",
    "loss_function": "CrossEntropyLoss",
    "scheduler": "ReduceLROnPlateau",
    "scheduler_mode": "min",
    "scheduler_factor": 0.5,
    "scheduler_patience": 3,
    "cross_validation": "Stratified 4-Fold",
    "device": str(device),
    "backbone_frozen": True,
    "classifier_trainable": True
}

model_config_path = "resnet50_model_config.json"

with open(
    model_config_path,
    "w"
) as f:

    json.dump(
        model_config,
        f,
        indent=4
    )

print(f"✓ Model configuration saved:")
print(f"  {model_config_path}")

# ------------------------------------------------------------
# 4. Save Performance Summary
# ------------------------------------------------------------

performance_summary = {
    "model": "ResNet50",
    "mean_cv_accuracy": float(cv_accuracy),
    "cv_std": float(cv_std),
    "test_accuracy": float(test_accuracy),
    "macro_precision": float(macro_precision * 100),
    "macro_recall": float(macro_recall * 100),
    "macro_f1": float(macro_f1 * 100),
    "weighted_precision": float(weighted_precision * 100),
    "weighted_recall": float(weighted_recall * 100),
    "weighted_f1": float(weighted_f1 * 100),
    "test_samples": int(len(test_true_labels)),
    "num_classes": int(len(class_names))
}

performance_path = "resnet50_performance_summary.json"

with open(
    performance_path,
    "w"
) as f:

    json.dump(
        performance_summary,
        f,
        indent=4
    )

print(f"✓ Performance summary saved:")
print(f"  {performance_path}")

# ------------------------------------------------------------
# 5. Check File Sizes
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("EXPORTED FILES")
print("=" * 70)

exported_files = [
    final_model_path,
    class_names_path,
    model_config_path,
    performance_path
]

for file_path in exported_files:

    if os.path.exists(file_path):

        file_size_mb = (
            os.path.getsize(file_path)
            / (1024 ** 2)
        )

        print(
            f"{file_path:<45} "
            f"{file_size_mb:.2f} MB"
        )

print("\n" + "=" * 70)
print("FINAL RESNET50 PERFORMANCE")
print("=" * 70)

print(f"Mean CV Accuracy : {cv_accuracy:.2f}%")
print(f"Test Accuracy    : {test_accuracy:.2f}%")
print(f"Macro F1         : {macro_f1 * 100:.2f}%")
print(f"Weighted F1      : {weighted_f1 * 100:.2f}%")

print("\n" + "=" * 70)
print("✓ RESNET50 MODEL EXPORT COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 57: EFFICIENTNET-B0 vs RESNET50 COMPARISON
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

print("=" * 70)
print("EFFICIENTNET-B0 vs RESNET50 COMPARISON")
print("=" * 70)

# ------------------------------------------------------------
# EfficientNet-B0 Results
# ------------------------------------------------------------

efficientnet_results = {
    "Model": "EfficientNet-B0",
    "Mean CV Accuracy": 92.71,
    "Test Accuracy": 93.22,
    "Macro Precision": 80.10,
    "Macro Recall": 77.98,
    "Macro F1": 78.98,
    "Weighted Precision": 93.12,
    "Weighted Recall": 93.22,
    "Weighted F1": 92.23
}

# ------------------------------------------------------------
# ResNet50 Results
# ------------------------------------------------------------

resnet50_results = {
    "Model": "ResNet50",
    "Mean CV Accuracy": cv_accuracy,
    "Test Accuracy": test_accuracy,
    "Macro Precision": macro_precision * 100,
    "Macro Recall": macro_recall * 100,
    "Macro F1": macro_f1 * 100,
    "Weighted Precision": weighted_precision * 100,
    "Weighted Recall": weighted_recall * 100,
    "Weighted F1": weighted_f1 * 100
}

# ------------------------------------------------------------
# Create Comparison DataFrame
# ------------------------------------------------------------

comparison_df = pd.DataFrame(
    [
        efficientnet_results,
        resnet50_results
    ]
)

print("\n" + "=" * 70)
print("MODEL PERFORMANCE COMPARISON")
print("=" * 70)

print(
    comparison_df.to_string(
        index=False,
        formatters={
            "Mean CV Accuracy": "{:.2f}%".format,
            "Test Accuracy": "{:.2f}%".format,
            "Macro Precision": "{:.2f}%".format,
            "Macro Recall": "{:.2f}%".format,
            "Macro F1": "{:.2f}%".format,
            "Weighted Precision": "{:.2f}%".format,
            "Weighted Recall": "{:.2f}%".format,
            "Weighted F1": "{:.2f}%".format
        }
    )
)

# ------------------------------------------------------------
# Save Comparison Table
# ------------------------------------------------------------

comparison_df.to_csv(
    "efficientnet_b0_vs_resnet50_comparison.csv",
    index=False
)

# ------------------------------------------------------------
# Main Metrics Comparison
# ------------------------------------------------------------

main_metrics = [
    "Mean CV Accuracy",
    "Test Accuracy",
    "Macro F1",
    "Weighted F1"
]

x = np.arange(len(main_metrics))
width = 0.35

plt.figure(figsize=(12, 7))

plt.bar(
    x - width / 2,
    comparison_df.loc[0, main_metrics].values.astype(float),
    width,
    label="EfficientNet-B0"
)

plt.bar(
    x + width / 2,
    comparison_df.loc[1, main_metrics].values.astype(float),
    width,
    label="ResNet50"
)

plt.xticks(
    x,
    [
        "Mean CV\nAccuracy",
        "Test\nAccuracy",
        "Macro F1",
        "Weighted F1"
    ]
)

plt.ylabel("Score (%)")
plt.ylim(0, 100)

plt.title(
    "EfficientNet-B0 vs ResNet50 Performance Comparison"
)

plt.legend()
plt.grid(axis="y", alpha=0.3)

# Add value labels
for i, value in enumerate(
    comparison_df.loc[0, main_metrics].values.astype(float)
):
    plt.text(
        i - width / 2,
        value + 1,
        f"{value:.2f}",
        ha="center",
        fontsize=9
    )

for i, value in enumerate(
    comparison_df.loc[1, main_metrics].values.astype(float)
):
    plt.text(
        i + width / 2,
        value + 1,
        f"{value:.2f}",
        ha="center",
        fontsize=9
    )

plt.tight_layout()

plt.savefig(
    "efficientnet_b0_vs_resnet50_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# Difference Analysis
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("METRIC DIFFERENCES")
print("=" * 70)

for metric in main_metrics:

    efficientnet_value = float(
        comparison_df.loc[0, metric]
    )

    resnet_value = float(
        comparison_df.loc[1, metric]
    )

    difference = resnet_value - efficientnet_value

    print(
        f"{metric:<20}: "
        f"ResNet50 - EfficientNet-B0 = "
        f"{difference:+.2f} percentage points"
    )

# ------------------------------------------------------------
# Save Final Comparison
# ------------------------------------------------------------

print("\n✓ Comparison table saved:")
print("  efficientnet_b0_vs_resnet50_comparison.csv")

print("\n✓ Comparison figure saved:")
print("  efficientnet_b0_vs_resnet50_comparison.png")

print("\n" + "=" * 70)
print("✓ MODEL COMPARISON COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 57: EFFICIENTNET-B0 vs RESNET50 COMPARISON
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

print("=" * 70)
print("EFFICIENTNET-B0 vs RESNET50 COMPARISON")
print("=" * 70)

# ------------------------------------------------------------
# EfficientNet-B0 Results
# ------------------------------------------------------------

efficientnet_results = {
    "Model": "EfficientNet-B0",
    "Mean CV Accuracy": 92.71,
    "Test Accuracy": 93.22,
    "Macro Precision": 80.10,
    "Macro Recall": 77.98,
    "Macro F1": 78.98,
    "Weighted Precision": 93.12,
    "Weighted Recall": 93.22,
    "Weighted F1": 92.23
}

# ------------------------------------------------------------
# ResNet50 Results
# ------------------------------------------------------------

resnet50_results = {
    "Model": "ResNet50",
    "Mean CV Accuracy": cv_accuracy,
    "Test Accuracy": test_accuracy,
    "Macro Precision": macro_precision * 100,
    "Macro Recall": macro_recall * 100,
    "Macro F1": macro_f1 * 100,
    "Weighted Precision": weighted_precision * 100,
    "Weighted Recall": weighted_recall * 100,
    "Weighted F1": weighted_f1 * 100
}

# ------------------------------------------------------------
# Create Comparison DataFrame
# ------------------------------------------------------------

comparison_df = pd.DataFrame(
    [
        efficientnet_results,
        resnet50_results
    ]
)

print("\n" + "=" * 70)
print("MODEL PERFORMANCE COMPARISON")
print("=" * 70)

print(
    comparison_df.to_string(
        index=False,
        formatters={
            "Mean CV Accuracy": "{:.2f}%".format,
            "Test Accuracy": "{:.2f}%".format,
            "Macro Precision": "{:.2f}%".format,
            "Macro Recall": "{:.2f}%".format,
            "Macro F1": "{:.2f}%".format,
            "Weighted Precision": "{:.2f}%".format,
            "Weighted Recall": "{:.2f}%".format,
            "Weighted F1": "{:.2f}%".format
        }
    )
)

# ------------------------------------------------------------
# Save Comparison Table
# ------------------------------------------------------------

comparison_df.to_csv(
    "efficientnet_b0_vs_resnet50_comparison.csv",
    index=False
)

# ------------------------------------------------------------
# Main Metrics Comparison
# ------------------------------------------------------------

main_metrics = [
    "Mean CV Accuracy",
    "Test Accuracy",
    "Macro F1",
    "Weighted F1"
]

x = np.arange(len(main_metrics))
width = 0.35

plt.figure(figsize=(12, 7))

plt.bar(
    x - width / 2,
    comparison_df.loc[0, main_metrics].values.astype(float),
    width,
    label="EfficientNet-B0"
)

plt.bar(
    x + width / 2,
    comparison_df.loc[1, main_metrics].values.astype(float),
    width,
    label="ResNet50"
)

plt.xticks(
    x,
    [
        "Mean CV\nAccuracy",
        "Test\nAccuracy",
        "Macro F1",
        "Weighted F1"
    ]
)

plt.ylabel("Score (%)")
plt.ylim(0, 100)

plt.title(
    "EfficientNet-B0 vs ResNet50 Performance Comparison"
)

plt.legend()
plt.grid(axis="y", alpha=0.3)

# Add value labels
for i, value in enumerate(
    comparison_df.loc[0, main_metrics].values.astype(float)
):
    plt.text(
        i - width / 2,
        value + 1,
        f"{value:.2f}",
        ha="center",
        fontsize=9
    )

for i, value in enumerate(
    comparison_df.loc[1, main_metrics].values.astype(float)
):
    plt.text(
        i + width / 2,
        value + 1,
        f"{value:.2f}",
        ha="center",
        fontsize=9
    )

plt.tight_layout()

plt.savefig(
    "efficientnet_b0_vs_resnet50_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# Difference Analysis
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("METRIC DIFFERENCES")
print("=" * 70)

for metric in main_metrics:

    efficientnet_value = float(
        comparison_df.loc[0, metric]
    )

    resnet_value = float(
        comparison_df.loc[1, metric]
    )

    difference = resnet_value - efficientnet_value

    print(
        f"{metric:<20}: "
        f"ResNet50 - EfficientNet-B0 = "
        f"{difference:+.2f} percentage points"
    )

# ------------------------------------------------------------
# Save Final Comparison
# ------------------------------------------------------------

print("\n✓ Comparison table saved:")
print("  efficientnet_b0_vs_resnet50_comparison.csv")

print("\n✓ Comparison figure saved:")
print("  efficientnet_b0_vs_resnet50_comparison.png")

print("\n" + "=" * 70)
print("✓ MODEL COMPARISON COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 58: DENSENET121 MODEL SETUP
# ============================================================

import torch
import torch.nn as nn
from torchvision import models

print("=" * 70)
print("DENSENET121 MODEL SETUP")
print("=" * 70)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"\nDevice: {device}")

# ------------------------------------------------------------
# Number of Classes
# ------------------------------------------------------------

NUM_CLASSES = len(class_names)

print(f"Number of Classes: {NUM_CLASSES}")

# ------------------------------------------------------------
# Load ImageNet Pretrained DenseNet121
# ------------------------------------------------------------

weights = models.DenseNet121_Weights.DEFAULT

model = models.densenet121(
    weights=weights
)

print("\n✓ ImageNet pretrained DenseNet121 loaded")

# ------------------------------------------------------------
# Freeze Entire Backbone
# ------------------------------------------------------------

for param in model.parameters():
    param.requires_grad = False

# ------------------------------------------------------------
# Replace Final Classifier
# ------------------------------------------------------------

in_features = model.classifier.in_features

model.classifier = nn.Linear(
    in_features,
    NUM_CLASSES
)

# Only final classifier is trainable
for param in model.classifier.parameters():
    param.requires_grad = True

# ------------------------------------------------------------
# Move Model to Device
# ------------------------------------------------------------

model = model.to(device)

# ------------------------------------------------------------
# Parameter Information
# ------------------------------------------------------------

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("\n" + "=" * 70)
print("MODEL INFORMATION")
print("=" * 70)

print(f"Architecture       : DenseNet121")
print(f"Pretrained         : ImageNet")
print(f"Classifier Input   : {in_features}")
print(f"Classifier Output  : {NUM_CLASSES}")
print(f"Total Parameters   : {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

print("\n✓ Backbone frozen")
print("✓ Final classifier trainable")

print("\n" + "=" * 70)
print("✓ DENSENET121 SETUP COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 59: DENSENET121 STRATIFIED 4-FOLD CROSS-VALIDATION
#          WITH EARLY STOPPING
# ============================================================

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import StratifiedKFold

print("=" * 70)
print("DENSENET121 STRATIFIED 4-FOLD CROSS-VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

NUM_FOLDS = 4
MAX_EPOCHS = 150
EARLY_STOPPING_PATIENCE = 10

INITIAL_LR = 0.001
WEIGHT_DECAY = 0.0001
BATCH_SIZE = 32

print("\nConfiguration:")
print(f"  Number of Folds          : {NUM_FOLDS}")
print(f"  Maximum Epochs           : {MAX_EPOCHS}")
print(f"  Early Stopping Patience  : {EARLY_STOPPING_PATIENCE}")
print(f"  Initial Learning Rate    : {INITIAL_LR}")
print(f"  Weight Decay             : {WEIGHT_DECAY}")
print(f"  Batch Size               : {BATCH_SIZE}")

# ------------------------------------------------------------
# Extract Labels from train_dataset
# ------------------------------------------------------------

all_dataset_labels = []

for i in range(len(train_dataset)):

    label = train_dataset[i][1]

    if torch.is_tensor(label):
        label = label.item()

    all_dataset_labels.append(int(label))

all_dataset_labels = np.array(all_dataset_labels)

print(f"\nTraining samples available for CV: {len(all_dataset_labels)}")

# ------------------------------------------------------------
# Stratified K-Fold
# ------------------------------------------------------------

skf = StratifiedKFold(
    n_splits=NUM_FOLDS,
    shuffle=True,
    random_state=42
)

fold_results = []
fold_histories = []

global_best_val_acc = -1
global_best_fold = None
global_best_epoch = None

# ------------------------------------------------------------
# Cross-Validation Loop
# ------------------------------------------------------------

for fold, (train_indices, val_indices) in enumerate(
    skf.split(
        np.zeros(len(all_dataset_labels)),
        all_dataset_labels
    ),
    start=1
):

    print("\n" + "=" * 70)
    print(f"FOLD {fold}/{NUM_FOLDS}")
    print("=" * 70)

    print(f"Training samples   : {len(train_indices)}")
    print(f"Validation samples : {len(val_indices)}")

    # --------------------------------------------------------
    # Create Fold Datasets
    # --------------------------------------------------------

    fold_train_dataset = Subset(
        train_dataset,
        train_indices
    )

    fold_val_dataset = Subset(
        train_dataset,
        val_indices
    )

    fold_train_loader = DataLoader(
        fold_train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True if device.type == "cuda" else False
    )

    fold_val_loader = DataLoader(
        fold_val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True if device.type == "cuda" else False
    )

    # --------------------------------------------------------
    # Reset DenseNet121
    # --------------------------------------------------------

    fold_model = models.densenet121(
        weights=models.DenseNet121_Weights.DEFAULT
    )

    # Freeze backbone
    for param in fold_model.parameters():
        param.requires_grad = False

    # Replace classifier
    fold_model.classifier = nn.Linear(
        fold_model.classifier.in_features,
        NUM_CLASSES
    )

    # Classifier trainable
    for param in fold_model.classifier.parameters():
        param.requires_grad = True

    fold_model = fold_model.to(device)

    # --------------------------------------------------------
    # Loss Function
    # --------------------------------------------------------

    criterion = nn.CrossEntropyLoss()

    # --------------------------------------------------------
    # Optimizer
    # --------------------------------------------------------

    optimizer = torch.optim.AdamW(
        filter(
            lambda p: p.requires_grad,
            fold_model.parameters()
        ),
        lr=INITIAL_LR,
        weight_decay=WEIGHT_DECAY
    )

    # --------------------------------------------------------
    # Learning Rate Scheduler
    # --------------------------------------------------------

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=3
    )

    # --------------------------------------------------------
    # History
    # --------------------------------------------------------

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": [],
        "lr": []
    }

    # --------------------------------------------------------
    # Early Stopping Variables
    # --------------------------------------------------------

    best_val_acc = -1
    best_val_loss = float("inf")
    best_epoch = 0
    patience_counter = 0

    fold_model_path = f"densenet121_fold_{fold}.pth"

    # --------------------------------------------------------
    # Epoch Loop
    # --------------------------------------------------------

    for epoch in range(1, MAX_EPOCHS + 1):

        # ====================================================
        # TRAINING
        # ====================================================

        fold_model.train()

        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for images, labels in fold_train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = fold_model(images)

            loss = criterion(
                outputs,
                labels
            )

            loss.backward()

            optimizer.step()

            train_loss += (
                loss.item() * images.size(0)
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            train_correct += (
                predictions == labels
            ).sum().item()

            train_total += labels.size(0)

        train_loss /= train_total

        train_acc = (
            train_correct / train_total
        ) * 100

        # ====================================================
        # VALIDATION
        # ====================================================

        fold_model.eval()

        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():

            for images, labels in fold_val_loader:

                images = images.to(device)
                labels = labels.to(device)

                outputs = fold_model(images)

                loss = criterion(
                    outputs,
                    labels
                )

                val_loss += (
                    loss.item() * images.size(0)
                )

                predictions = torch.argmax(
                    outputs,
                    dim=1
                )

                val_correct += (
                    predictions == labels
                ).sum().item()

                val_total += labels.size(0)

        val_loss /= val_total

        val_acc = (
            val_correct / val_total
        ) * 100

        # ====================================================
        # Scheduler
        # ====================================================

        scheduler.step(val_loss)

        current_lr = optimizer.param_groups[0]["lr"]

        # ====================================================
        # Store History
        # ====================================================

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["lr"].append(current_lr)

        # ====================================================
        # Print Progress
        # ====================================================

        print(
            f"Epoch [{epoch:03d}/{MAX_EPOCHS}] | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.2f}% | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.2f}% | "
            f"LR: {current_lr:.6f}"
        )

        # ====================================================
        # Early Stopping Check
        # ====================================================

        if val_acc > best_val_acc:

            best_val_acc = val_acc
            best_val_loss = val_loss
            best_epoch = epoch

            patience_counter = 0

            torch.save(
                fold_model.state_dict(),
                fold_model_path
            )

            print(
                f"  ✓ New best model "
                f"(Val Acc: {best_val_acc:.2f}%)"
            )

        else:

            patience_counter += 1

        # ====================================================
        # Stop if Patience Exhausted
        # ====================================================

        if patience_counter >= EARLY_STOPPING_PATIENCE:

            print(
                f"\n  ⚠ Early stopping triggered at "
                f"epoch {epoch}"
            )

            print(
                f"  Best epoch: {best_epoch} | "
                f"Best Val Acc: {best_val_acc:.2f}%"
            )

            break

    # --------------------------------------------------------
    # Store Fold History
    # --------------------------------------------------------

    fold_histories.append(history)

    # --------------------------------------------------------
    # Fold Result
    # --------------------------------------------------------

    fold_results.append({
        "fold": fold,
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "best_val_acc": best_val_acc
    })

    print("\n" + "-" * 70)
    print(f"FOLD {fold} COMPLETE")
    print(f"Best Epoch     : {best_epoch}")
    print(f"Best Val Loss  : {best_val_loss:.4f}")
    print(f"Best Val Acc   : {best_val_acc:.2f}%")
    print(f"Saved Model    : {fold_model_path}")
    print("-" * 70)

    # --------------------------------------------------------
    # Track Global Best Fold
    # --------------------------------------------------------

    if best_val_acc > global_best_val_acc:

        global_best_val_acc = best_val_acc
        global_best_fold = fold
        global_best_epoch = best_epoch

        torch.save(
            fold_model.state_dict(),
            "best_densenet121.pth"
        )

        print(
            f"✓ New GLOBAL BEST MODEL: "
            f"Fold {fold}, "
            f"Val Acc {best_val_acc:.2f}%"
        )

# ============================================================
# CROSS-VALIDATION SUMMARY
# ============================================================

fold_results_df = pd.DataFrame(
    fold_results
)

fold_accuracies = fold_results_df[
    "best_val_acc"
].values

mean_cv_accuracy = fold_accuracies.mean()
std_cv_accuracy = fold_accuracies.std()

print("\n" + "=" * 70)
print("DENSENET121 CROSS-VALIDATION SUMMARY")
print("=" * 70)

print(
    f"Mean CV Accuracy : "
    f"{mean_cv_accuracy:.2f}%"
)

print(
    f"Std CV Accuracy  : "
    f"{std_cv_accuracy:.2f}%"
)

print(
    f"Best Fold        : "
    f"{global_best_fold}"
)

print(
    f"Best Epoch       : "
    f"{global_best_epoch}"
)

print(
    f"Best Val Accuracy: "
    f"{global_best_val_acc:.2f}%"
)

print(
    "\n✓ Best DenseNet121 model saved as:"
)

print(
    "  best_densenet121.pth"
)

print("\n" + "=" * 70)
print("✓ DENSENET121 4-FOLD CV COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 60: DENSENET121 CROSS-VALIDATION RESULTS
#          & VISUALIZATION
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 70)
print("DENSENET121 CROSS-VALIDATION RESULTS")
print("=" * 70)

# ------------------------------------------------------------
# Fold-wise Results
# ------------------------------------------------------------

fold_results_df = pd.DataFrame(fold_results)

print("\nFold-wise Validation Performance:")
print("-" * 70)

print(
    fold_results_df.to_string(
        index=False,
        formatters={
            "best_val_loss": "{:.4f}".format,
            "best_val_acc": "{:.2f}%".format
        }
    )
)

# ------------------------------------------------------------
# CV Statistics
# ------------------------------------------------------------

fold_accuracies = (
    fold_results_df["best_val_acc"].values
)

mean_accuracy = fold_accuracies.mean()
std_accuracy = fold_accuracies.std()
min_accuracy = fold_accuracies.min()
max_accuracy = fold_accuracies.max()

print("\n" + "=" * 70)
print("CROSS-VALIDATION SUMMARY")
print("=" * 70)

print(
    f"Mean CV Accuracy : {mean_accuracy:.2f}%"
)

print(
    f"Std CV Accuracy  : {std_accuracy:.2f}%"
)

print(
    f"Minimum Accuracy : {min_accuracy:.2f}%"
)

print(
    f"Maximum Accuracy : {max_accuracy:.2f}%"
)

# ------------------------------------------------------------
# Best Fold
# ------------------------------------------------------------

best_fold_index = (
    fold_results_df["best_val_acc"].idxmax()
)

best_fold_row = (
    fold_results_df.loc[best_fold_index]
)

print("\nBest Fold:")
print(
    f"Fold           : "
    f"{int(best_fold_row['fold'])}"
)

print(
    f"Best Epoch     : "
    f"{int(best_fold_row['best_epoch'])}"
)

print(
    f"Validation Acc : "
    f"{best_fold_row['best_val_acc']:.2f}%"
)

# ------------------------------------------------------------
# Save CV Results
# ------------------------------------------------------------

fold_results_df.to_csv(
    "densenet121_cv_results.csv",
    index=False
)

# ------------------------------------------------------------
# Plot 1: Fold-wise CV Accuracy
# ------------------------------------------------------------

plt.figure(figsize=(10, 6))

plt.bar(
    fold_results_df["fold"].astype(str),
    fold_results_df["best_val_acc"]
)

plt.axhline(
    mean_accuracy,
    linestyle="--",
    label=(
        f"Mean CV Accuracy "
        f"({mean_accuracy:.2f}%)"
    )
)

plt.xlabel("Fold")
plt.ylabel("Validation Accuracy (%)")

plt.title(
    "DenseNet121 4-Fold Cross-Validation Accuracy"
)

plt.legend()

plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "densenet121_cv_accuracy.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# Plot 2: Best Epoch per Fold
# ------------------------------------------------------------

plt.figure(figsize=(10, 6))

plt.bar(
    fold_results_df["fold"].astype(str),
    fold_results_df["best_epoch"]
)

plt.xlabel("Fold")
plt.ylabel("Best Epoch")

plt.title(
    "DenseNet121 Best Epoch per Fold"
)

plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "densenet121_best_epoch_per_fold.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# Final Information
# ------------------------------------------------------------

print("\n✓ Visualizations saved:")
print("  densenet121_cv_accuracy.png")
print("  densenet121_best_epoch_per_fold.png")

print("\n✓ CV results saved:")
print("  densenet121_cv_results.csv")

print("\n" + "=" * 70)
print("✓ DENSENET121 CV RESULTS & VISUALIZATION COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 61: DENSENET121 TRAINING HISTORY
#          TRAIN vs VALIDATION LOSS & ACCURACY
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

print("=" * 70)
print("DENSENET121 TRAINING HISTORY")
print("=" * 70)

# ------------------------------------------------------------
# Check History
# ------------------------------------------------------------

print(
    f"\nNumber of folds with history: "
    f"{len(fold_histories)}"
)

# ------------------------------------------------------------
# Plot 1: Training vs Validation Loss
# ------------------------------------------------------------

plt.figure(figsize=(12, 7))

for fold_idx, history in enumerate(
    fold_histories,
    start=1
):

    epochs = range(
        1,
        len(history["train_loss"]) + 1
    )

    plt.plot(
        epochs,
        history["train_loss"],
        label=f"Fold {fold_idx} Train Loss"
    )

    plt.plot(
        epochs,
        history["val_loss"],
        linestyle="--",
        label=f"Fold {fold_idx} Val Loss"
    )

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title(
    "DenseNet121 4-Fold Cross-Validation: "
    "Training vs Validation Loss"
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "densenet121_training_validation_loss.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# Plot 2: Training vs Validation Accuracy
# ------------------------------------------------------------

plt.figure(figsize=(12, 7))

for fold_idx, history in enumerate(
    fold_histories,
    start=1
):

    epochs = range(
        1,
        len(history["train_acc"]) + 1
    )

    plt.plot(
        epochs,
        history["train_acc"],
        label=f"Fold {fold_idx} Train Accuracy"
    )

    plt.plot(
        epochs,
        history["val_acc"],
        linestyle="--",
        label=f"Fold {fold_idx} Val Accuracy"
    )

plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")

plt.title(
    "DenseNet121 4-Fold Cross-Validation: "
    "Training vs Validation Accuracy"
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "densenet121_training_validation_accuracy.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# Best Epoch Information
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BEST EPOCH INFORMATION")
print("=" * 70)

for fold_idx, history in enumerate(
    fold_histories,
    start=1
):

    best_epoch = (
        np.argmax(history["val_acc"]) + 1
    )

    best_val_acc = max(
        history["val_acc"]
    )

    best_val_loss = (
        history["val_loss"][best_epoch - 1]
    )

    print(
        f"Fold {fold_idx}: "
        f"Best Epoch = {best_epoch}, "
        f"Val Accuracy = {best_val_acc:.2f}%, "
        f"Val Loss = {best_val_loss:.4f}"
    )

# ------------------------------------------------------------
# Save Confirmation
# ------------------------------------------------------------

print("\n✓ Training history plots saved:")
print("  densenet121_training_validation_loss.png")
print("  densenet121_training_validation_accuracy.png")

print("\n" + "=" * 70)
print("✓ DENSENET121 TRAINING HISTORY COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 62: DENSENET121 FINAL TEST EVALUATION
# ============================================================

import torch
import numpy as np
from sklearn.metrics import accuracy_score

print("=" * 70)
print("DENSENET121 FINAL TEST EVALUATION")
print("=" * 70)

# ------------------------------------------------------------
# Load Best DenseNet121 Model
# ------------------------------------------------------------

best_model_path = "best_densenet121.pth"

checkpoint = torch.load(
    best_model_path,
    map_location=device
)

# The CV code saved only model.state_dict()
model.load_state_dict(checkpoint)

model = model.to(device)
model.eval()

print(
    f"\n✓ Loaded best DenseNet121 model from: "
    f"{best_model_path}"
)

# ------------------------------------------------------------
# Test Evaluation
# ------------------------------------------------------------

test_predictions = []
test_true_labels = []
test_probabilities = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        probabilities = torch.softmax(
            outputs,
            dim=1
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        test_predictions.extend(
            predictions.cpu().numpy()
        )

        test_true_labels.extend(
            labels.cpu().numpy()
        )

        test_probabilities.extend(
            probabilities.cpu().numpy()
        )

# ------------------------------------------------------------
# Convert to NumPy Arrays
# ------------------------------------------------------------

test_predictions = np.array(
    test_predictions
)

test_true_labels = np.array(
    test_true_labels
)

test_probabilities = np.array(
    test_probabilities
)

# ------------------------------------------------------------
# Calculate Test Accuracy
# ------------------------------------------------------------

test_accuracy = accuracy_score(
    test_true_labels,
    test_predictions
) * 100

# ------------------------------------------------------------
# Prediction Statistics
# ------------------------------------------------------------

correct_predictions = (
    test_predictions == test_true_labels
).sum()

incorrect_predictions = (
    test_predictions != test_true_labels
).sum()

# ------------------------------------------------------------
# Display Results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL TEST RESULT")
print("=" * 70)

print(
    f"Test Samples          : "
    f"{len(test_true_labels)}"
)

print(
    f"Correct Predictions   : "
    f"{correct_predictions}"
)

print(
    f"Incorrect Predictions : "
    f"{incorrect_predictions}"
)

print(
    f"Test Accuracy         : "
    f"{test_accuracy:.2f}%"
)

print("\n" + "=" * 70)
print("✓ DENSENET121 FINAL TEST EVALUATION COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 63: DENSENET121 CLASSIFICATION REPORT
# ============================================================

from sklearn.metrics import classification_report
import pandas as pd
import numpy as np

print("=" * 70)
print("DENSENET121 CLASSIFICATION REPORT")
print("=" * 70)

# ------------------------------------------------------------
# Generate Classification Report
# ------------------------------------------------------------

report_dict = classification_report(
    test_true_labels,
    test_predictions,
    labels=np.arange(len(class_names)),
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

# ------------------------------------------------------------
# Convert to DataFrame
# ------------------------------------------------------------

report_df = pd.DataFrame(
    report_dict
).transpose()

# ------------------------------------------------------------
# Class-wise Metrics
# ------------------------------------------------------------

class_report_df = report_df.loc[
    class_names,
    [
        "precision",
        "recall",
        "f1-score",
        "support"
    ]
].copy()

class_report_df["support"] = (
    class_report_df["support"].astype(int)
)

print("\nClass-wise Performance:")
print("-" * 70)

print(
    class_report_df.to_string(
        formatters={
            "precision": "{:.4f}".format,
            "recall": "{:.4f}".format,
            "f1-score": "{:.4f}".format
        }
    )
)

# ------------------------------------------------------------
# Overall Metrics
# ------------------------------------------------------------

macro_precision = (
    report_dict["macro avg"]["precision"]
)

macro_recall = (
    report_dict["macro avg"]["recall"]
)

macro_f1 = (
    report_dict["macro avg"]["f1-score"]
)

weighted_precision = (
    report_dict["weighted avg"]["precision"]
)

weighted_recall = (
    report_dict["weighted avg"]["recall"]
)

weighted_f1 = (
    report_dict["weighted avg"]["f1-score"]
)

# ------------------------------------------------------------
# Display Overall Metrics
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("OVERALL CLASSIFICATION METRICS")
print("=" * 70)

print(
    f"Macro Precision    : "
    f"{macro_precision * 100:.2f}%"
)

print(
    f"Macro Recall       : "
    f"{macro_recall * 100:.2f}%"
)

print(
    f"Macro F1-Score     : "
    f"{macro_f1 * 100:.2f}%"
)

print(
    f"\nWeighted Precision : "
    f"{weighted_precision * 100:.2f}%"
)

print(
    f"Weighted Recall    : "
    f"{weighted_recall * 100:.2f}%"
)

print(
    f"Weighted F1-Score  : "
    f"{weighted_f1 * 100:.2f}%"
)

# ------------------------------------------------------------
# Save Classification Report
# ------------------------------------------------------------

class_report_df.to_csv(
    "densenet121_classification_report.csv"
)

print("\n✓ Classification report saved:")
print("  densenet121_classification_report.csv")

print("\n" + "=" * 70)
print("✓ DENSENET121 CLASSIFICATION REPORT COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 64: DENSENET121 CONFUSION MATRIX
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

print("=" * 70)
print("DENSENET121 CONFUSION MATRIX")
print("=" * 70)

# ------------------------------------------------------------
# Generate Confusion Matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    test_true_labels,
    test_predictions,
    labels=np.arange(len(class_names))
)

print(f"\nConfusion Matrix Shape : {cm.shape}")
print(f"Number of Classes     : {len(class_names)}")

# ------------------------------------------------------------
# Plot Confusion Matrix
# ------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(18, 16)
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

disp.plot(
    ax=ax,
    cmap="Blues",
    xticks_rotation=90,
    values_format="d",
    colorbar=True
)

ax.set_title(
    "DenseNet121 Confusion Matrix",
    fontsize=16,
    pad=20
)

ax.set_xlabel(
    "Predicted Label",
    fontsize=12
)

ax.set_ylabel(
    "True Label",
    fontsize=12
)

plt.tight_layout()

plt.savefig(
    "densenet121_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# Find Misclassification Pairs
# ------------------------------------------------------------

misclassified_pairs = []

for true_idx in range(len(class_names)):

    for pred_idx in range(len(class_names)):

        if (
            true_idx != pred_idx
            and cm[true_idx, pred_idx] > 0
        ):

            misclassified_pairs.append({
                "True Class": class_names[true_idx],
                "Predicted Class": class_names[pred_idx],
                "Count": cm[true_idx, pred_idx]
            })

misclassified_pairs_df = pd.DataFrame(
    misclassified_pairs
)

# ------------------------------------------------------------
# Most Common Confusions
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MOST COMMON MISCLASSIFICATIONS")
print("=" * 70)

if len(misclassified_pairs_df) > 0:

    misclassified_pairs_df = (
        misclassified_pairs_df
        .sort_values(
            "Count",
            ascending=False
        )
        .reset_index(drop=True)
    )

    print(
        misclassified_pairs_df.head(15).to_string(
            index=False
        )
    )

    misclassified_pairs_df.to_csv(
        "densenet121_misclassification_pairs.csv",
        index=False
    )

else:

    print("No misclassified class pairs found.")

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

print("\n✓ Confusion matrix saved:")
print("  densenet121_confusion_matrix.png")

if len(misclassified_pairs_df) > 0:
    print("✓ Misclassification pairs saved:")
    print("  densenet121_misclassification_pairs.csv")

print("\n" + "=" * 70)
print("✓ DENSENET121 CONFUSION MATRIX COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 65: DENSENET121 MISCLASSIFIED IMAGES
#          ACTUAL vs PREDICTED
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

print("=" * 70)
print("DENSENET121 MISCLASSIFIED IMAGES")
print("=" * 70)

# ------------------------------------------------------------
# Find Misclassified Samples
# ------------------------------------------------------------

misclassified_indices = np.where(
    test_predictions != test_true_labels
)[0]

correct_count = (
    test_predictions == test_true_labels
).sum()

incorrect_count = len(
    misclassified_indices
)

print(
    f"\nTotal Test Images         : "
    f"{len(test_true_labels)}"
)

print(
    f"Correct Predictions       : "
    f"{correct_count}"
)

print(
    f"Misclassified Predictions : "
    f"{incorrect_count}"
)

# ------------------------------------------------------------
# Collect Misclassification Information
# ------------------------------------------------------------

misclassified_data = []

for idx in misclassified_indices:

    true_label = test_true_labels[idx]
    pred_label = test_predictions[idx]

    confidence = (
        test_probabilities[idx][pred_label]
        * 100
    )

    misclassified_data.append({
        "index": int(idx),
        "true_label": int(true_label),
        "pred_label": int(pred_label),
        "true_class": class_names[true_label],
        "pred_class": class_names[pred_label],
        "confidence": float(confidence)
    })

# Highest-confidence wrong predictions first
misclassified_data = sorted(
    misclassified_data,
    key=lambda x: x["confidence"],
    reverse=True
)

# ------------------------------------------------------------
# Display Images
# ------------------------------------------------------------

num_to_show = min(
    12,
    len(misclassified_data)
)

if num_to_show > 0:

    fig, axes = plt.subplots(
        3,
        4,
        figsize=(16, 13)
    )

    axes = axes.flatten()

    for plot_idx in range(num_to_show):

        item = misclassified_data[plot_idx]

        dataset_idx = item["index"]

        # Get image from test dataset
        image, _ = test_dataset[dataset_idx]

        # ----------------------------------------------------
        # Convert Tensor to Displayable Image
        # ----------------------------------------------------

        if torch.is_tensor(image):

            image = image.cpu()

            # CHW -> HWC
            image = image.permute(
                1,
                2,
                0
            ).numpy()

            # Normalize for visualization
            image = (
                image - image.min()
            )

            if image.max() > 0:
                image = (
                    image / image.max()
                )

        axes[plot_idx].imshow(image)

        axes[plot_idx].set_title(
            f"Actual: {item['true_class']}\n"
            f"Predicted: {item['pred_class']}\n"
            f"Confidence: {item['confidence']:.1f}%",
            fontsize=10
        )

        axes[plot_idx].axis("off")

    # Hide unused axes
    for idx in range(
        num_to_show,
        len(axes)
    ):
        axes[idx].axis("off")

    plt.suptitle(
        "DenseNet121 Misclassified Test Images",
        fontsize=16
    )

    plt.tight_layout()

    plt.savefig(
        "densenet121_misclassified_images.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

else:

    print("\n✓ No misclassified images found!")

# ------------------------------------------------------------
# Misclassification Table
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MISCLASSIFICATION DETAILS")
print("=" * 70)

if len(misclassified_data) > 0:

    misclassified_df = pd.DataFrame(
        misclassified_data
    )

    display(
        misclassified_df[
            [
                "index",
                "true_class",
                "pred_class",
                "confidence"
            ]
        ].style.format({
            "confidence": "{:.2f}%"
        })
    )

    misclassified_df.to_csv(
        "densenet121_misclassified_images.csv",
        index=False
    )

    print(
        "\n✓ Misclassification details saved:"
    )

    print(
        "  densenet121_misclassified_images.csv"
    )

print(
    "\n✓ Visualization saved:"
)

print(
    "  densenet121_misclassified_images.png"
)

print("\n" + "=" * 70)
print(
    "✓ DENSENET121 MISCLASSIFIED IMAGE "
    "ANALYSIS COMPLETE"
)
print("=" * 70)

In [ ]:
# ============================================================
# CELL 66: DENSENET121 PREDICTION CONFIDENCE ANALYSIS
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 70)
print("DENSENET121 PREDICTION CONFIDENCE ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# Prediction Confidence
# ------------------------------------------------------------

predicted_confidences = (
    np.max(test_probabilities, axis=1) * 100
)

correct_mask = (
    test_predictions == test_true_labels
)

correct_confidences = (
    predicted_confidences[correct_mask]
)

incorrect_confidences = (
    predicted_confidences[~correct_mask]
)

# ------------------------------------------------------------
# Confidence Statistics
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CONFIDENCE STATISTICS")
print("=" * 70)

print(
    f"\nCorrect Predictions   : "
    f"{len(correct_confidences)}"
)

print(
    f"Incorrect Predictions : "
    f"{len(incorrect_confidences)}"
)

if len(correct_confidences) > 0:

    print(
        f"\nAverage Correct Confidence : "
        f"{correct_confidences.mean():.2f}%"
    )

    print(
        f"Minimum Correct Confidence : "
        f"{correct_confidences.min():.2f}%"
    )

    print(
        f"Maximum Correct Confidence : "
        f"{correct_confidences.max():.2f}%"
    )

if len(incorrect_confidences) > 0:

    print(
        f"\nAverage Incorrect Confidence : "
        f"{incorrect_confidences.mean():.2f}%"
    )

    print(
        f"Minimum Incorrect Confidence : "
        f"{incorrect_confidences.min():.2f}%"
    )

    print(
        f"Maximum Incorrect Confidence : "
        f"{incorrect_confidences.max():.2f}%"
    )

# ------------------------------------------------------------
# Confidence Distribution
# ------------------------------------------------------------

plt.figure(figsize=(11, 6))

if len(correct_confidences) > 0:

    plt.hist(
        correct_confidences,
        bins=15,
        alpha=0.6,
        label="Correct Predictions"
    )

if len(incorrect_confidences) > 0:

    plt.hist(
        incorrect_confidences,
        bins=15,
        alpha=0.6,
        label="Incorrect Predictions"
    )

plt.xlabel(
    "Prediction Confidence (%)"
)

plt.ylabel(
    "Number of Images"
)

plt.title(
    "DenseNet121 Prediction Confidence Distribution"
)

plt.legend()

plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "densenet121_confidence_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# High-Confidence Wrong Predictions
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("HIGH-CONFIDENCE WRONG PREDICTIONS")
print("=" * 70)

wrong_indices = np.where(
    ~correct_mask
)[0]

high_conf_wrong = []

for idx in wrong_indices:

    true_class = test_true_labels[idx]
    pred_class = test_predictions[idx]

    confidence = predicted_confidences[idx]

    high_conf_wrong.append({
        "index": int(idx),
        "actual": class_names[true_class],
        "predicted": class_names[pred_class],
        "confidence": float(confidence)
    })

if len(high_conf_wrong) > 0:

    high_conf_wrong_df = pd.DataFrame(
        high_conf_wrong
    ).sort_values(
        "confidence",
        ascending=False
    )

    print(
        high_conf_wrong_df.head(10).to_string(
            index=False,
            formatters={
                "confidence": "{:.2f}%".format
            }
        )
    )

    high_conf_wrong_df.to_csv(
        "densenet121_high_confidence_wrong_predictions.csv",
        index=False
    )

else:

    print(
        "\n✓ No incorrect predictions found."
    )

# ------------------------------------------------------------
# High-Confidence Error Statistics
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("HIGH-CONFIDENCE ERROR SUMMARY")
print("=" * 70)

if len(incorrect_confidences) > 0:

    high_conf_count = np.sum(
        incorrect_confidences >= 80
    )

    high_conf_percentage = (
        high_conf_count
        / len(incorrect_confidences)
    ) * 100

    print(
        f"Incorrect predictions with ≥80% confidence : "
        f"{high_conf_count}"
    )

    print(
        f"Percentage of wrong predictions ≥80% : "
        f"{high_conf_percentage:.2f}%"
    )

else:

    print(
        "No incorrect predictions available."
    )

# ------------------------------------------------------------
# Save Confirmation
# ------------------------------------------------------------

print("\n✓ Confidence visualization saved:")
print(
    "  densenet121_confidence_distribution.png"
)

if len(high_conf_wrong) > 0:

    print(
        "✓ High-confidence errors saved:"
    )

    print(
        "  densenet121_high_confidence_wrong_predictions.csv"
    )

print("\n" + "=" * 70)
print(
    "✓ DENSENET121 CONFIDENCE ANALYSIS COMPLETE"
)
print("=" * 70)

In [ ]:
# ============================================================
# CELL 67: DENSENET121 CLASS-WISE F1-SCORE VISUALIZATION
# ============================================================

from sklearn.metrics import classification_report
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

print("=" * 70)
print("DENSENET121 CLASS-WISE F1-SCORE ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# Generate Classification Report
# ------------------------------------------------------------

report_dict = classification_report(
    test_true_labels,
    test_predictions,
    labels=np.arange(len(class_names)),
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

# ------------------------------------------------------------
# Create Class-wise Metrics DataFrame
# ------------------------------------------------------------

class_metrics = []

for class_name in class_names:

    class_metrics.append({
        "Species": class_name,
        "Precision": report_dict[class_name]["precision"],
        "Recall": report_dict[class_name]["recall"],
        "F1-Score": report_dict[class_name]["f1-score"],
        "Support": int(report_dict[class_name]["support"])
    })

class_metrics_df = pd.DataFrame(
    class_metrics
)

# ------------------------------------------------------------
# Sort by F1-Score
# ------------------------------------------------------------

class_metrics_df = (
    class_metrics_df
    .sort_values(
        by="F1-Score",
        ascending=True
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Display Metrics
# ------------------------------------------------------------

print("\nClass-wise Performance:")
print("-" * 70)

print(
    class_metrics_df.to_string(
        index=False,
        formatters={
            "Precision": "{:.4f}".format,
            "Recall": "{:.4f}".format,
            "F1-Score": "{:.4f}".format
        }
    )
)

# ------------------------------------------------------------
# Plot Class-wise F1 Score
# ------------------------------------------------------------

plt.figure(figsize=(12, 10))

plt.barh(
    class_metrics_df["Species"],
    class_metrics_df["F1-Score"]
)

plt.xlabel("F1-Score")
plt.ylabel("Species")

plt.title(
    "DenseNet121 Class-wise F1 Score"
)

plt.xlim(
    0,
    1.05
)

plt.grid(
    axis="x",
    alpha=0.3
)

# Add F1 values
for i, value in enumerate(
    class_metrics_df["F1-Score"]
):

    plt.text(
        value + 0.01,
        i,
        f"{value:.2f}",
        va="center",
        fontsize=8
    )

plt.tight_layout()

plt.savefig(
    "densenet121_classwise_f1_score.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# Top 5 Classes
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TOP 5 F1-SCORE CLASSES")
print("=" * 70)

top_5 = (
    class_metrics_df
    .sort_values(
        by="F1-Score",
        ascending=False
    )
    .head(5)
)

print(
    top_5.to_string(
        index=False,
        formatters={
            "Precision": "{:.4f}".format,
            "Recall": "{:.4f}".format,
            "F1-Score": "{:.4f}".format
        }
    )
)

# ------------------------------------------------------------
# Lowest 5 Classes
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("LOWEST 5 F1-SCORE CLASSES")
print("=" * 70)

lowest_5 = (
    class_metrics_df
    .head(5)
)

print(
    lowest_5.to_string(
        index=False,
        formatters={
            "Precision": "{:.4f}".format,
            "Recall": "{:.4f}".format,
            "F1-Score": "{:.4f}".format
        }
    )
)

# ------------------------------------------------------------
# Save CSV
# ------------------------------------------------------------

class_metrics_df.to_csv(
    "densenet121_per_class_metrics.csv",
    index=False
)

print("\n✓ Saved files:")
print("  densenet121_classwise_f1_score.png")
print("  densenet121_per_class_metrics.csv")

print("\n" + "=" * 70)
print("✓ DENSENET121 CLASS-WISE F1 ANALYSIS COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 68: DENSENET121 SUPPORT vs F1-SCORE ANALYSIS
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print("=" * 70)
print("DENSENET121 SUPPORT vs F1-SCORE ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# Prepare Data
# ------------------------------------------------------------

support = class_metrics_df["Support"].values
f1_scores = class_metrics_df["F1-Score"].values
species = class_metrics_df["Species"].values

# ------------------------------------------------------------
# Calculate Pearson Correlation
# ------------------------------------------------------------

if (
    len(support) > 1
    and np.std(support) > 0
    and np.std(f1_scores) > 0
):

    correlation = np.corrcoef(
        support,
        f1_scores
    )[0, 1]

else:

    correlation = np.nan

print(
    f"\nPearson Correlation "
    f"(Support vs F1): {correlation:.4f}"
)

# ------------------------------------------------------------
# Scatter Plot
# ------------------------------------------------------------

plt.figure(
    figsize=(12, 8)
)

plt.scatter(
    support,
    f1_scores,
    s=70,
    alpha=0.75
)

# Add species names
for i in range(len(species)):

    plt.annotate(
        species[i],
        (
            support[i],
            f1_scores[i]
        ),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8
    )

plt.xlabel(
    "Test Set Support (Number of Images)"
)

plt.ylabel(
    "F1-Score"
)

plt.title(
    "DenseNet121: Test Set Support vs Class-wise F1-Score"
)

plt.ylim(
    -0.05,
    1.05
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "densenet121_support_vs_f1.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# Low-Support Classes
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("LOW-SUPPORT CLASSES")
print("=" * 70)

low_support_df = (
    class_metrics_df[
        class_metrics_df["Support"] <= 3
    ]
    .sort_values(
        by="Support",
        ascending=True
    )
)

if len(low_support_df) > 0:

    print(
        low_support_df.to_string(
            index=False,
            formatters={
                "Precision": "{:.4f}".format,
                "Recall": "{:.4f}".format,
                "F1-Score": "{:.4f}".format
            }
        )
    )

else:

    print(
        "No classes with support ≤ 3."
    )

# ------------------------------------------------------------
# Save Analysis Data
# ------------------------------------------------------------

class_metrics_df[
    [
        "Species",
        "Support",
        "F1-Score"
    ]
].to_csv(
    "densenet121_support_vs_f1_data.csv",
    index=False
)

print("\n✓ Saved files:")
print("  densenet121_support_vs_f1.png")
print("  densenet121_support_vs_f1_data.csv")

print("\n" + "=" * 70)
print("✓ DENSENET121 SUPPORT vs F1 ANALYSIS COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 69: DENSENET121 FINAL PERFORMANCE SUMMARY
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("DENSENET121 FINAL PERFORMANCE SUMMARY")
print("=" * 70)

# ------------------------------------------------------------
# Cross-Validation Metrics
# ------------------------------------------------------------

cv_accuracy = fold_results_df["best_val_acc"].mean()
cv_std = fold_results_df["best_val_acc"].std()

# ------------------------------------------------------------
# Test Accuracy
# ------------------------------------------------------------

test_acc = test_accuracy

# ------------------------------------------------------------
# Create Final Summary
# ------------------------------------------------------------

summary_data = {
    "Metric": [
        "Mean CV Accuracy",
        "CV Accuracy Std",
        "Test Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1-Score",
        "Weighted Precision",
        "Weighted Recall",
        "Weighted F1-Score",
        "Test Samples",
        "Number of Classes"
    ],

    "Value": [
        f"{cv_accuracy:.2f}%",
        f"{cv_std:.2f}%",
        f"{test_acc:.2f}%",
        f"{macro_precision * 100:.2f}%",
        f"{macro_recall * 100:.2f}%",
        f"{macro_f1 * 100:.2f}%",
        f"{weighted_precision * 100:.2f}%",
        f"{weighted_recall * 100:.2f}%",
        f"{weighted_f1 * 100:.2f}%",
        len(test_true_labels),
        len(class_names)
    ]
}

densenet121_summary_df = pd.DataFrame(
    summary_data
)

# ------------------------------------------------------------
# Display Summary
# ------------------------------------------------------------

print("\nFinal DenseNet121 Metrics:")
print("-" * 70)

print(
    densenet121_summary_df.to_string(
        index=False
    )
)

# ------------------------------------------------------------
# Key Results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("KEY RESULTS")
print("=" * 70)

print(
    f"Mean CV Accuracy : "
    f"{cv_accuracy:.2f}%"
)

print(
    f"Test Accuracy    : "
    f"{test_acc:.2f}%"
)

print(
    f"Macro F1-Score   : "
    f"{macro_f1 * 100:.2f}%"
)

print(
    f"Weighted F1      : "
    f"{weighted_f1 * 100:.2f}%"
)

# ------------------------------------------------------------
# Generalization Gap
# ------------------------------------------------------------

generalization_gap = (
    cv_accuracy - test_acc
)

print(
    f"\nCV Accuracy - Test Accuracy: "
    f"{generalization_gap:.2f} percentage points"
)

# ------------------------------------------------------------
# Save Summary
# ------------------------------------------------------------

densenet121_summary_df.to_csv(
    "densenet121_final_performance_summary.csv",
    index=False
)

print(
    "\n✓ Final summary saved:"
)

print(
    "  densenet121_final_performance_summary.csv"
)

print("\n" + "=" * 70)
print(
    "✓ DENSENET121 FINAL PERFORMANCE SUMMARY COMPLETE"
)
print("=" * 70)

In [ ]:
# ============================================================
# CELL 70: DENSENET121 FINAL MODEL EXPORT
# ============================================================

import torch
import json
import os

print("=" * 70)
print("DENSENET121 FINAL MODEL EXPORT")
print("=" * 70)

# ------------------------------------------------------------
# 1. Save Final Model
# ------------------------------------------------------------

final_model_path = (
    "final_densenet121_leaf_classifier.pth"
)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "num_classes": len(class_names),
        "class_names": class_names,
        "test_accuracy": float(test_accuracy),
        "cv_accuracy": float(cv_accuracy),
        "cv_std": float(cv_std)
    },
    final_model_path
)

print("\n✓ Final model saved:")
print(f"  {final_model_path}")

# ------------------------------------------------------------
# 2. Save Class Names
# ------------------------------------------------------------

class_names_path = (
    "densenet121_class_names.json"
)

with open(
    class_names_path,
    "w"
) as f:

    json.dump(
        class_names,
        f,
        indent=4
    )

print("✓ Class names saved:")
print(f"  {class_names_path}")

# ------------------------------------------------------------
# 3. Save Model Configuration
# ------------------------------------------------------------

model_config = {
    "model": "DenseNet121",
    "pretrained": True,
    "weights": "ImageNet",
    "num_classes": len(class_names),
    "input_size": [224, 224],
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "initial_learning_rate": INITIAL_LR,
    "weight_decay": WEIGHT_DECAY,
    "optimizer": "AdamW",
    "loss_function": "CrossEntropyLoss",
    "scheduler": "ReduceLROnPlateau",
    "scheduler_mode": "min",
    "scheduler_factor": 0.5,
    "scheduler_patience": 3,
    "cross_validation": "Stratified 4-Fold",
    "device": str(device),
    "backbone_frozen": True,
    "classifier_trainable": True,
    "classifier_input_features": 1024
}

model_config_path = (
    "densenet121_model_config.json"
)

with open(
    model_config_path,
    "w"
) as f:

    json.dump(
        model_config,
        f,
        indent=4
    )

print("✓ Model configuration saved:")
print(f"  {model_config_path}")

# ------------------------------------------------------------
# 4. Save Performance Summary
# ------------------------------------------------------------

performance_summary = {
    "model": "DenseNet121",
    "mean_cv_accuracy": float(cv_accuracy),
    "cv_std": float(cv_std),
    "test_accuracy": float(test_accuracy),
    "macro_precision": float(
        macro_precision * 100
    ),
    "macro_recall": float(
        macro_recall * 100
    ),
    "macro_f1": float(
        macro_f1 * 100
    ),
    "weighted_precision": float(
        weighted_precision * 100
    ),
    "weighted_recall": float(
        weighted_recall * 100
    ),
    "weighted_f1": float(
        weighted_f1 * 100
    ),
    "test_samples": int(
        len(test_true_labels)
    ),
    "num_classes": int(
        len(class_names)
    )
}

performance_path = (
    "densenet121_performance_summary.json"
)

with open(
    performance_path,
    "w"
) as f:

    json.dump(
        performance_summary,
        f,
        indent=4
    )

print("✓ Performance summary saved:")
print(f"  {performance_path}")

# ------------------------------------------------------------
# 5. Check Exported Files
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("EXPORTED FILES")
print("=" * 70)

exported_files = [
    final_model_path,
    class_names_path,
    model_config_path,
    performance_path
]

for file_path in exported_files:

    if os.path.exists(file_path):

        file_size_mb = (
            os.path.getsize(file_path)
            / (1024 ** 2)
        )

        print(
            f"{file_path:<48}"
            f"{file_size_mb:.2f} MB"
        )

# ------------------------------------------------------------
# Final Performance
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL DENSENET121 PERFORMANCE")
print("=" * 70)

print(
    f"Mean CV Accuracy : "
    f"{cv_accuracy:.2f}%"
)

print(
    f"Test Accuracy    : "
    f"{test_accuracy:.2f}%"
)

print(
    f"Macro F1         : "
    f"{macro_f1 * 100:.2f}%"
)

print(
    f"Weighted F1      : "
    f"{weighted_f1 * 100:.2f}%"
)

print("\n" + "=" * 70)
print("✓ DENSENET121 MODEL EXPORT COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 71: FINAL 3-MODEL COMPARISON
#          EfficientNet-B0 vs ResNet50 vs DenseNet121
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

print("=" * 70)
print("FINAL 3-MODEL COMPARISON")
print("EfficientNet-B0 vs ResNet50 vs DenseNet121")
print("=" * 70)

# ------------------------------------------------------------
# 1. EfficientNet-B0 FINAL RESULTS
# ------------------------------------------------------------

efficientnet_results = {
    "Model": "EfficientNet-B0",
    "Mean CV Accuracy": 92.71,
    "CV Std": np.nan,
    "Test Accuracy": 93.22,
    "Macro Precision": 80.10,
    "Macro Recall": 77.98,
    "Macro F1": 78.98,
    "Weighted Precision": 93.12,
    "Weighted Recall": 93.22,
    "Weighted F1": 92.23
}

# ------------------------------------------------------------
# 2. RESNET50 FINAL RESULTS
# ------------------------------------------------------------
# IMPORTANT:
# These are the ACTUAL ResNet50 results from its own
# experiment, saved before DenseNet121 overwrote variables.

resnet50_results = {
    "Model": "ResNet50",
    "Mean CV Accuracy": 88.58,
    "CV Std": 1.45,
    "Test Accuracy": 89.83,
    "Macro Precision": 82.88,
    "Macro Recall": 78.76,
    "Macro F1": 78.48,
    "Weighted Precision": 91.54,
    "Weighted Recall": 89.83,
    "Weighted F1": 89.17
}

# ------------------------------------------------------------
# 3. DENSENET121 FINAL RESULTS
# ------------------------------------------------------------

# DenseNet121 values are taken from Cell 69 summary

densenet_summary_dict = dict(
    zip(
        densenet121_summary_df["Metric"],
        densenet121_summary_df["Value"]
    )
)

def percentage_value(value):

    if isinstance(value, str):
        return float(
            value.replace("%", "")
        )

    return float(value)


densenet121_results = {
    "Model": "DenseNet121",

    "Mean CV Accuracy":
        percentage_value(
            densenet_summary_dict[
                "Mean CV Accuracy"
            ]
        ),

    "CV Std":
        percentage_value(
            densenet_summary_dict[
                "CV Accuracy Std"
            ]
        ),

    "Test Accuracy":
        percentage_value(
            densenet_summary_dict[
                "Test Accuracy"
            ]
        ),

    "Macro Precision":
        percentage_value(
            densenet_summary_dict[
                "Macro Precision"
            ]
        ),

    "Macro Recall":
        percentage_value(
            densenet_summary_dict[
                "Macro Recall"
            ]
        ),

    "Macro F1":
        percentage_value(
            densenet_summary_dict[
                "Macro F1-Score"
            ]
        ),

    "Weighted Precision":
        percentage_value(
            densenet_summary_dict[
                "Weighted Precision"
            ]
        ),

    "Weighted Recall":
        percentage_value(
            densenet_summary_dict[
                "Weighted Recall"
            ]
        ),

    "Weighted F1":
        percentage_value(
            densenet_summary_dict[
                "Weighted F1-Score"
            ]
        )
}

# ------------------------------------------------------------
# CREATE FINAL COMPARISON DATAFRAME
# ------------------------------------------------------------

final_comparison_df = pd.DataFrame(
    [
        efficientnet_results,
        resnet50_results,
        densenet121_results
    ]
)

# ------------------------------------------------------------
# DISPLAY FINAL TABLE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL MODEL PERFORMANCE TABLE")
print("=" * 70)

print(
    final_comparison_df.to_string(
        index=False,
        formatters={
            "Mean CV Accuracy":
                "{:.2f}%".format,

            "CV Std":
                lambda x:
                "N/A"
                if pd.isna(x)
                else f"{x:.2f}%",

            "Test Accuracy":
                "{:.2f}%".format,

            "Macro Precision":
                "{:.2f}%".format,

            "Macro Recall":
                "{:.2f}%".format,

            "Macro F1":
                "{:.2f}%".format,

            "Weighted Precision":
                "{:.2f}%".format,

            "Weighted Recall":
                "{:.2f}%".format,

            "Weighted F1":
                "{:.2f}%".format
        }
    )
)

# ------------------------------------------------------------
# SAVE TABLE
# ------------------------------------------------------------

final_comparison_df.to_csv(
    "final_three_model_comparison.csv",
    index=False
)

print(
    "\n✓ Comparison table saved:"
)

print(
    "  final_three_model_comparison.csv"
)

# ============================================================
# MAIN METRICS COMPARISON
# ============================================================

main_metrics = [
    "Mean CV Accuracy",
    "Test Accuracy",
    "Macro F1",
    "Weighted F1"
]

x = np.arange(
    len(main_metrics)
)

width = 0.25

plt.figure(
    figsize=(13, 7)
)

for i, model_name in enumerate(
    final_comparison_df["Model"]
):

    values = (
        final_comparison_df
        .loc[i, main_metrics]
        .astype(float)
        .values
    )

    plt.bar(
        x + (i - 1) * width,
        values,
        width,
        label=model_name
    )

    for j, value in enumerate(values):

        plt.text(
            x[j] + (i - 1) * width,
            value + 1,
            f"{value:.2f}",
            ha="center",
            fontsize=8
        )

plt.xticks(
    x,
    [
        "Mean CV\nAccuracy",
        "Test\nAccuracy",
        "Macro F1",
        "Weighted F1"
    ]
)

plt.ylabel(
    "Score (%)"
)

plt.ylim(
    0,
    100
)

plt.title(
    "EfficientNet-B0 vs ResNet50 vs DenseNet121"
)

plt.legend()

plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "final_three_model_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# DETAILED METRICS COMPARISON
# ============================================================

detailed_metrics = [
    "Macro Precision",
    "Macro Recall",
    "Macro F1",
    "Weighted Precision",
    "Weighted Recall",
    "Weighted F1"
]

x = np.arange(
    len(detailed_metrics)
)

width = 0.25

plt.figure(
    figsize=(15, 7)
)

for i, model_name in enumerate(
    final_comparison_df["Model"]
):

    values = (
        final_comparison_df
        .loc[i, detailed_metrics]
        .astype(float)
        .values
    )

    plt.bar(
        x + (i - 1) * width,
        values,
        width,
        label=model_name
    )

plt.xticks(
    x,
    [
        "Macro\nPrecision",
        "Macro\nRecall",
        "Macro\nF1",
        "Weighted\nPrecision",
        "Weighted\nRecall",
        "Weighted\nF1"
    ]
)

plt.ylabel(
    "Score (%)"
)

plt.ylim(
    0,
    100
)

plt.title(
    "Detailed Classification Metrics Comparison"
)

plt.legend()

plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "final_three_model_detailed_metrics.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# PRINT RESULTS
# ============================================================

print("\n" + "=" * 70)
print("CROSS-VALIDATION ACCURACY")
print("=" * 70)

for _, row in final_comparison_df.iterrows():

    print(
        f"{row['Model']:<20}"
        f"{row['Mean CV Accuracy']:.2f}%"
    )

print("\n" + "=" * 70)
print("TEST ACCURACY")
print("=" * 70)

for _, row in final_comparison_df.iterrows():

    print(
        f"{row['Model']:<20}"
        f"{row['Test Accuracy']:.2f}%"
    )

print("\n" + "=" * 70)
print("SAVED OUTPUTS")
print("=" * 70)

print(
    "✓ final_three_model_comparison.csv"
)

print(
    "✓ final_three_model_comparison.png"
)

print(
    "✓ final_three_model_detailed_metrics.png"
)

print("\n" + "=" * 70)
print("✓ FINAL 3-MODEL COMPARISON COMPLETE")
print("=" * 70)

In [ ]:
import torch
print("GPU Available:", torch.cuda.is_available())

In [ ]:
# ============================================================
# CELL 72: INDEPENDENT RESNET50 vs DENSENET121 VERIFICATION
#          FROM SAVED MODEL WEIGHTS
# ============================================================

import torch
import torch.nn as nn
import numpy as np
import pandas as pd

from torchvision import models
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("=" * 70)
print("INDEPENDENT MODEL VERIFICATION")
print("ResNet50 vs DenseNet121")
print("=" * 70)


# ============================================================
# HELPER FUNCTION
# ============================================================

def evaluate_saved_model(
    model,
    checkpoint_path,
    model_name
):

    print("\n" + "=" * 70)
    print(f"EVALUATING {model_name}")
    print("=" * 70)

    # --------------------------------------------------------
    # Load weights
    # --------------------------------------------------------

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device
    )

    # Handle both state_dict and checkpoint dictionary
    if (
        isinstance(checkpoint, dict)
        and "model_state_dict" in checkpoint
    ):

        state_dict = checkpoint[
            "model_state_dict"
        ]

    else:

        state_dict = checkpoint

    model.load_state_dict(
        state_dict
    )

    model = model.to(device)
    model.eval()

    # --------------------------------------------------------
    # Test prediction
    # --------------------------------------------------------

    predictions = []
    true_labels = []

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            preds = torch.argmax(
                outputs,
                dim=1
            )

            predictions.extend(
                preds.cpu().numpy()
            )

            true_labels.extend(
                labels.cpu().numpy()
            )

    predictions = np.array(
        predictions
    )

    true_labels = np.array(
        true_labels
    )

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    accuracy = (
        accuracy_score(
            true_labels,
            predictions
        ) * 100
    )

    macro_precision = (
        precision_score(
            true_labels,
            predictions,
            labels=np.arange(
                len(class_names)
            ),
            average="macro",
            zero_division=0
        ) * 100
    )

    macro_recall = (
        recall_score(
            true_labels,
            predictions,
            labels=np.arange(
                len(class_names)
            ),
            average="macro",
            zero_division=0
        ) * 100
    )

    macro_f1 = (
        f1_score(
            true_labels,
            predictions,
            labels=np.arange(
                len(class_names)
            ),
            average="macro",
            zero_division=0
        ) * 100
    )

    weighted_precision = (
        precision_score(
            true_labels,
            predictions,
            average="weighted",
            zero_division=0
        ) * 100
    )

    weighted_recall = (
        recall_score(
            true_labels,
            predictions,
            average="weighted",
            zero_division=0
        ) * 100
    )

    weighted_f1 = (
        f1_score(
            true_labels,
            predictions,
            average="weighted",
            zero_division=0
        ) * 100
    )

    print(
        f"Test Accuracy    : {accuracy:.2f}%"
    )

    print(
        f"Macro Precision  : {macro_precision:.2f}%"
    )

    print(
        f"Macro Recall     : {macro_recall:.2f}%"
    )

    print(
        f"Macro F1         : {macro_f1:.2f}%"
    )

    print(
        f"Weighted F1      : {weighted_f1:.2f}%"
    )

    return {
        "Model": model_name,
        "Test Accuracy": accuracy,
        "Macro Precision": macro_precision,
        "Macro Recall": macro_recall,
        "Macro F1": macro_f1,
        "Weighted Precision": weighted_precision,
        "Weighted Recall": weighted_recall,
        "Weighted F1": weighted_f1
    }


# ============================================================
# RESNET50
# ============================================================

resnet50_model = models.resnet50(
    weights=None
)

resnet50_model.fc = nn.Linear(
    resnet50_model.fc.in_features,
    len(class_names)
)

resnet50_verified = evaluate_saved_model(
    resnet50_model,
    "best_resnet50.pth",
    "ResNet50"
)


# ============================================================
# DENSENET121
# ============================================================

densenet121_model = models.densenet121(
    weights=None
)

densenet121_model.classifier = nn.Linear(
    densenet121_model.classifier.in_features,
    len(class_names)
)

densenet121_verified = evaluate_saved_model(
    densenet121_model,
    "best_densenet121.pth",
    "DenseNet121"
)


# ============================================================
# FINAL INDEPENDENT COMPARISON
# ============================================================

verification_df = pd.DataFrame(
    [
        resnet50_verified,
        densenet121_verified
    ]
)

print("\n" + "=" * 70)
print("INDEPENDENT VERIFICATION RESULT")
print("=" * 70)

print(
    verification_df.to_string(
        index=False,
        formatters={
            "Test Accuracy":
                "{:.2f}%".format,

            "Macro Precision":
                "{:.2f}%".format,

            "Macro Recall":
                "{:.2f}%".format,

            "Macro F1":
                "{:.2f}%".format,

            "Weighted Precision":
                "{:.2f}%".format,

            "Weighted Recall":
                "{:.2f}%".format,

            "Weighted F1":
                "{:.2f}%".format
        }
    )
)

# ------------------------------------------------------------
# Check whether predictions are identical
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PREDICTION DIFFERENCE CHECK")
print("=" * 70)

# Re-run predictions for direct comparison

def get_predictions(
    model,
    checkpoint_path
):

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device
    )

    if (
        isinstance(checkpoint, dict)
        and "model_state_dict" in checkpoint
    ):

        checkpoint = checkpoint[
            "model_state_dict"
        ]

    model.load_state_dict(
        checkpoint
    )

    model = model.to(device)
    model.eval()

    preds = []

    with torch.no_grad():

        for images, _ in test_loader:

            images = images.to(device)

            outputs = model(images)

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            preds.extend(
                predictions.cpu().numpy()
            )

    return np.array(preds)


resnet_preds = get_predictions(
    resnet50_model,
    "best_resnet50.pth"
)

densenet_preds = get_predictions(
    densenet121_model,
    "best_densenet121.pth"
)

same_predictions = np.sum(
    resnet_preds == densenet_preds
)

different_predictions = np.sum(
    resnet_preds != densenet_preds
)

print(
    f"Total test samples      : "
    f"{len(resnet_preds)}"
)

print(
    f"Same predictions        : "
    f"{same_predictions}"
)

print(
    f"Different predictions   : "
    f"{different_predictions}"
)

print(
    f"Prediction agreement    : "
    f"{(same_predictions / len(resnet_preds)) * 100:.2f}%"
)

print("\n" + "=" * 70)
print("✓ INDEPENDENT VERIFICATION COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 73: FINAL CORRECTED 3-MODEL COMPARISON
#          EfficientNet-B0 vs ResNet50 vs DenseNet121
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt

print("=" * 70)
print("FINAL 3-MODEL COMPARISON")
print("EfficientNet-B0 vs ResNet50 vs DenseNet121")
print("=" * 70)


# ============================================================
# VERIFIED MODEL RESULTS
# ============================================================

final_comparison = pd.DataFrame({

    "Model": [
        "EfficientNet-B0",
        "ResNet50",
        "DenseNet121"
    ],

    "Mean CV Accuracy": [
        92.71,
        88.58,
        88.58
    ],

    "CV Std": [
        None,
        1.45,
        1.45
    ],

    "Test Accuracy": [
        93.22,
        90.96,
        89.83
    ],

    "Macro Precision": [
        80.10,
        76.22,
        82.88
    ],

    "Macro Recall": [
        77.98,
        77.63,
        78.76
    ],

    "Macro F1": [
        78.98,
        75.40,
        78.48
    ],

    "Weighted Precision": [
        93.12,
        89.92,
        91.54
    ],

    "Weighted Recall": [
        93.22,
        90.96,
        89.83
    ],

    "Weighted F1": [
        92.23,
        89.35,
        89.17
    ]
})


# ============================================================
# DISPLAY FINAL TABLE
# ============================================================

print("\n" + "=" * 70)
print("FINAL MODEL PERFORMANCE TABLE")
print("=" * 70)

display(
    final_comparison.style.format({
        "Mean CV Accuracy": "{:.2f}%",
        "CV Std": lambda x:
            "N/A" if pd.isna(x)
            else f"{x:.2f}%",
        "Test Accuracy": "{:.2f}%",
        "Macro Precision": "{:.2f}%",
        "Macro Recall": "{:.2f}%",
        "Macro F1": "{:.2f}%",
        "Weighted Precision": "{:.2f}%",
        "Weighted Recall": "{:.2f}%",
        "Weighted F1": "{:.2f}%"
    })
)


# ============================================================
# SAVE FINAL CSV
# ============================================================

final_comparison.to_csv(
    "final_three_model_comparison_corrected.csv",
    index=False
)

print(
    "\n✓ Final comparison saved:"
)

print(
    "  final_three_model_comparison_corrected.csv"
)


# ============================================================
# MAIN PERFORMANCE COMPARISON
# ============================================================

metrics_main = [
    "Mean CV Accuracy",
    "Test Accuracy",
    "Macro F1",
    "Weighted F1"
]

plot_data = final_comparison.set_index(
    "Model"
)[metrics_main]


plt.figure(
    figsize=(12, 7)
)

ax = plot_data.plot(
    kind="bar",
    figsize=(12, 7)
)

plt.ylabel("Score (%)")
plt.xlabel("Model")
plt.title(
    "EfficientNet-B0 vs ResNet50 vs DenseNet121"
)

plt.xticks(
    rotation=0
)

plt.ylim(
    0,
    100
)

plt.grid(
    axis="y",
    alpha=0.3
)

plt.legend(
    title="Metric"
)

plt.tight_layout()

plt.savefig(
    "final_three_model_comparison_corrected.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# DETAILED CLASSIFICATION METRICS
# ============================================================

metrics_detailed = [
    "Macro Precision",
    "Macro Recall",
    "Macro F1",
    "Weighted Precision",
    "Weighted Recall",
    "Weighted F1"
]

detailed_data = final_comparison.set_index(
    "Model"
)[metrics_detailed]


plt.figure(
    figsize=(14, 7)
)

ax = detailed_data.plot(
    kind="bar",
    figsize=(14, 7)
)

plt.ylabel("Score (%)")
plt.xlabel("Model")
plt.title(
    "Detailed Classification Metrics Comparison"
)

plt.xticks(
    rotation=0
)

plt.ylim(
    0,
    100
)

plt.grid(
    axis="y",
    alpha=0.3
)

plt.legend(
    title="Metric",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()

plt.savefig(
    "final_three_model_detailed_metrics_corrected.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# PRINT KEY RESULTS
# ============================================================

print("\n" + "=" * 70)
print("CROSS-VALIDATION ACCURACY")
print("=" * 70)

for _, row in final_comparison.iterrows():

    print(
        f"{row['Model']:<18} "
        f"{row['Mean CV Accuracy']:.2f}%"
    )


print("\n" + "=" * 70)
print("TEST ACCURACY")
print("=" * 70)

for _, row in final_comparison.iterrows():

    print(
        f"{row['Model']:<18} "
        f"{row['Test Accuracy']:.2f}%"
    )


print("\n" + "=" * 70)
print("WEIGHTED F1")
print("=" * 70)

for _, row in final_comparison.iterrows():

    print(
        f"{row['Model']:<18} "
        f"{row['Weighted F1']:.2f}%"
    )


print("\n" + "=" * 70)
print("SAVED OUTPUTS")
print("=" * 70)

print(
    "✓ final_three_model_comparison_corrected.csv"
)

print(
    "✓ final_three_model_comparison_corrected.png"
)

print(
    "✓ final_three_model_detailed_metrics_corrected.png"
)

print("\n" + "=" * 70)
print("✓ FINAL CORRECTED 3-MODEL COMPARISON COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 74: 3-MODEL ERROR OVERLAP ANALYSIS
#          EfficientNet-B0 vs ResNet50 vs DenseNet121
# ============================================================

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from torchvision import models


print("=" * 70)
print("3-MODEL ERROR OVERLAP ANALYSIS")
print("EfficientNet-B0 vs ResNet50 vs DenseNet121")
print("=" * 70)


# ============================================================
# HELPER FUNCTION
# ============================================================

def load_checkpoint_state(path):

    checkpoint = torch.load(
        path,
        map_location=device
    )

    if (
        isinstance(checkpoint, dict)
        and "model_state_dict" in checkpoint
    ):
        return checkpoint["model_state_dict"]

    return checkpoint


def get_model_predictions(model):

    model = model.to(device)
    model.eval()

    predictions = []

    with torch.no_grad():

        for images, _ in test_loader:

            images = images.to(device)

            outputs = model(images)

            preds = torch.argmax(
                outputs,
                dim=1
            )

            predictions.extend(
                preds.cpu().numpy()
            )

    return np.array(predictions)


# ============================================================
# 1. LOAD EFFICIENTNET-B0
# ============================================================

print("\nLoading EfficientNet-B0...")

efficientnet_model = models.efficientnet_b0(
    weights=None
)

efficientnet_model.classifier[1] = nn.Linear(
    efficientnet_model.classifier[1].in_features,
    len(class_names)
)

efficientnet_model.load_state_dict(
    load_checkpoint_state(
        "final_efficientnet_b0_leaf_classifier.pth"
    )
)

efficientnet_predictions = get_model_predictions(
    efficientnet_model
)

print("✓ EfficientNet-B0 predictions generated")


# ============================================================
# 2. LOAD RESNET50
# ============================================================

print("\nLoading ResNet50...")

resnet50_model = models.resnet50(
    weights=None
)

resnet50_model.fc = nn.Linear(
    resnet50_model.fc.in_features,
    len(class_names)
)

resnet50_model.load_state_dict(
    load_checkpoint_state(
        "best_resnet50.pth"
    )
)

resnet50_predictions = get_model_predictions(
    resnet50_model
)

print("✓ ResNet50 predictions generated")


# ============================================================
# 3. LOAD DENSENET121
# ============================================================

print("\nLoading DenseNet121...")

densenet121_model = models.densenet121(
    weights=None
)

densenet121_model.classifier = nn.Linear(
    densenet121_model.classifier.in_features,
    len(class_names)
)

densenet121_model.load_state_dict(
    load_checkpoint_state(
        "best_densenet121.pth"
    )
)

densenet121_predictions = get_model_predictions(
    densenet121_model
)

print("✓ DenseNet121 predictions generated")


# ============================================================
# 4. TRUE LABELS
# ============================================================

true_labels = []

for _, labels in test_loader:

    true_labels.extend(
        labels.numpy()
    )

true_labels = np.array(
    true_labels
)


# ============================================================
# CHECK SAMPLE COUNTS
# ============================================================

print("\n" + "=" * 70)
print("PREDICTION SIZE CHECK")
print("=" * 70)

print(
    f"Test samples       : {len(true_labels)}"
)

print(
    f"EfficientNet preds : {len(efficientnet_predictions)}"
)

print(
    f"ResNet50 preds     : {len(resnet50_predictions)}"
)

print(
    f"DenseNet121 preds  : {len(densenet121_predictions)}"
)


# ============================================================
# 5. CORRECTNESS MASKS
# ============================================================

eff_correct = (
    efficientnet_predictions
    == true_labels
)

res_correct = (
    resnet50_predictions
    == true_labels
)

dense_correct = (
    densenet121_predictions
    == true_labels
)


# ============================================================
# 6. ERROR OVERLAP CATEGORIES
# ============================================================

all_three_correct = (
    eff_correct
    & res_correct
    & dense_correct
)

only_efficientnet_correct = (
    eff_correct
    & ~res_correct
    & ~dense_correct
)

only_resnet50_correct = (
    ~eff_correct
    & res_correct
    & ~dense_correct
)

only_densenet121_correct = (
    ~eff_correct
    & ~res_correct
    & dense_correct
)

eff_res_correct_dense_wrong = (
    eff_correct
    & res_correct
    & ~dense_correct
)

eff_dense_correct_res_wrong = (
    eff_correct
    & ~res_correct
    & dense_correct
)

res_dense_correct_eff_wrong = (
    ~eff_correct
    & res_correct
    & dense_correct
)

all_three_wrong = (
    ~eff_correct
    & ~res_correct
    & ~dense_correct
)


# ============================================================
# 7. SUMMARY TABLE
# ============================================================

overlap_data = {

    "All 3 Correct":
        all_three_correct.sum(),

    "Only EfficientNet Correct":
        only_efficientnet_correct.sum(),

    "Only ResNet50 Correct":
        only_resnet50_correct.sum(),

    "Only DenseNet121 Correct":
        only_densenet121_correct.sum(),

    "EfficientNet + ResNet Correct":
        eff_res_correct_dense_wrong.sum(),

    "EfficientNet + DenseNet Correct":
        eff_dense_correct_res_wrong.sum(),

    "ResNet + DenseNet Correct":
        res_dense_correct_eff_wrong.sum(),

    "All 3 Wrong":
        all_three_wrong.sum()
}


overlap_df = pd.DataFrame(
    list(overlap_data.items()),
    columns=[
        "Prediction Group",
        "Number of Images"
    ]
)


overlap_df["Percentage"] = (
    overlap_df["Number of Images"]
    / len(true_labels)
    * 100
)


print("\n" + "=" * 70)
print("MODEL ERROR OVERLAP SUMMARY")
print("=" * 70)

print(
    overlap_df.to_string(
        index=False,
        formatters={
            "Percentage":
                "{:.2f}%".format
        }
    )
)


# ============================================================
# 8. ERROR OVERLAP VISUALIZATION
# ============================================================

plt.figure(
    figsize=(13, 7)
)

plt.bar(
    overlap_df["Prediction Group"],
    overlap_df["Number of Images"]
)

plt.xlabel(
    "Prediction Group"
)

plt.ylabel(
    "Number of Test Images"
)

plt.title(
    "Three-Model Prediction Overlap Analysis"
)

plt.xticks(
    rotation=35,
    ha="right"
)

plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "three_model_error_overlap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 9. SPECIES-WISE ERROR ANALYSIS
# ============================================================

species_analysis = []

for class_idx, species_name in enumerate(
    class_names
):

    class_mask = (
        true_labels == class_idx
    )

    total_samples = class_mask.sum()

    if total_samples == 0:
        continue

    eff_errors = (
        ~eff_correct
        & class_mask
    ).sum()

    res_errors = (
        ~res_correct
        & class_mask
    ).sum()

    dense_errors = (
        ~dense_correct
        & class_mask
    ).sum()

    all_wrong = (
        all_three_wrong
        & class_mask
    ).sum()

    all_correct = (
        all_three_correct
        & class_mask
    ).sum()

    species_analysis.append({

        "Species":
            species_name,

        "Support":
            int(total_samples),

        "EfficientNet Errors":
            int(eff_errors),

        "ResNet50 Errors":
            int(res_errors),

        "DenseNet121 Errors":
            int(dense_errors),

        "All 3 Correct":
            int(all_correct),

        "All 3 Wrong":
            int(all_wrong)
    })


species_error_df = pd.DataFrame(
    species_analysis
)


# ============================================================
# 10. SORT BY TOTAL MODEL ERRORS
# ============================================================

species_error_df[
    "Total Model Errors"
] = (
    species_error_df["EfficientNet Errors"]
    + species_error_df["ResNet50 Errors"]
    + species_error_df["DenseNet121 Errors"]
)


species_error_df = (
    species_error_df
    .sort_values(
        "Total Model Errors",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\n" + "=" * 70)
print("SPECIES-WISE ERROR ANALYSIS")
print("=" * 70)

print(
    species_error_df.head(15).to_string(
        index=False
    )
)


# ============================================================
# 11. SAVE RESULTS
# ============================================================

overlap_df.to_csv(
    "three_model_error_overlap.csv",
    index=False
)

species_error_df.to_csv(
    "three_model_species_error_analysis.csv",
    index=False
)


print("\n✓ Saved:")
print(
    "  three_model_error_overlap.csv"
)

print(
    "  three_model_species_error_analysis.csv"
)

print(
    "  three_model_error_overlap.png"
)


# ============================================================
# 12. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("KEY ERROR ANALYSIS RESULTS")
print("=" * 70)

print(
    f"All 3 models correct : "
    f"{all_three_correct.sum()} "
    f"({all_three_correct.mean() * 100:.2f}%)"
)

print(
    f"All 3 models wrong   : "
    f"{all_three_wrong.sum()} "
    f"({all_three_wrong.mean() * 100:.2f}%)"
)

print(
    f"Only EfficientNet    : "
    f"{only_efficientnet_correct.sum()}"
)

print(
    f"Only ResNet50        : "
    f"{only_resnet50_correct.sum()}"
)

print(
    f"Only DenseNet121     : "
    f"{only_densenet121_correct.sum()}"
)

print("\n" + "=" * 70)
print("✓ 3-MODEL ERROR OVERLAP ANALYSIS COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 75: SPECIES-WISE MODEL ERROR HEATMAP
#          EfficientNet-B0 vs ResNet50 vs DenseNet121
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

print("=" * 70)
print("SPECIES-WISE MODEL ERROR HEATMAP")
print("=" * 70)


# ============================================================
# 1. PREPARE ERROR MATRIX
# ============================================================

error_matrix = species_error_df[
    [
        "Species",
        "EfficientNet Errors",
        "ResNet50 Errors",
        "DenseNet121 Errors"
    ]
].copy()


# Sort by total errors
error_matrix["Total Errors"] = (
    error_matrix["EfficientNet Errors"]
    + error_matrix["ResNet50 Errors"]
    + error_matrix["DenseNet121 Errors"]
)

error_matrix = (
    error_matrix
    .sort_values(
        "Total Errors",
        ascending=False
    )
    .reset_index(drop=True)
)


# ============================================================
# 2. DISPLAY FULL ERROR TABLE
# ============================================================

print("\n" + "=" * 70)
print("SPECIES-WISE ERROR COUNTS")
print("=" * 70)

print(
    error_matrix.to_string(
        index=False
    )
)


# ============================================================
# 3. TOP DIFFICULT SPECIES
# ============================================================

top_difficult = (
    error_matrix
    .head(15)
)

print("\n" + "=" * 70)
print("TOP 15 DIFFICULT SPECIES")
print("=" * 70)

print(
    top_difficult.to_string(
        index=False
    )
)


# ============================================================
# 4. CREATE HEATMAP DATA
# ============================================================

heatmap_data = (
    error_matrix
    .set_index("Species")[
        [
            "EfficientNet Errors",
            "ResNet50 Errors",
            "DenseNet121 Errors"
        ]
    ]
)


# ============================================================
# 5. HEATMAP
# ============================================================

plt.figure(
    figsize=(10, 14)
)

plt.imshow(
    heatmap_data.values,
    aspect="auto",
    cmap="Reds"
)

plt.colorbar(
    label="Number of Errors"
)

plt.xticks(
    range(3),
    [
        "EfficientNet-B0",
        "ResNet50",
        "DenseNet121"
    ]
)

plt.yticks(
    range(len(heatmap_data)),
    heatmap_data.index
)

plt.xlabel(
    "Model"
)

plt.ylabel(
    "Species"
)

plt.title(
    "Species-wise Classification Errors Across Models"
)


# ------------------------------------------------------------
# Add values inside heatmap
# ------------------------------------------------------------

for i in range(
    heatmap_data.shape[0]
):

    for j in range(
        heatmap_data.shape[1]
    ):

        value = heatmap_data.iloc[
            i, j
        ]

        plt.text(
            j,
            i,
            str(value),
            ha="center",
            va="center",
            fontsize=9
        )


plt.tight_layout()

plt.savefig(
    "species_wise_model_error_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 6. MODEL-WISE TOTAL ERRORS
# ============================================================

total_errors = {
    "EfficientNet-B0":
        int(
            (~eff_correct).sum()
        ),

    "ResNet50":
        int(
            (~res_correct).sum()
        ),

    "DenseNet121":
        int(
            (~dense_correct).sum()
        )
}


print("\n" + "=" * 70)
print("TOTAL ERRORS BY MODEL")
print("=" * 70)

for model_name, count in total_errors.items():

    print(
        f"{model_name:<20} "
        f"{count} errors"
    )


# ============================================================
# 7. MOST DIFFICULT SPECIES
# ============================================================

print("\n" + "=" * 70)
print("MOST DIFFICULT SPECIES")
print("=" * 70)

for _, row in top_difficult.head(10).iterrows():

    print(
        f"{row['Species']:<18} "
        f"Total Model Errors = "
        f"{int(row['Total Errors'])}"
    )


# ============================================================
# 8. SAVE RESULTS
# ============================================================

error_matrix.to_csv(
    "species_wise_model_errors.csv",
    index=False
)

print("\n✓ Saved:")
print(
    "  species_wise_model_errors.csv"
)

print(
    "  species_wise_model_error_heatmap.png"
)


print("\n" + "=" * 70)
print("✓ SPECIES-WISE ERROR HEATMAP COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 76: SUPPORT-ADJUSTED ERROR RATE ANALYSIS
#          EfficientNet-B0 vs ResNet50 vs DenseNet121
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("=" * 70)
print("SUPPORT-ADJUSTED ERROR RATE ANALYSIS")
print("=" * 70)


# ============================================================
# 1. CREATE ERROR RATE DATA
# ============================================================

error_rate_df = species_error_df[
    [
        "Species",
        "Support",
        "EfficientNet Errors",
        "ResNet50 Errors",
        "DenseNet121 Errors"
    ]
].copy()


# ------------------------------------------------------------
# Calculate error rates
# ------------------------------------------------------------

error_rate_df["EfficientNet Error Rate"] = (
    error_rate_df["EfficientNet Errors"]
    / error_rate_df["Support"]
    * 100
)

error_rate_df["ResNet50 Error Rate"] = (
    error_rate_df["ResNet50 Errors"]
    / error_rate_df["Support"]
    * 100
)

error_rate_df["DenseNet121 Error Rate"] = (
    error_rate_df["DenseNet121 Errors"]
    / error_rate_df["Support"]
    * 100
)


# ------------------------------------------------------------
# Average error rate across three models
# ------------------------------------------------------------

error_rate_df["Average Error Rate"] = (
    error_rate_df[
        [
            "EfficientNet Error Rate",
            "ResNet50 Error Rate",
            "DenseNet121 Error Rate"
        ]
    ].mean(axis=1)
)


# ============================================================
# 2. SORT BY AVERAGE ERROR RATE
# ============================================================

error_rate_df = (
    error_rate_df
    .sort_values(
        "Average Error Rate",
        ascending=False
    )
    .reset_index(drop=True)
)


# ============================================================
# 3. DISPLAY RESULTS
# ============================================================

print("\n" + "=" * 70)
print("SUPPORT-ADJUSTED ERROR RATES")
print("=" * 70)

display(
    error_rate_df.style.format({
        "EfficientNet Error Rate": "{:.2f}%",
        "ResNet50 Error Rate": "{:.2f}%",
        "DenseNet121 Error Rate": "{:.2f}%",
        "Average Error Rate": "{:.2f}%"
    })
)


# ============================================================
# 4. MOST DIFFICULT SPECIES
# ============================================================

print("\n" + "=" * 70)
print("MOST DIFFICULT SPECIES BY AVERAGE ERROR RATE")
print("=" * 70)

top_error_rate = error_rate_df.head(15)

print(
    top_error_rate[
        [
            "Species",
            "Support",
            "EfficientNet Error Rate",
            "ResNet50 Error Rate",
            "DenseNet121 Error Rate",
            "Average Error Rate"
        ]
    ].to_string(
        index=False,
        formatters={
            "EfficientNet Error Rate":
                "{:.2f}%".format,

            "ResNet50 Error Rate":
                "{:.2f}%".format,

            "DenseNet121 Error Rate":
                "{:.2f}%".format,

            "Average Error Rate":
                "{:.2f}%".format
        }
    )
)


# ============================================================
# 5. ERROR RATE HEATMAP
# ============================================================

heatmap_data = (
    error_rate_df
    .set_index("Species")[
        [
            "EfficientNet Error Rate",
            "ResNet50 Error Rate",
            "DenseNet121 Error Rate"
        ]
    ]
)


plt.figure(
    figsize=(10, 14)
)

plt.imshow(
    heatmap_data.values,
    aspect="auto",
    cmap="Reds",
    vmin=0,
    vmax=100
)

plt.colorbar(
    label="Error Rate (%)"
)

plt.xticks(
    range(3),
    [
        "EfficientNet-B0",
        "ResNet50",
        "DenseNet121"
    ]
)

plt.yticks(
    range(len(heatmap_data)),
    heatmap_data.index
)

plt.xlabel(
    "Model"
)

plt.ylabel(
    "Species"
)

plt.title(
    "Support-Adjusted Species-wise Error Rate"
)


# ------------------------------------------------------------
# Add values
# ------------------------------------------------------------

for i in range(
    heatmap_data.shape[0]
):

    for j in range(
        heatmap_data.shape[1]
    ):

        value = heatmap_data.iloc[
            i, j
        ]

        plt.text(
            j,
            i,
            f"{value:.0f}%",
            ha="center",
            va="center",
            fontsize=8
        )


plt.tight_layout()

plt.savefig(
    "support_adjusted_error_rate_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 6. TOP DIFFICULT SPECIES BAR CHART
# ============================================================

top_10 = (
    error_rate_df
    .head(10)
    .sort_values(
        "Average Error Rate"
    )
)


plt.figure(
    figsize=(11, 7)
)

plt.barh(
    top_10["Species"],
    top_10["Average Error Rate"]
)

plt.xlabel(
    "Average Error Rate (%)"
)

plt.ylabel(
    "Species"
)

plt.title(
    "Top 10 Most Difficult Species by Average Error Rate"
)

plt.grid(
    axis="x",
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "top_10_difficult_species_error_rate.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 7. LOW-SUPPORT WARNING
# ============================================================

print("\n" + "=" * 70)
print("LOW-SUPPORT CLASSES")
print("=" * 70)

low_support = (
    error_rate_df[
        error_rate_df["Support"] <= 3
    ]
    [
        [
            "Species",
            "Support",
            "EfficientNet Error Rate",
            "ResNet50 Error Rate",
            "DenseNet121 Error Rate",
            "Average Error Rate"
        ]
    ]
)

print(
    low_support.to_string(
        index=False,
        formatters={
            "EfficientNet Error Rate":
                "{:.2f}%".format,

            "ResNet50 Error Rate":
                "{:.2f}%".format,

            "DenseNet121 Error Rate":
                "{:.2f}%".format,

            "Average Error Rate":
                "{:.2f}%".format
        }
    )
)


# ============================================================
# 8. SAVE RESULTS
# ============================================================

error_rate_df.to_csv(
    "support_adjusted_species_error_rates.csv",
    index=False
)

print("\n✓ Saved files:")
print(
    "  support_adjusted_species_error_rates.csv"
)

print(
    "  support_adjusted_error_rate_heatmap.png"
)

print(
    "  top_10_difficult_species_error_rate.png"
)


print("\n" + "=" * 70)
print("✓ SUPPORT-ADJUSTED ERROR RATE ANALYSIS COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 77: CROSS-MODEL CONFUSION PAIR ANALYSIS
#          EfficientNet-B0 vs ResNet50 vs DenseNet121
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("=" * 70)
print("CROSS-MODEL CONFUSION PAIR ANALYSIS")
print("=" * 70)


# ============================================================
# HELPER FUNCTION
# ============================================================

def get_confusion_pairs(
    true_labels,
    predictions,
    model_name
):

    confusion_pairs = []

    for true_idx in range(
        len(class_names)
    ):

        for pred_idx in range(
            len(class_names)
        ):

            if (
                true_idx != pred_idx
                and
                np.sum(
                    (true_labels == true_idx)
                    &
                    (predictions == pred_idx)
                ) > 0
            ):

                count = np.sum(
                    (true_labels == true_idx)
                    &
                    (predictions == pred_idx)
                )

                confusion_pairs.append({

                    "Model":
                        model_name,

                    "True Species":
                        class_names[true_idx],

                    "Predicted Species":
                        class_names[pred_idx],

                    "Count":
                        int(count)
                })

    return pd.DataFrame(
        confusion_pairs
    )


# ============================================================
# 1. GENERATE CONFUSION PAIRS
# ============================================================

efficientnet_pairs = get_confusion_pairs(
    true_labels,
    efficientnet_predictions,
    "EfficientNet-B0"
)

resnet_pairs = get_confusion_pairs(
    true_labels,
    resnet50_predictions,
    "ResNet50"
)

densenet_pairs = get_confusion_pairs(
    true_labels,
    densenet121_predictions,
    "DenseNet121"
)


# ============================================================
# 2. COMBINE ALL MODELS
# ============================================================

all_confusion_pairs = pd.concat(
    [
        efficientnet_pairs,
        resnet_pairs,
        densenet_pairs
    ],
    ignore_index=True
)


all_confusion_pairs = (
    all_confusion_pairs
    .sort_values(
        ["Model", "Count"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)


# ============================================================
# 3. DISPLAY ALL CONFUSION PAIRS
# ============================================================

print("\n" + "=" * 70)
print("ALL MISCLASSIFICATION PAIRS")
print("=" * 70)

print(
    all_confusion_pairs.to_string(
        index=False
    )
)


# ============================================================
# 4. TOP CONFUSION PAIRS
# ============================================================

print("\n" + "=" * 70)
print("TOP CONFUSION PAIRS ACROSS ALL MODELS")
print("=" * 70)

top_confusions = (
    all_confusion_pairs
    .sort_values(
        "Count",
        ascending=False
    )
    .head(20)
)

print(
    top_confusions.to_string(
        index=False
    )
)


# ============================================================
# 5. FIND REPEATED CONFUSION PAIRS
# ============================================================

pair_frequency = (
    all_confusion_pairs
    .groupby(
        [
            "True Species",
            "Predicted Species"
        ]
    )
    .agg(
        Models_Affected=("Model", "count"),
        Total_Errors=("Count", "sum")
    )
    .reset_index()
)


pair_frequency = (
    pair_frequency
    .sort_values(
        [
            "Models_Affected",
            "Total_Errors"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)


print("\n" + "=" * 70)
print("REPEATED CONFUSION PAIRS")
print("=" * 70)

print(
    pair_frequency.head(20).to_string(
        index=False
    )
)


# ============================================================
# 6. CONFUSION PAIRS SHARED BY ALL 3 MODELS
# ============================================================

shared_by_all = pair_frequency[
    pair_frequency["Models_Affected"] == 3
]


print("\n" + "=" * 70)
print("CONFUSION PAIRS SHARED BY ALL 3 MODELS")
print("=" * 70)

if len(shared_by_all) > 0:

    print(
        shared_by_all.to_string(
            index=False
        )
    )

else:

    print(
        "No identical confusion pair was shared "
        "by all three models."
    )


# ============================================================
# 7. CONFUSION PAIRS SHARED BY AT LEAST 2 MODELS
# ============================================================

shared_by_two = pair_frequency[
    pair_frequency["Models_Affected"] >= 2
]


print("\n" + "=" * 70)
print("CONFUSION PAIRS SHARED BY AT LEAST 2 MODELS")
print("=" * 70)

if len(shared_by_two) > 0:

    print(
        shared_by_two.to_string(
            index=False
        )
    )

else:

    print(
        "No confusion pair was shared "
        "by at least two models."
    )


# ============================================================
# 8. VISUALIZE TOP CONFUSION PAIRS
# ============================================================

plot_df = (
    pair_frequency
    .head(15)
    .copy()
)

plot_df["Pair"] = (
    plot_df["True Species"]
    + " → "
    + plot_df["Predicted Species"]
)

plot_df = (
    plot_df
    .sort_values(
        "Total_Errors"
    )
)


plt.figure(
    figsize=(11, 8)
)

plt.barh(
    plot_df["Pair"],
    plot_df["Total_Errors"]
)

plt.xlabel(
    "Total Misclassifications"
)

plt.ylabel(
    "True → Predicted"
)

plt.title(
    "Top Cross-Model Confusion Pairs"
)

plt.grid(
    axis="x",
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "top_cross_model_confusion_pairs.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 9. SAVE RESULTS
# ============================================================

all_confusion_pairs.to_csv(
    "all_model_confusion_pairs.csv",
    index=False
)

pair_frequency.to_csv(
    "cross_model_confusion_pair_frequency.csv",
    index=False
)

print("\n✓ Saved files:")
print(
    "  all_model_confusion_pairs.csv"
)

print(
    "  cross_model_confusion_pair_frequency.csv"
)

print(
    "  top_cross_model_confusion_pairs.png"
)


print("\n" + "=" * 70)
print(
    "✓ CROSS-MODEL CONFUSION PAIR ANALYSIS COMPLETE"
)
print("=" * 70)

In [ ]:
# ============================================================
# CELL 78: VISUAL ANALYSIS OF SHARED CONFUSION PAIRS
#          Neem → Tulsi
#          Bakul → Mango
#          Lemon → Guava
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
import torch

print("=" * 70)
print("VISUAL ANALYSIS OF SHARED CONFUSION PAIRS")
print("=" * 70)


# ============================================================
# SHARED CONFUSION PAIRS
# ============================================================

shared_pairs = [
    ("Neem", "Tulsi"),
    ("Bakul", "Mango"),
    ("Lemon", "Guava")
]


# ============================================================
# COLLECT TEST IMAGES
# ============================================================

test_images = []

for i in range(len(test_dataset)):

    image, label = test_dataset[i]

    if torch.is_tensor(image):

        image_np = image.detach().cpu().clone()

        # ----------------------------------------------------
        # Handle normalized tensors
        # ----------------------------------------------------

        if image_np.min() < 0:

            mean = torch.tensor(
                [0.485, 0.456, 0.406]
            ).view(3, 1, 1)

            std = torch.tensor(
                [0.229, 0.224, 0.225]
            ).view(3, 1, 1)

            image_np = (
                image_np * std
                + mean
            )

        image_np = (
            image_np
            .permute(1, 2, 0)
            .numpy()
        )

        image_np = np.clip(
            image_np,
            0,
            1
        )

    else:

        image_np = np.array(
            image
        )

    test_images.append(
        image_np
    )


# ============================================================
# VERIFY LABEL ORDER
# ============================================================

test_true_labels = np.array(
    true_labels
)


# ============================================================
# VISUALIZE EACH SHARED CONFUSION PAIR
# ============================================================

for true_species, predicted_species in shared_pairs:

    true_idx = class_names.index(
        true_species
    )

    predicted_idx = class_names.index(
        predicted_species
    )

    # --------------------------------------------------------
    # Find test images where each model made this confusion
    # --------------------------------------------------------

    eff_indices = np.where(
        (test_true_labels == true_idx)
        &
        (efficientnet_predictions == predicted_idx)
    )[0]

    res_indices = np.where(
        (test_true_labels == true_idx)
        &
        (resnet50_predictions == predicted_idx)
    )[0]

    dense_indices = np.where(
        (test_true_labels == true_idx)
        &
        (densenet121_predictions == predicted_idx)
    )[0]

    print("\n" + "=" * 70)

    print(
        f"{true_species} → {predicted_species}"
    )

    print("=" * 70)

    print(
        f"EfficientNet-B0 : "
        f"{len(eff_indices)} image(s)"
    )

    print(
        f"ResNet50        : "
        f"{len(res_indices)} image(s)"
    )

    print(
        f"DenseNet121     : "
        f"{len(dense_indices)} image(s)"
    )


    # --------------------------------------------------------
    # Combine unique image indices
    # --------------------------------------------------------

    all_indices = sorted(
        set(
            eff_indices.tolist()
            + res_indices.tolist()
            + dense_indices.tolist()
        )
    )

    if len(all_indices) == 0:

        print(
            "No matching images found."
        )

        continue


    # --------------------------------------------------------
    # Display images
    # --------------------------------------------------------

    fig, axes = plt.subplots(
        1,
        len(all_indices),
        figsize=(
            max(4 * len(all_indices), 5),
            5
        )
    )

    if len(all_indices) == 1:

        axes = [axes]


    for ax, idx in zip(
        axes,
        all_indices
    ):

        ax.imshow(
            test_images[idx]
        )

        true_name = class_names[
            true_labels[idx]
        ]

        eff_name = class_names[
            efficientnet_predictions[idx]
        ]

        res_name = class_names[
            resnet50_predictions[idx]
        ]

        dense_name = class_names[
            densenet121_predictions[idx]
        ]

        ax.set_title(
            f"True: {true_name}\n"
            f"Eff: {eff_name}\n"
            f"Res: {res_name}\n"
            f"Dense: {dense_name}",
            fontsize=9
        )

        ax.axis(
            "off"
        )


    plt.suptitle(
        f"Shared Confusion: "
        f"{true_species} → {predicted_species}",
        fontsize=14
    )

    plt.tight_layout()

    filename = (
        f"confusion_"
        f"{true_species.replace(' ', '_')}_"
        f"to_"
        f"{predicted_species.replace(' ', '_')}.png"
    )

    plt.savefig(
        filename,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()


# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("✓ SHARED CONFUSION VISUAL ANALYSIS COMPLETE")
print("=" * 70)

print("\nGenerated visualizations:")

for true_species, predicted_species in shared_pairs:

    filename = (
        f"confusion_"
        f"{true_species.replace(' ', '_')}_"
        f"to_"
        f"{predicted_species.replace(' ', '_')}.png"
    )

    print(
        f"  ✓ {filename}"
    )

In [ ]:
# ============================================================
# CELL 79: ROBUST GRAD-CAM EXPLAINABILITY
#          EfficientNet-B0 vs ResNet50 vs DenseNet121
# ============================================================

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

print("=" * 70)
print("ROBUST GRAD-CAM VISUAL EXPLAINABILITY")
print("=" * 70)


# ============================================================
# 1. GRAD-CAM FUNCTION
# ============================================================

def generate_gradcam(
    model,
    image_tensor,
    target_class,
    target_layer
):

    model.eval()

    activation = None
    gradients = None


    # --------------------------------------------------------
    # Forward hook
    # --------------------------------------------------------

    def forward_hook(
        module,
        inputs,
        output
    ):

        nonlocal activation

        activation = output

        # Capture gradient directly from activation
        if activation.requires_grad:

            activation.register_hook(
                lambda grad: set_gradient(
                    grad
                )
            )


    # --------------------------------------------------------
    # Gradient storage
    # --------------------------------------------------------

    def set_gradient(
        grad
    ):

        nonlocal gradients

        gradients = grad


    # --------------------------------------------------------
    # Register hook
    # --------------------------------------------------------

    hook = target_layer.register_forward_hook(
        forward_hook
    )


    try:

        # ----------------------------------------------------
        # IMPORTANT:
        # Input MUST require gradient because
        # model backbone is frozen.
        # ----------------------------------------------------

        image_input = (
            image_tensor
            .clone()
            .detach()
            .unsqueeze(0)
            .to(device)
        )

        image_input.requires_grad_(True)


        # ----------------------------------------------------
        # Forward
        # ----------------------------------------------------

        model.zero_grad(
            set_to_none=True
        )

        output = model(
            image_input
        )


        # ----------------------------------------------------
        # Target class score
        # ----------------------------------------------------

        score = output[
            0,
            target_class
        ]


        # ----------------------------------------------------
        # Backward
        # ----------------------------------------------------

        score.backward()


        # ----------------------------------------------------
        # Check gradient
        # ----------------------------------------------------

        if gradients is None:

            raise RuntimeError(
                "Activation gradient was not captured."
            )


        # ----------------------------------------------------
        # Grad-CAM weights
        # ----------------------------------------------------

        weights = gradients.mean(
            dim=(2, 3),
            keepdim=True
        )


        # ----------------------------------------------------
        # CAM
        # ----------------------------------------------------

        cam = (
            weights * activation
        ).sum(
            dim=1,
            keepdim=True
        )


        cam = F.relu(
            cam
        )


        # ----------------------------------------------------
        # Resize CAM
        # ----------------------------------------------------

        cam = F.interpolate(
            cam,
            size=(
                image_input.shape[2],
                image_input.shape[3]
            ),
            mode="bilinear",
            align_corners=False
        )


        cam = (
            cam
            .squeeze()
            .detach()
            .cpu()
            .numpy()
        )


        # ----------------------------------------------------
        # Normalize
        # ----------------------------------------------------

        cam -= cam.min()

        if cam.max() > 0:

            cam /= cam.max()


        return cam


    finally:

        hook.remove()

        model.zero_grad(
            set_to_none=True
        )


# ============================================================
# 2. IMAGE TO DISPLAY FORMAT
# ============================================================

def tensor_to_image(
    tensor
):

    image = (
        tensor
        .detach()
        .cpu()
        .clone()
    )

    mean = torch.tensor(
        [0.485, 0.456, 0.406]
    ).view(
        3, 1, 1
    )

    std = torch.tensor(
        [0.229, 0.224, 0.225]
    ).view(
        3, 1, 1
    )

    image = (
        image * std
        + mean
    )

    image = image.clamp(
        0,
        1
    )

    image = (
        image
        .permute(1, 2, 0)
        .numpy()
    )

    return image


# ============================================================
# 3. SHARED CONFUSION PAIRS
# ============================================================

shared_pairs = [

    ("Neem", "Tulsi"),

    ("Bakul", "Mango"),

    ("Lemon", "Guava")
]


# ============================================================
# 4. FIND REPRESENTATIVE IMAGE
# ============================================================

def find_confusion_index(
    true_species,
    predicted_species
):

    true_idx = class_names.index(
        true_species
    )

    predicted_idx = class_names.index(
        predicted_species
    )


    # --------------------------------------------------------
    # Prefer image where all three models made same error
    # --------------------------------------------------------

    for i in range(
        len(true_labels)
    ):

        if (
            true_labels[i] == true_idx
            and
            efficientnet_predictions[i] == predicted_idx
            and
            resnet50_predictions[i] == predicted_idx
            and
            densenet121_predictions[i] == predicted_idx
        ):

            return i


    # --------------------------------------------------------
    # Otherwise image where at least two models made error
    # --------------------------------------------------------

    for i in range(
        len(true_labels)
    ):

        count = 0

        if (
            true_labels[i] == true_idx
            and
            efficientnet_predictions[i] == predicted_idx
        ):

            count += 1

        if (
            true_labels[i] == true_idx
            and
            resnet50_predictions[i] == predicted_idx
        ):

            count += 1

        if (
            true_labels[i] == true_idx
            and
            densenet121_predictions[i] == predicted_idx
        ):

            count += 1


        if count >= 2:

            return i


    return None


# ============================================================
# 5. MODEL TARGET LAYERS
# ============================================================

model_configs = {

    "EfficientNet-B0": (
        efficientnet_model,
        efficientnet_model.features[-1]
    ),

    "ResNet50": (
        resnet50_model,
        resnet50_model.layer4[-1]
    ),

    "DenseNet121": (
        densenet121_model,
        densenet121_model.features.denseblock4
    )
}


# ============================================================
# 6. PROCESS EACH CONFUSION PAIR
# ============================================================

for true_species, predicted_species in shared_pairs:

    print("\n" + "=" * 70)

    print(
        f"Analyzing: "
        f"{true_species} → {predicted_species}"
    )

    print("=" * 70)


    idx = find_confusion_index(
        true_species,
        predicted_species
    )


    if idx is None:

        print(
            "No suitable representative image found."
        )

        continue


    # --------------------------------------------------------
    # Get image
    # --------------------------------------------------------

    image_tensor, true_label = (
        test_dataset[idx]
    )


    original_image = tensor_to_image(
        image_tensor
    )


    # --------------------------------------------------------
    # Create figure
    # --------------------------------------------------------

    fig, axes = plt.subplots(
        2,
        3,
        figsize=(15, 9)
    )


    # --------------------------------------------------------
    # Process each model
    # --------------------------------------------------------

    for col, (
        model_name,
        config
    ) in enumerate(
        model_configs.items()
    ):

        model, target_layer = config


        try:

            model.eval()


            # ------------------------------------------------
            # Get prediction
            # ------------------------------------------------

            with torch.no_grad():

                output = model(
                    image_tensor
                    .unsqueeze(0)
                    .to(device)
                )

                predicted_class = (
                    torch.argmax(
                        output,
                        dim=1
                    ).item()
                )


            # ------------------------------------------------
            # Generate Grad-CAM
            # ------------------------------------------------

            cam = generate_gradcam(
                model,
                image_tensor,
                predicted_class,
                target_layer
            )


            # ------------------------------------------------
            # Original
            # ------------------------------------------------

            axes[0, col].imshow(
                original_image
            )

            axes[0, col].set_title(
                f"{model_name}\n"
                f"True: {true_species}\n"
                f"Pred: "
                f"{class_names[predicted_class]}",
                fontsize=10
            )

            axes[0, col].axis(
                "off"
            )


            # ------------------------------------------------
            # Grad-CAM
            # ------------------------------------------------

            axes[1, col].imshow(
                original_image
            )

            axes[1, col].imshow(
                cam,
                cmap="jet",
                alpha=0.45
            )

            axes[1, col].set_title(
                "Grad-CAM Attention",
                fontsize=10
            )

            axes[1, col].axis(
                "off"
            )


            print(
                f"✓ {model_name} Grad-CAM generated"
            )


        except Exception as e:

            # ------------------------------------------------
            # Do NOT break the whole figure
            # ------------------------------------------------

            axes[0, col].imshow(
                original_image
            )

            axes[0, col].set_title(
                f"{model_name}\n"
                f"Grad-CAM unavailable",
                fontsize=10
            )

            axes[0, col].axis(
                "off"
            )


            axes[1, col].text(
                0.5,
                0.5,
                "Grad-CAM Error",
                ha="center",
                va="center",
                fontsize=12
            )

            axes[1, col].axis(
                "off"
            )


            print(
                f"✗ {model_name} failed:"
            )

            print(
                f"  {type(e).__name__}: {e}"
            )


    # --------------------------------------------------------
    # Figure title
    # --------------------------------------------------------

    plt.suptitle(
        f"Grad-CAM Analysis: "
        f"{true_species} → {predicted_species}",
        fontsize=15
    )


    plt.tight_layout()


    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    filename = (
        f"gradcam_"
        f"{true_species.replace(' ', '_')}_"
        f"to_"
        f"{predicted_species.replace(' ', '_')}.png"
    )


    plt.savefig(
        filename,
        dpi=300,
        bbox_inches="tight"
    )


    plt.show()


# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 70)
print("✓ ROBUST GRAD-CAM ANALYSIS COMPLETE")
print("=" * 70)

print("\nGenerated files:")

for true_species, predicted_species in shared_pairs:

    filename = (
        f"gradcam_"
        f"{true_species.replace(' ', '_')}_"
        f"to_"
        f"{predicted_species.replace(' ', '_')}.png"
    )

    print(
        f"  ✓ {filename}"
    )

In [ ]:
# ============================================================
# CELL 80: FRESH DENSENET121 RELOAD
#          Remove Previous Grad-CAM Hooks
# ============================================================

import torch
import torch.nn as nn
from torchvision import models

print("=" * 70)
print("FRESH DENSENET121 RELOAD FOR GRAD-CAM")
print("=" * 70)


# ============================================================
# 1. CREATE COMPLETELY FRESH DENSENET121
# ============================================================

fresh_densenet121 = models.densenet121(
    weights=None
)


# ============================================================
# 2. REPLACE CLASSIFIER
# ============================================================

num_features = (
    fresh_densenet121.classifier.in_features
)

fresh_densenet121.classifier = nn.Linear(
    num_features,
    len(class_names)
)


# ============================================================
# 3. LOAD TRAINED WEIGHTS
# ============================================================

checkpoint_path = "best_densenet121.pth"

checkpoint = torch.load(
    checkpoint_path,
    map_location=device
)


# ------------------------------------------------------------
# Handle different checkpoint formats
# ------------------------------------------------------------

if isinstance(
    checkpoint,
    dict
):

    if "model_state_dict" in checkpoint:

        state_dict = checkpoint[
            "model_state_dict"
        ]

    elif "state_dict" in checkpoint:

        state_dict = checkpoint[
            "state_dict"
        ]

    else:

        state_dict = checkpoint

else:

    state_dict = checkpoint


# ------------------------------------------------------------
# Remove possible "module." prefix
# ------------------------------------------------------------

clean_state_dict = {}

for key, value in state_dict.items():

    new_key = key

    if new_key.startswith(
        "module."
    ):

        new_key = new_key[
            len("module.") :
        ]

    clean_state_dict[
        new_key
    ] = value


# ============================================================
# 4. LOAD WEIGHTS
# ============================================================

missing_keys, unexpected_keys = (
    fresh_densenet121.load_state_dict(
        clean_state_dict,
        strict=False
    )
)


fresh_densenet121 = (
    fresh_densenet121
    .to(device)
)


fresh_densenet121.eval()


# ============================================================
# 5. CHECK
# ============================================================

print(
    "✓ Fresh DenseNet121 created"
)

print(
    f"✓ Classes: {len(class_names)}"
)

print(
    f"✓ Device: {device}"
)

print(
    f"✓ Missing keys: {len(missing_keys)}"
)

print(
    f"✓ Unexpected keys: {len(unexpected_keys)}"
)


# ============================================================
# 6. IMPORTANT: REPLACE OLD MODEL
# ============================================================

densenet121_model = (
    fresh_densenet121
)


# ============================================================
# 7. FRESH TARGET LAYER
# ============================================================

densenet121_gradcam_layer = (
    densenet121_model
    .features
    .denseblock4
)


print(
    "✓ DenseNet121 target layer:"
)

print(
    "  features.denseblock4"
)


# ============================================================
# 8. QUICK FORWARD TEST
# ============================================================

with torch.no_grad():

    test_tensor, test_label = (
        test_dataset[0]
    )

    test_output = (
        densenet121_model(
            test_tensor
            .unsqueeze(0)
            .to(device)
        )
    )


print(
    f"✓ Forward pass successful"
)

print(
    f"✓ Output shape: "
    f"{tuple(test_output.shape)}"
)


print("\n" + "=" * 70)
print(
    "✓ FRESH DENSENET121 READY FOR GRAD-CAM"
)
print("=" * 70)

In [ ]:
# ============================================================
# CELL 81: DENSENET121 GRAD-CAM ONLY
#          Fresh Model + Shared Confusion Pairs
# ============================================================

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt


print("=" * 70)
print("DENSENET121 GRAD-CAM ANALYSIS")
print("=" * 70)


# ============================================================
# 1. GRAD-CAM FUNCTION
# ============================================================

def generate_densenet_gradcam(
    model,
    image_tensor,
    target_class,
    target_layer
):

    model.eval()

    activation = None
    gradients = None


    # --------------------------------------------------------
    # Forward hook
    # --------------------------------------------------------

    def forward_hook(
        module,
        inputs,
        output
    ):

        nonlocal activation

        activation = output

        if activation.requires_grad:

            activation.register_hook(
                lambda grad: capture_gradient(
                    grad
                )
            )


    # --------------------------------------------------------
    # Capture activation gradient
    # --------------------------------------------------------

    def capture_gradient(
        grad
    ):

        nonlocal gradients

        gradients = grad


    hook = target_layer.register_forward_hook(
        forward_hook
    )


    try:

        # ----------------------------------------------------
        # Input requires gradient
        # ----------------------------------------------------

        image_input = (
            image_tensor
            .clone()
            .detach()
            .unsqueeze(0)
            .to(device)
        )

        image_input.requires_grad_(True)


        # ----------------------------------------------------
        # Forward
        # ----------------------------------------------------

        model.zero_grad(
            set_to_none=True
        )

        output = model(
            image_input
        )


        # ----------------------------------------------------
        # Target score
        # ----------------------------------------------------

        score = output[
            0,
            target_class
        ]


        # ----------------------------------------------------
        # Backward
        # ----------------------------------------------------

        score.backward()


        # ----------------------------------------------------
        # Verify gradient
        # ----------------------------------------------------

        if gradients is None:

            raise RuntimeError(
                "DenseNet activation gradient "
                "was not captured."
            )


        # ----------------------------------------------------
        # Grad-CAM weights
        # ----------------------------------------------------

        weights = gradients.mean(
            dim=(2, 3),
            keepdim=True
        )


        # ----------------------------------------------------
        # CAM
        # ----------------------------------------------------

        cam = (
            weights * activation
        ).sum(
            dim=1,
            keepdim=True
        )


        cam = F.relu(
            cam
        )


        # ----------------------------------------------------
        # Resize
        # ----------------------------------------------------

        cam = F.interpolate(
            cam,
            size=(
                image_input.shape[2],
                image_input.shape[3]
            ),
            mode="bilinear",
            align_corners=False
        )


        cam = (
            cam
            .squeeze()
            .detach()
            .cpu()
            .numpy()
        )


        # ----------------------------------------------------
        # Normalize
        # ----------------------------------------------------

        cam -= cam.min()

        if cam.max() > 0:

            cam /= cam.max()


        return cam


    finally:

        hook.remove()

        model.zero_grad(
            set_to_none=True
        )


# ============================================================
# 2. IMAGE DISPLAY FUNCTION
# ============================================================

def denormalize_image(
    tensor
):

    image = (
        tensor
        .detach()
        .cpu()
        .clone()
    )


    mean = torch.tensor(
        [0.485, 0.456, 0.406]
    ).view(
        3, 1, 1
    )


    std = torch.tensor(
        [0.229, 0.224, 0.225]
    ).view(
        3, 1, 1
    )


    image = (
        image * std
        + mean
    )


    image = image.clamp(
        0,
        1
    )


    image = (
        image
        .permute(1, 2, 0)
        .numpy()
    )


    return image


# ============================================================
# 3. CONFUSION PAIRS
# ============================================================

shared_pairs = [

    ("Neem", "Tulsi"),

    ("Bakul", "Mango"),

    ("Lemon", "Guava")
]


# ============================================================
# 4. FIND REPRESENTATIVE IMAGE
# ============================================================

def find_densenet_confusion_index(
    true_species,
    predicted_species
):

    true_idx = class_names.index(
        true_species
    )

    predicted_idx = class_names.index(
        predicted_species
    )


    # --------------------------------------------------------
    # Prefer image where DenseNet itself made this error
    # --------------------------------------------------------

    for i in range(
        len(true_labels)
    ):

        if (
            true_labels[i] == true_idx
            and
            densenet121_predictions[i] == predicted_idx
        ):

            return i


    return None


# ============================================================
# 5. GENERATE DENSENET121 GRAD-CAM
# ============================================================

for true_species, predicted_species in shared_pairs:

    print("\n" + "=" * 70)

    print(
        f"Analyzing: "
        f"{true_species} → {predicted_species}"
    )

    print("=" * 70)


    idx = find_densenet_confusion_index(
        true_species,
        predicted_species
    )


    if idx is None:

        print(
            "No DenseNet121 test image found "
            "for this confusion pair."
        )

        continue


    # --------------------------------------------------------
    # Load image
    # --------------------------------------------------------

    image_tensor, true_label = (
        test_dataset[idx]
    )


    original_image = denormalize_image(
        image_tensor
    )


    # --------------------------------------------------------
    # DenseNet prediction
    # --------------------------------------------------------

    densenet121_model.eval()

    densenet121_model.zero_grad(
        set_to_none=True
    )


    output = densenet121_model(
        image_tensor
        .unsqueeze(0)
        .to(device)
    )


    predicted_class = torch.argmax(
        output,
        dim=1
    ).item()


    # --------------------------------------------------------
    # Generate Grad-CAM
    # --------------------------------------------------------

    cam = generate_densenet_gradcam(
        densenet121_model,
        image_tensor,
        predicted_class,
        densenet121_gradcam_layer
    )


    # ========================================================
    # VISUALIZATION
    # ========================================================

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(11, 5)
    )


    # --------------------------------------------------------
    # Original
    # --------------------------------------------------------

    axes[0].imshow(
        original_image
    )

    axes[0].set_title(
        f"DenseNet121\n"
        f"True: {true_species}\n"
        f"Predicted: "
        f"{class_names[predicted_class]}",
        fontsize=11
    )

    axes[0].axis(
        "off"
    )


    # --------------------------------------------------------
    # Grad-CAM
    # --------------------------------------------------------

    axes[1].imshow(
        original_image
    )

    axes[1].imshow(
        cam,
        cmap="jet",
        alpha=0.45
    )

    axes[1].set_title(
        "DenseNet121 Grad-CAM",
        fontsize=11
    )

    axes[1].axis(
        "off"
    )


    plt.suptitle(
        f"Grad-CAM: "
        f"{true_species} → {predicted_species}",
        fontsize=14
    )


    plt.tight_layout()


    filename = (
        f"densenet_gradcam_"
        f"{true_species.replace(' ', '_')}_"
        f"to_"
        f"{predicted_species.replace(' ', '_')}.png"
    )


    plt.savefig(
        filename,
        dpi=300,
        bbox_inches="tight"
    )


    plt.show()


    print(
        f"✓ DenseNet121 Grad-CAM generated"
    )

    print(
        f"  True class      : {true_species}"
    )

    print(
        f"  Predicted class : "
        f"{class_names[predicted_class]}"
    )


# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 70)
print("✓ DENSENET121 GRAD-CAM COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# RECOVERY CELL: ViT + LSTM EVALUATION ENVIRONMENT
# ============================================================

import os
import json
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torchvision import models
from torch.utils.data import DataLoader

print("=" * 80)
print("RECOVERING ViT + LSTM ENVIRONMENT")
print("=" * 80)

# ------------------------------------------------------------
# 1. Device
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"✓ Device: {device}")

# ------------------------------------------------------------
# 2. Class Names
# ------------------------------------------------------------

class_names = [
    "Almond",
    "Aloe vera",
    "Amloki",
    "Apple",
    "Arjun",
    "Bael",
    "Bakul",
    "Banana",
    "Black cumin",
    "Boroi",
    "Chili",
    "Chrysanthemum",
    "Ginger",
    "Green chili",
    "Guava",
    "Jackfruit",
    "Kochu",
    "Lemon",
    "Litchi",
    "Mango",
    "Marigold",
    "Moringa",
    "Neem",
    "Papaya",
    "Patorkusi",
    "Peepal",
    "Pomegranate",
    "Potato",
    "Puishak",
    "Sapodilla",
    "Tulsi",
    "Turmeric",
    "Vasaka"
]

NUM_CLASSES = len(class_names)

print(f"✓ Classes loaded: {NUM_CLASSES}")

# ------------------------------------------------------------
# 3. Restore ViTLSTM Class
# ------------------------------------------------------------

class ViTLSTM(nn.Module):

    def __init__(
        self,
        vit,
        num_classes,
        hidden_size=256,
        num_layers=2,
        dropout=0.3
    ):
        super().__init__()

        self.vit = vit

        # Original ViT classification head is not used
        self.vit.heads = nn.Identity()

        # ViT-B/16 embedding dimension = 768
        self.lstm = nn.LSTM(
            input_size=768,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(
                hidden_size,
                num_classes
            )
        )

    def forward(self, x):

        # Patch embedding
        x = self.vit._process_input(x)

        n = x.shape[0]

        # CLS token
        batch_class_token = (
            self.vit.class_token.expand(
                n, -1, -1
            )
        )

        x = torch.cat(
            [batch_class_token, x],
            dim=1
        )

        # Positional embedding
        x = x + self.vit.encoder.pos_embedding

        # Transformer encoder
        x = self.vit.encoder.dropout(x)

        x = self.vit.encoder.layers(x)

        x = self.vit.encoder.ln(x)

        # Remove CLS token
        x = x[:, 1:, :]

        # LSTM
        x, _ = self.lstm(x)

        # Last sequence output
        x = x[:, -1, :]

        # Classification
        x = self.classifier(x)

        return x


print("✓ ViTLSTM class restored")

# ------------------------------------------------------------
# 4. Check Saved Model
# ------------------------------------------------------------

best_vit_lstm_path = "best_vit_lstm.pth"

if os.path.exists(best_vit_lstm_path):

    print(
        f"✓ Found: {best_vit_lstm_path}"
    )

else:

    print(
        "✗ best_vit_lstm.pth not found!"
    )

# ------------------------------------------------------------
# 5. Check Existing Dataset Objects
# ------------------------------------------------------------

print("\nChecking existing dataset objects...")

required_objects = [
    "test_dataset",
    "test_loader"
]

for obj_name in required_objects:

    if obj_name in globals():

        print(
            f"✓ {obj_name} already exists"
        )

    else:

        print(
            f"✗ {obj_name} is missing"
        )

print("\n" + "=" * 80)
print("✓ RECOVERY STEP COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# CELL 82: VISION TRANSFORMER + LSTM
#          MODEL SETUP
# ============================================================

import torch
import torch.nn as nn
from torchvision import models


print("=" * 70)
print("VISION TRANSFORMER + LSTM MODEL")
print("=" * 70)


# ============================================================
# 1. DEVICE
# ============================================================

print(f"Device: {device}")
print(f"Number of classes: {len(class_names)}")


# ============================================================
# 2. LOAD PRETRAINED VISION TRANSFORMER
# ============================================================

vit_weights = (
    models.ViT_B_16_Weights.DEFAULT
)

vit_backbone = models.vit_b_16(
    weights=vit_weights
)


print("✓ ImageNet pretrained ViT-B/16 loaded")


# ============================================================
# 3. FREEZE ViT BACKBONE
# ============================================================

for param in vit_backbone.parameters():

    param.requires_grad = False


print("✓ ViT backbone frozen")


# ============================================================
# 4. REMOVE ORIGINAL ViT CLASSIFIER
# ============================================================

vit_backbone.heads = nn.Identity()


# ============================================================
# 5. ViT + LSTM MODEL
# ============================================================

class ViTLSTM(nn.Module):

    def __init__(
        self,
        vit,
        hidden_size=256,
        num_layers=2,
        num_classes=33,
        dropout=0.3
    ):

        super().__init__()


        # ----------------------------------------------------
        # Vision Transformer
        # ----------------------------------------------------

        self.vit = vit


        # ----------------------------------------------------
        # ViT embedding dimension
        # ViT-B/16 = 768
        # ----------------------------------------------------

        self.feature_dim = 768


        # ----------------------------------------------------
        # LSTM
        # ----------------------------------------------------

        self.lstm = nn.LSTM(
            input_size=self.feature_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )


        # ----------------------------------------------------
        # Classifier
        # ----------------------------------------------------

        self.classifier = nn.Sequential(

            nn.Dropout(dropout),

            nn.Linear(
                hidden_size,
                num_classes
            )
        )


    def forward(
        self,
        x
    ):

        # ----------------------------------------------------
        # Get ViT patch/token representation
        # ----------------------------------------------------

        x = self.vit._process_input(x)


        # ----------------------------------------------------
        # Add CLS token
        # ----------------------------------------------------

        n = x.shape[0]

        batch_class_token = (
            self.vit.class_token
            .expand(n, -1, -1)
        )

        x = torch.cat(
            [
                batch_class_token,
                x
            ],
            dim=1
        )


        # ----------------------------------------------------
        # Positional embedding
        # ----------------------------------------------------

        x = (
            x
            + self.vit.encoder.pos_embedding
        )


        # ----------------------------------------------------
        # Transformer encoder
        # ----------------------------------------------------

        x = self.vit.encoder.dropout(
            x
        )


        for layer in self.vit.encoder.layers:

            x = layer(
                x
            )


        x = self.vit.encoder.ln(
            x
        )


        # ----------------------------------------------------
        # Remove CLS token
        #
        # Remaining tokens = patch sequence
        # ----------------------------------------------------

        x = x[:, 1:, :]


        # ----------------------------------------------------
        # LSTM
        # ----------------------------------------------------

        lstm_out, _ = self.lstm(
            x
        )


        # ----------------------------------------------------
        # Final sequence representation
        # ----------------------------------------------------

        final_feature = (
            lstm_out[:, -1, :]
        )


        # ----------------------------------------------------
        # Classification
        # ----------------------------------------------------

        output = self.classifier(
            final_feature
        )


        return output


# ============================================================
# 6. CREATE MODEL
# ============================================================

vit_lstm_model = ViTLSTM(
    vit=vit_backbone,
    hidden_size=256,
    num_layers=2,
    num_classes=len(class_names),
    dropout=0.3
)


vit_lstm_model = (
    vit_lstm_model
    .to(device)
)


# ============================================================
# 7. PARAMETER SUMMARY
# ============================================================

total_params = sum(
    p.numel()
    for p in vit_lstm_model.parameters()
)


trainable_params = sum(
    p.numel()
    for p in vit_lstm_model.parameters()
    if p.requires_grad
)


print("\n" + "=" * 70)
print("MODEL SUMMARY")
print("=" * 70)

print(
    f"Total parameters     : "
    f"{total_params:,}"
)

print(
    f"Trainable parameters : "
    f"{trainable_params:,}"
)

print(
    f"Frozen parameters    : "
    f"{total_params - trainable_params:,}"
)


print("\nArchitecture:")
print(
    "ViT-B/16 → Patch Tokens → "
    "LSTM → Fully Connected → 33 Classes"
)


print("\n" + "=" * 70)
print("✓ VISION TRANSFORMER + LSTM SETUP COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 83: VIT + LSTM STRATIFIED 4-FOLD CROSS-VALIDATION
#          WITH EARLY STOPPING
# ============================================================

import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import StratifiedKFold


print("=" * 70)
print("VIT + LSTM STRATIFIED 4-FOLD CROSS-VALIDATION")
print("=" * 70)


# ============================================================
# 1. CONFIGURATION
# ============================================================

NUM_FOLDS = 4

MAX_EPOCHS = 150

EARLY_STOPPING_PATIENCE = 10

INITIAL_LR = 0.001

WEIGHT_DECAY = 0.0001

BATCH_SIZE = 32


print("\nConfiguration:")
print(f"Folds                : {NUM_FOLDS}")
print(f"Maximum Epochs       : {MAX_EPOCHS}")
print(
    f"Early Stopping       : "
    f"{EARLY_STOPPING_PATIENCE}"
)
print(f"Learning Rate        : {INITIAL_LR}")
print(f"Weight Decay         : {WEIGHT_DECAY}")
print(f"Batch Size           : {BATCH_SIZE}")


# ============================================================
# 2. EXTRACT TRAINING LABELS
# ============================================================

all_dataset_labels = []

for i in range(
    len(train_dataset)
):

    label = train_dataset[i][1]

    if torch.is_tensor(label):

        label = label.item()

    all_dataset_labels.append(
        int(label)
    )


all_dataset_labels = np.array(
    all_dataset_labels
)


print(
    f"\nTraining samples for CV: "
    f"{len(all_dataset_labels)}"
)

print(
    f"Number of classes: "
    f"{len(np.unique(all_dataset_labels))}"
)


# ============================================================
# 3. STRATIFIED K-FOLD
# ============================================================

skf = StratifiedKFold(
    n_splits=NUM_FOLDS,
    shuffle=True,
    random_state=42
)


# ============================================================
# 4. STORAGE
# ============================================================

vit_lstm_fold_results = []

vit_lstm_histories = []

global_best_val_acc = 0.0

global_best_fold = None

global_best_epoch = None


# ============================================================
# 5. CROSS-VALIDATION LOOP
# ============================================================

for fold, (
    train_indices,
    val_indices
) in enumerate(
    skf.split(
        np.zeros(
            len(all_dataset_labels)
        ),
        all_dataset_labels
    ),
    start=1
):

    print("\n")
    print("=" * 70)
    print(
        f"FOLD {fold}/{NUM_FOLDS}"
    )
    print("=" * 70)


    # --------------------------------------------------------
    # Dataset subsets
    # --------------------------------------------------------

    fold_train_dataset = Subset(
        train_dataset,
        train_indices
    )

    fold_val_dataset = Subset(
        train_dataset,
        val_indices
    )


    # --------------------------------------------------------
    # DataLoaders
    # --------------------------------------------------------

    fold_train_loader = DataLoader(
        fold_train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    fold_val_loader = DataLoader(
        fold_val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )


    print(
        f"Train samples: "
        f"{len(fold_train_dataset)}"
    )

    print(
        f"Validation samples: "
        f"{len(fold_val_dataset)}"
    )


    # ========================================================
    # 6. CREATE FRESH MODEL FOR EACH FOLD
    # ========================================================

    fold_model = ViTLSTM(
        vit=models.vit_b_16(
            weights=None
        ),
        hidden_size=256,
        num_layers=2,
        num_classes=len(class_names),
        dropout=0.3
    )


    # --------------------------------------------------------
    # Load pretrained ViT weights
    # --------------------------------------------------------

    pretrained_vit = models.vit_b_16(
        weights=vit_weights
    )


    pretrained_vit.heads = nn.Identity()


    # Load ViT backbone weights
    fold_model.vit.load_state_dict(
        pretrained_vit.state_dict(),
        strict=False
    )


    # --------------------------------------------------------
    # Freeze ViT
    # --------------------------------------------------------

    for param in fold_model.vit.parameters():

        param.requires_grad = False


    fold_model = fold_model.to(
        device
    )


    # ========================================================
    # 7. LOSS FUNCTION
    # ========================================================

    criterion = nn.CrossEntropyLoss()


    # ========================================================
    # 8. OPTIMIZER
    # ========================================================

    optimizer = torch.optim.AdamW(
        filter(
            lambda p: p.requires_grad,
            fold_model.parameters()
        ),
        lr=INITIAL_LR,
        weight_decay=WEIGHT_DECAY
    )


    # ========================================================
    # 9. LEARNING RATE SCHEDULER
    # ========================================================

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=3
    )


    # ========================================================
    # 10. HISTORY
    # ========================================================

    history = {

        "train_loss": [],

        "val_loss": [],

        "train_acc": [],

        "val_acc": [],

        "lr": []
    }


    # ========================================================
    # 11. EARLY STOPPING
    # ========================================================

    best_val_acc = 0.0

    best_val_loss = float(
        "inf"
    )

    best_epoch = 0

    patience_counter = 0

    best_model_state = None


    # ========================================================
    # 12. EPOCH LOOP
    # ========================================================

    for epoch in range(
        1,
        MAX_EPOCHS + 1
    ):


        # ====================================================
        # TRAIN
        # ====================================================

        fold_model.train()


        running_loss = 0.0

        correct = 0

        total = 0


        for images, labels in fold_train_loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )


            optimizer.zero_grad(
                set_to_none=True
            )


            outputs = fold_model(
                images
            )


            loss = criterion(
                outputs,
                labels
            )


            loss.backward()


            optimizer.step()


            running_loss += (
                loss.item()
                * images.size(0)
            )


            predictions = torch.argmax(
                outputs,
                dim=1
            )


            correct += (
                (predictions == labels)
                .sum()
                .item()
            )


            total += labels.size(0)


        train_loss = (
            running_loss / total
        )

        train_acc = (
            100.0
            * correct
            / total
        )


        # ====================================================
        # VALIDATION
        # ====================================================

        fold_model.eval()


        val_running_loss = 0.0

        val_correct = 0

        val_total = 0


        with torch.no_grad():

            for images, labels in fold_val_loader:

                images = images.to(
                    device,
                    non_blocking=True
                )

                labels = labels.to(
                    device,
                    non_blocking=True
                )


                outputs = fold_model(
                    images
                )


                loss = criterion(
                    outputs,
                    labels
                )


                val_running_loss += (
                    loss.item()
                    * images.size(0)
                )


                predictions = torch.argmax(
                    outputs,
                    dim=1
                )


                val_correct += (
                    (predictions == labels)
                    .sum()
                    .item()
                )


                val_total += labels.size(0)


        val_loss = (
            val_running_loss
            / val_total
        )


        val_acc = (
            100.0
            * val_correct
            / val_total
        )


        # ====================================================
        # SCHEDULER
        # ====================================================

        scheduler.step(
            val_loss
        )


        current_lr = (
            optimizer.param_groups[0]["lr"]
        )


        # ====================================================
        # SAVE HISTORY
        # ====================================================

        history[
            "train_loss"
        ].append(
            train_loss
        )

        history[
            "val_loss"
        ].append(
            val_loss
        )

        history[
            "train_acc"
        ].append(
            train_acc
        )

        history[
            "val_acc"
        ].append(
            val_acc
        )

        history[
            "lr"
        ].append(
            current_lr
        )


        # ====================================================
        # BEST MODEL
        # ====================================================

        if val_acc > best_val_acc:

            best_val_acc = val_acc

            best_val_loss = val_loss

            best_epoch = epoch

            patience_counter = 0

            best_model_state = copy.deepcopy(
                fold_model.state_dict()
            )

        else:

            patience_counter += 1


        # ====================================================
        # PRINT PROGRESS
        # ====================================================

        print(
            f"Epoch "
            f"{epoch:03d}/{MAX_EPOCHS} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.2f}% | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.2f}% | "
            f"LR: {current_lr:.6f}"
        )


        # ====================================================
        # EARLY STOPPING
        # ====================================================

        if (
            patience_counter
            >= EARLY_STOPPING_PATIENCE
        ):

            print(
                f"\n✓ Early stopping at "
                f"epoch {epoch}"
            )

            print(
                f"Best validation accuracy: "
                f"{best_val_acc:.2f}% "
                f"at epoch {best_epoch}"
            )

            break


    # ========================================================
    # 13. RESTORE BEST MODEL
    # ========================================================

    if best_model_state is not None:

        fold_model.load_state_dict(
            best_model_state
        )


    # ========================================================
    # 14. SAVE FOLD MODEL
    # ========================================================

    fold_model_path = (
        f"vit_lstm_fold_{fold}.pth"
    )


    torch.save(
        fold_model.state_dict(),
        fold_model_path
    )


    # ========================================================
    # 15. STORE FOLD RESULT
    # ========================================================

    vit_lstm_fold_results.append({

        "fold": fold,

        "best_epoch": best_epoch,

        "best_val_loss": best_val_loss,

        "best_val_acc": best_val_acc
    })


    vit_lstm_histories.append(
        history
    )


    # ========================================================
    # 16. GLOBAL BEST
    # ========================================================

    if (
        best_val_acc
        > global_best_val_acc
    ):

        global_best_val_acc = (
            best_val_acc
        )

        global_best_fold = fold

        global_best_epoch = best_epoch

        torch.save(
            fold_model.state_dict(),
            "best_vit_lstm.pth"
        )


    print(
        f"\n✓ Fold {fold} complete"
    )

    print(
        f"Best Epoch: {best_epoch}"
    )

    print(
        f"Best Validation Accuracy: "
        f"{best_val_acc:.2f}%"
    )


# ============================================================
# 17. CV SUMMARY
# ============================================================

vit_lstm_results_df = pd.DataFrame(
    vit_lstm_fold_results
)


cv_accuracies = (
    vit_lstm_results_df[
        "best_val_acc"
    ].values
)


mean_cv_accuracy = (
    cv_accuracies.mean()
)


std_cv_accuracy = (
    cv_accuracies.std()
)


print("\n" + "=" * 70)
print("VIT + LSTM CROSS-VALIDATION SUMMARY")
print("=" * 70)


print(
    vit_lstm_results_df.to_string(
        index=False,
        formatters={
            "best_val_loss":
                "{:.4f}".format,

            "best_val_acc":
                "{:.2f}%".format
        }
    )
)


print("\n" + "-" * 70)

print(
    f"Mean CV Accuracy : "
    f"{mean_cv_accuracy:.2f}%"
)

print(
    f"Std CV Accuracy  : "
    f"{std_cv_accuracy:.2f}%"
)

print(
    f"Best Fold        : "
    f"{global_best_fold}"
)

print(
    f"Best Epoch       : "
    f"{global_best_epoch}"
)


print("\n✓ Saved:")
print("  best_vit_lstm.pth")

for fold in range(
    1,
    NUM_FOLDS + 1
):

    print(
        f"  vit_lstm_fold_{fold}.pth"
    )


print("\n" + "=" * 70)
print("✓ VIT + LSTM 4-FOLD CV COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 84: ViT + LSTM CV RESULTS & VISUALIZATION
#          FIXED VERSION
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 70)
print("VIT + LSTM CROSS-VALIDATION RESULTS")
print("=" * 70)


# ============================================================
# 1. RECONSTRUCT FOLD RESULTS
# ============================================================

# These are the completed CV results from Cell 83

vit_lstm_fold_results = [

    {
        "fold": 1,
        "best_epoch": 38,
        "best_val_loss": np.nan,
        "best_val_acc": 85.44
    },

    {
        "fold": 2,
        "best_epoch": 43,
        "best_val_loss": np.nan,
        "best_val_acc": 85.92
    },

    {
        "fold": 3,
        "best_epoch": 50,
        "best_val_loss": np.nan,
        "best_val_acc": 86.89
    },

    {
        "fold": 4,
        "best_epoch": 25,
        "best_val_loss": np.nan,
        "best_val_acc": 82.44
    }
]


# ============================================================
# 2. CREATE DATAFRAME
# ============================================================

vit_lstm_fold_results_df = pd.DataFrame(
    vit_lstm_fold_results
)


print("\nFold-wise Validation Performance:")
print("-" * 70)

print(
    vit_lstm_fold_results_df.to_string(
        index=False,
        formatters={
            "best_val_loss": (
                lambda x:
                "N/A"
                if pd.isna(x)
                else f"{x:.4f}"
            ),

            "best_val_acc":
                "{:.2f}%".format
        }
    )
)


# ============================================================
# 3. CV STATISTICS
# ============================================================

fold_accuracies = (
    vit_lstm_fold_results_df[
        "best_val_acc"
    ].values
)


mean_accuracy = (
    fold_accuracies.mean()
)


std_accuracy = (
    fold_accuracies.std()
)


min_accuracy = (
    fold_accuracies.min()
)


max_accuracy = (
    fold_accuracies.max()
)


print("\n" + "=" * 70)
print("CROSS-VALIDATION SUMMARY")
print("=" * 70)

print(
    f"Mean CV Accuracy : "
    f"{mean_accuracy:.2f}%"
)

print(
    f"Std CV Accuracy  : "
    f"{std_accuracy:.2f}%"
)

print(
    f"Minimum Accuracy : "
    f"{min_accuracy:.2f}%"
)

print(
    f"Maximum Accuracy : "
    f"{max_accuracy:.2f}%"
)


# ============================================================
# 4. BEST FOLD
# ============================================================

best_fold_index = (
    vit_lstm_fold_results_df[
        "best_val_acc"
    ].idxmax()
)


best_fold_row = (
    vit_lstm_fold_results_df.loc[
        best_fold_index
    ]
)


print("\nBest Fold:")

print(
    f"Fold           : "
    f"{int(best_fold_row['fold'])}"
)

print(
    f"Best Epoch     : "
    f"{int(best_fold_row['best_epoch'])}"
)

print(
    f"Validation Acc : "
    f"{best_fold_row['best_val_acc']:.2f}%"
)


# ============================================================
# 5. SAVE RESULTS
# ============================================================

vit_lstm_fold_results_df.to_csv(
    "vit_lstm_cv_results.csv",
    index=False
)


# ============================================================
# 6. CV ACCURACY VISUALIZATION
# ============================================================

plt.figure(
    figsize=(10, 6)
)

plt.bar(
    vit_lstm_fold_results_df[
        "fold"
    ].astype(str),

    vit_lstm_fold_results_df[
        "best_val_acc"
    ]
)


plt.axhline(
    mean_accuracy,
    linestyle="--",
    label=(
        f"Mean CV Accuracy "
        f"({mean_accuracy:.2f}%)"
    )
)


plt.xlabel(
    "Fold"
)

plt.ylabel(
    "Validation Accuracy (%)"
)

plt.title(
    "ViT + LSTM 4-Fold Cross-Validation Accuracy"
)

plt.legend()

plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()


plt.savefig(
    "vit_lstm_cv_accuracy.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 7. BEST EPOCH VISUALIZATION
# ============================================================

plt.figure(
    figsize=(10, 6)
)

plt.bar(
    vit_lstm_fold_results_df[
        "fold"
    ].astype(str),

    vit_lstm_fold_results_df[
        "best_epoch"
    ]
)


plt.xlabel(
    "Fold"
)

plt.ylabel(
    "Best Epoch"
)

plt.title(
    "ViT + LSTM Best Epoch per Fold"
)

plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()


plt.savefig(
    "vit_lstm_best_epoch_per_fold.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 8. FINAL
# ============================================================

print("\n✓ Saved:")
print(
    "  vit_lstm_cv_results.csv"
)

print(
    "  vit_lstm_cv_accuracy.png"
)

print(
    "  vit_lstm_best_epoch_per_fold.png"
)


print("\n" + "=" * 70)
print(
    "✓ VIT + LSTM CV RESULTS & VISUALIZATION COMPLETE"
)
print("=" * 70)

In [ ]:
# ============================================================
# CHECK FOR SAVED ViT + LSTM CHECKPOINT
# ============================================================

import os

print("=" * 70)
print("SEARCHING FOR ViT + LSTM CHECKPOINT")
print("=" * 70)

for root, dirs, files in os.walk(OUTPUT_ROOT):

    for file in files:

        if "vit" in file.lower() or "lstm" in file.lower():

            print(
                os.path.join(root, file)
            )

print("\nCurrent working directory:")
print(os.getcwd())

print("\nFiles in output directory:")
print(os.listdir(OUTPUT_ROOT))

print("=" * 70)

In [ ]:
# ============================================================
# CELL 82-R: ViT + LSTM MODEL RECOVERY
# ============================================================

import torch
import torch.nn as nn
from torchvision import models

print("=" * 80)
print("RECOVERING ViT + LSTM MODEL")
print("=" * 80)

# ------------------------------------------------------------
# 1. Device
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Device       : {device}")
print(f"Num Classes  : {len(class_names)}")

# ------------------------------------------------------------
# 2. Load ImageNet-pretrained ViT-B/16
# ------------------------------------------------------------

vit_weights = models.ViT_B_16_Weights.IMAGENET1K_V1

vit_backbone = models.vit_b_16(
    weights=vit_weights
)

print("✓ ViT-B/16 pretrained backbone loaded")

# ------------------------------------------------------------
# 3. ViT + LSTM Architecture
# ------------------------------------------------------------

class ViTLSTM(nn.Module):

    def __init__(
        self,
        vit,
        num_classes,
        hidden_size=256,
        num_layers=2,
        dropout=0.3
    ):
        super().__init__()

        self.vit = vit

        # Freeze ViT backbone
        for param in self.vit.parameters():
            param.requires_grad = False

        # Remove original classification head
        self.vit.heads = nn.Identity()

        # ViT-B/16 feature dimension
        self.feature_dim = 768

        # LSTM
        self.lstm = nn.LSTM(
            input_size=self.feature_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )

        # Classifier
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(
                hidden_size,
                num_classes
            )
        )

    def forward(self, x):

        # ----------------------------------------------------
        # ViT patch embedding
        # ----------------------------------------------------

        x = self.vit._process_input(x)

        n = x.shape[0]

        # ----------------------------------------------------
        # Add CLS token
        # ----------------------------------------------------

        batch_class_token = (
            self.vit.class_token.expand(
                n, -1, -1
            )
        )

        x = torch.cat(
            [batch_class_token, x],
            dim=1
        )

        # ----------------------------------------------------
        # Positional embedding
        # ----------------------------------------------------

        x = x + self.vit.encoder.pos_embedding

        # ----------------------------------------------------
        # Transformer encoder
        # ----------------------------------------------------

        x = self.vit.encoder.dropout(x)

        x = self.vit.encoder.layers(x)

        x = self.vit.encoder.ln(x)

        # ----------------------------------------------------
        # Remove CLS token
        # Keep patch tokens
        # ----------------------------------------------------

        x = x[:, 1:, :]

        # ----------------------------------------------------
        # LSTM
        # ----------------------------------------------------

        x, _ = self.lstm(x)

        # Use final sequence representation
        x = x[:, -1, :]

        # ----------------------------------------------------
        # Classification
        # ----------------------------------------------------

        x = self.classifier(x)

        return x


# ------------------------------------------------------------
# 4. Create Model
# ------------------------------------------------------------

vit_lstm_model = ViTLSTM(
    vit=vit_backbone,
    num_classes=len(class_names),
    hidden_size=256,
    num_layers=2,
    dropout=0.3
).to(device)

# ------------------------------------------------------------
# 5. Parameter Summary
# ------------------------------------------------------------

total_params = sum(
    p.numel()
    for p in vit_lstm_model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in vit_lstm_model.parameters()
    if p.requires_grad
)

frozen_params = (
    total_params - trainable_params
)

print("\n" + "=" * 80)
print("MODEL SUMMARY")
print("=" * 80)

print(
    f"Total parameters     : {total_params:,}"
)

print(
    f"Trainable parameters : {trainable_params:,}"
)

print(
    f"Frozen parameters    : {frozen_params:,}"
)

print("\nArchitecture:")
print("ViT-B/16 → Patch Tokens → LSTM → Fully Connected → 33 Classes")

print("\n" + "=" * 80)
print("✓ ViT + LSTM MODEL RECOVERY COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# CELL 85: ViT + LSTM FINAL TEST EVALUATION
# ============================================================

import torch
import numpy as np
from sklearn.metrics import accuracy_score
from torchvision import models

print("=" * 70)
print("VIT + LSTM FINAL TEST EVALUATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Recreate ViT Backbone
# ------------------------------------------------------------

vit_backbone = models.vit_b_16(
    weights=None
)

# ------------------------------------------------------------
# 2. Recreate ViT + LSTM Model
# ------------------------------------------------------------

vit_lstm_model = ViTLSTM(
    vit=vit_backbone,
    num_classes=len(class_names),
    hidden_size=256,
    num_layers=2,
    dropout=0.3
).to(device)

# ------------------------------------------------------------
# 3. Load Best Model
# ------------------------------------------------------------

best_vit_lstm_path = "best_vit_lstm.pth"

checkpoint = torch.load(
    best_vit_lstm_path,
    map_location=device
)

# Handle checkpoint dictionary or direct state_dict
if (
    isinstance(checkpoint, dict)
    and "model_state_dict" in checkpoint
):

    state_dict = checkpoint[
        "model_state_dict"
    ]

else:

    state_dict = checkpoint

# ------------------------------------------------------------
# 4. Load with strict=False
# ------------------------------------------------------------
# The checkpoint contains the original ViT classification-head
# weights, but our ViTLSTM does not use that head.
# Therefore, those unused keys are safely ignored.

load_result = vit_lstm_model.load_state_dict(
    state_dict,
    strict=False
)

print("\nCheckpoint loading:")

print(
    f"Missing keys   : "
    f"{len(load_result.missing_keys)}"
)

print(
    f"Unexpected keys: "
    f"{len(load_result.unexpected_keys)}"
)

if load_result.unexpected_keys:

    print(
        "\nIgnored unused keys:"
    )

    for key in load_result.unexpected_keys:

        print(
            f"  - {key}"
        )

# ------------------------------------------------------------
# 5. Evaluation Mode
# ------------------------------------------------------------

vit_lstm_model.eval()

print(
    f"\n✓ Model loaded: "
    f"{best_vit_lstm_path}"
)

print(
    f"✓ Device: {device}"
)

# ------------------------------------------------------------
# 6. Test Evaluation
# ------------------------------------------------------------

test_true = []
test_pred = []
test_probs = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(
            device
        )

        outputs = vit_lstm_model(
            images
        )

        probabilities = torch.softmax(
            outputs,
            dim=1
        )

        predictions = torch.argmax(
            probabilities,
            dim=1
        )

        test_true.extend(
            labels.cpu().numpy()
        )

        test_pred.extend(
            predictions.cpu().numpy()
        )

        test_probs.extend(
            probabilities.cpu().numpy()
        )

# ------------------------------------------------------------
# 7. Convert to NumPy
# ------------------------------------------------------------

test_true = np.array(
    test_true
)

test_pred = np.array(
    test_pred
)

test_probs = np.array(
    test_probs
)

# ------------------------------------------------------------
# 8. Accuracy
# ------------------------------------------------------------

vit_lstm_test_accuracy = (
    accuracy_score(
        test_true,
        test_pred
    ) * 100
)

# ------------------------------------------------------------
# 9. Prediction Summary
# ------------------------------------------------------------

correct = np.sum(
    test_true == test_pred
)

incorrect = np.sum(
    test_true != test_pred
)

print("\n" + "=" * 70)
print("FINAL TEST RESULT")
print("=" * 70)

print(
    f"Test Samples    : "
    f"{len(test_true)}"
)

print(
    f"Test Accuracy   : "
    f"{vit_lstm_test_accuracy:.2f}%"
)

print(
    f"Correct         : "
    f"{correct}"
)

print(
    f"Incorrect       : "
    f"{incorrect}"
)

print("=" * 70)
print(
    "✓ VIT + LSTM TEST EVALUATION COMPLETE"
)
print("=" * 70)

In [ ]:
# ============================================================
# CELL 86: ViT + LSTM CLASSIFICATION REPORT
# ============================================================

from sklearn.metrics import (
    classification_report,
    precision_score,
    recall_score,
    f1_score
)

print("=" * 80)
print("VIT + LSTM CLASSIFICATION REPORT")
print("=" * 80)

# ------------------------------------------------------------
# 1. Generate Classification Report
# ------------------------------------------------------------

vit_lstm_report = classification_report(
    test_true,
    test_pred,
    labels=np.arange(len(class_names)),
    target_names=class_names,
    zero_division=0
)

print(vit_lstm_report)

# ------------------------------------------------------------
# 2. Calculate Overall Metrics
# ------------------------------------------------------------

vit_lstm_macro_precision = precision_score(
    test_true,
    test_pred,
    labels=np.arange(len(class_names)),
    average="macro",
    zero_division=0
) * 100

vit_lstm_macro_recall = recall_score(
    test_true,
    test_pred,
    labels=np.arange(len(class_names)),
    average="macro",
    zero_division=0
) * 100

vit_lstm_macro_f1 = f1_score(
    test_true,
    test_pred,
    labels=np.arange(len(class_names)),
    average="macro",
    zero_division=0
) * 100

vit_lstm_weighted_precision = precision_score(
    test_true,
    test_pred,
    labels=np.arange(len(class_names)),
    average="weighted",
    zero_division=0
) * 100

vit_lstm_weighted_recall = recall_score(
    test_true,
    test_pred,
    labels=np.arange(len(class_names)),
    average="weighted",
    zero_division=0
) * 100

vit_lstm_weighted_f1 = f1_score(
    test_true,
    test_pred,
    labels=np.arange(len(class_names)),
    average="weighted",
    zero_division=0
) * 100

# ------------------------------------------------------------
# 3. Overall Summary
# ------------------------------------------------------------

print("=" * 80)
print("VIT + LSTM OVERALL PERFORMANCE")
print("=" * 80)

print(f"Test Accuracy          : {vit_lstm_test_accuracy:.2f}%")
print(f"Macro Precision        : {vit_lstm_macro_precision:.2f}%")
print(f"Macro Recall           : {vit_lstm_macro_recall:.2f}%")
print(f"Macro F1 Score         : {vit_lstm_macro_f1:.2f}%")
print(f"Weighted Precision     : {vit_lstm_weighted_precision:.2f}%")
print(f"Weighted Recall        : {vit_lstm_weighted_recall:.2f}%")
print(f"Weighted F1 Score      : {vit_lstm_weighted_f1:.2f}%")

print("=" * 80)
print("✓ CLASSIFICATION REPORT COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# CELL 87: ViT + LSTM CONFUSION MATRIX
# ============================================================

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

print("=" * 80)
print("VIT + LSTM CONFUSION MATRIX")
print("=" * 80)

# ------------------------------------------------------------
# 1. Generate Confusion Matrix
# ------------------------------------------------------------

cm_vit_lstm = confusion_matrix(
    test_true,
    test_pred,
    labels=np.arange(len(class_names))
)

# ------------------------------------------------------------
# 2. Plot Confusion Matrix
# ------------------------------------------------------------

plt.figure(figsize=(20, 17))

sns.heatmap(
    cm_vit_lstm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    linewidths=0.3,
    linecolor="white"
)

plt.title(
    "ViT + LSTM Confusion Matrix",
    fontsize=18,
    fontweight="bold",
    pad=15
)

plt.xlabel(
    "Predicted Class",
    fontsize=13,
    fontweight="bold"
)

plt.ylabel(
    "True Class",
    fontsize=13,
    fontweight="bold"
)

plt.xticks(
    rotation=90,
    fontsize=9
)

plt.yticks(
    rotation=0,
    fontsize=9
)

plt.tight_layout()

# Save figure
plt.savefig(
    "vit_lstm_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 3. Confusion Matrix Summary
# ------------------------------------------------------------

print("\nConfusion Matrix Shape:", cm_vit_lstm.shape)
print("✓ Saved: vit_lstm_confusion_matrix.png")

print("=" * 80)
print("✓ CONFUSION MATRIX COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# CELL 88: ViT + LSTM MISCLASSIFIED IMAGES
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
import torch

print("=" * 80)
print("VIT + LSTM MISCLASSIFIED IMAGES")
print("=" * 80)

# ------------------------------------------------------------
# 1. Find misclassified samples
# ------------------------------------------------------------

misclassified_indices = np.where(test_true != test_pred)[0]

print(f"Total Test Images       : {len(test_true)}")
print(f"Correct Predictions     : {np.sum(test_true == test_pred)}")
print(f"Misclassified Images    : {len(misclassified_indices)}")

# ------------------------------------------------------------
# 2. Get image paths from test dataset
# ------------------------------------------------------------

# test_dataset should contain the original test samples
# and return (image, label)

misclassified_count = len(misclassified_indices)

if misclassified_count == 0:

    print("\n🎉 No misclassified images found!")

else:

    # Show maximum 20 images
    num_to_show = min(20, misclassified_count)

    # Calculate grid size
    cols = 4
    rows = int(np.ceil(num_to_show / cols))

    plt.figure(figsize=(18, rows * 4))

    for plot_idx, test_idx in enumerate(
        misclassified_indices[:num_to_show]
    ):

        # Get image and true label
        image, true_label = test_dataset[test_idx]

        # Get predicted label
        pred_label = test_pred[test_idx]

        # Get prediction confidence
        confidence = test_probs[test_idx][pred_label] * 100

        # ----------------------------------------------------
        # Convert tensor image back for visualization
        # ----------------------------------------------------

        image_display = image.clone()

        # If normalized tensor, reverse ImageNet normalization
        mean = torch.tensor(
            [0.485, 0.456, 0.406]
        ).view(3, 1, 1)

        std = torch.tensor(
            [0.229, 0.224, 0.225]
        ).view(3, 1, 1)

        image_display = image_display * std + mean
        image_display = torch.clamp(
            image_display,
            0,
            1
        )

        image_display = image_display.permute(
            1, 2, 0
        ).cpu().numpy()

        # ----------------------------------------------------
        # Plot
        # ----------------------------------------------------

        ax = plt.subplot(rows, cols, plot_idx + 1)

        ax.imshow(image_display)

        ax.set_title(
            f"True: {class_names[true_label]}\n"
            f"Pred: {class_names[pred_label]}\n"
            f"Confidence: {confidence:.1f}%",
            fontsize=10
        )

        ax.axis("off")

    plt.suptitle(
        "ViT + LSTM Misclassified Test Images",
        fontsize=18,
        fontweight="bold"
    )

    plt.tight_layout(rect=[0, 0, 1, 0.96])

    plt.savefig(
        "vit_lstm_misclassified_images.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    print(
        f"\n✓ Displayed {num_to_show} of "
        f"{misclassified_count} misclassified images."
    )

    print("✓ Saved: vit_lstm_misclassified_images.png")

print("=" * 80)
print("✓ MISCLASSIFIED IMAGE ANALYSIS COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# CELL 89: ViT + LSTM PREDICTION CONFIDENCE ANALYSIS
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

print("=" * 80)
print("VIT + LSTM PREDICTION CONFIDENCE ANALYSIS")
print("=" * 80)

# ------------------------------------------------------------
# 1. Get prediction confidence
# ------------------------------------------------------------

prediction_confidences = np.max(
    test_probs,
    axis=1
) * 100

# Correct / Incorrect masks
correct_mask = test_true == test_pred
incorrect_mask = test_true != test_pred

correct_confidences = prediction_confidences[correct_mask]
incorrect_confidences = prediction_confidences[incorrect_mask]

# ------------------------------------------------------------
# 2. Basic Statistics
# ------------------------------------------------------------

print("\nCONFIDENCE SUMMARY")
print("-" * 80)

print(
    f"Average Confidence (All)       : "
    f"{np.mean(prediction_confidences):.2f}%"
)

print(
    f"Average Confidence (Correct)    : "
    f"{np.mean(correct_confidences):.2f}%"
)

if len(incorrect_confidences) > 0:
    print(
        f"Average Confidence (Incorrect)  : "
        f"{np.mean(incorrect_confidences):.2f}%"
    )
else:
    print(
        "Average Confidence (Incorrect)  : N/A"
    )

print(
    f"Minimum Confidence               : "
    f"{np.min(prediction_confidences):.2f}%"
)

print(
    f"Maximum Confidence               : "
    f"{np.max(prediction_confidences):.2f}%"
)

# ------------------------------------------------------------
# 3. Confidence Histogram
# ------------------------------------------------------------

plt.figure(figsize=(12, 6))

plt.hist(
    correct_confidences,
    bins=20,
    alpha=0.7,
    label="Correct Predictions"
)

if len(incorrect_confidences) > 0:
    plt.hist(
        incorrect_confidences,
        bins=20,
        alpha=0.7,
        label="Incorrect Predictions"
    )

plt.xlabel(
    "Prediction Confidence (%)",
    fontsize=12
)

plt.ylabel(
    "Number of Images",
    fontsize=12
)

plt.title(
    "ViT + LSTM Prediction Confidence Distribution",
    fontsize=16,
    fontweight="bold"
)

plt.legend()
plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "vit_lstm_prediction_confidence.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 4. Confidence of Incorrect Predictions
# ------------------------------------------------------------

if len(incorrect_confidences) > 0:

    print("\nINCORRECT PREDICTION CONFIDENCE")
    print("-" * 80)

    print(
        f"Highest Incorrect Confidence : "
        f"{np.max(incorrect_confidences):.2f}%"
    )

    print(
        f"Lowest Incorrect Confidence  : "
        f"{np.min(incorrect_confidences):.2f}%"
    )

    # Top 10 most confident wrong predictions
    wrong_indices = np.where(incorrect_mask)[0]

    sorted_wrong = wrong_indices[
        np.argsort(
            prediction_confidences[wrong_indices]
        )[::-1]
    ]

    top_n = min(10, len(sorted_wrong))

    print("\nTOP HIGH-CONFIDENCE WRONG PREDICTIONS")
    print("-" * 80)

    for rank, idx in enumerate(
        sorted_wrong[:top_n],
        start=1
    ):

        true_name = class_names[test_true[idx]]
        pred_name = class_names[test_pred[idx]]
        confidence = prediction_confidences[idx]

        print(
            f"{rank:2d}. "
            f"True: {true_name:<18} | "
            f"Predicted: {pred_name:<18} | "
            f"Confidence: {confidence:.2f}%"
        )

# ------------------------------------------------------------
# 5. Save confidence data
# ------------------------------------------------------------

confidence_data = []

for i in range(len(test_true)):

    confidence_data.append({
        "index": i,
        "true_class": class_names[test_true[i]],
        "predicted_class": class_names[test_pred[i]],
        "confidence": prediction_confidences[i],
        "correct": bool(test_true[i] == test_pred[i])
    })

import pandas as pd

confidence_df = pd.DataFrame(confidence_data)

confidence_df.to_csv(
    "vit_lstm_prediction_confidence.csv",
    index=False
)

print("\n✓ Saved:")
print("  vit_lstm_prediction_confidence.png")
print("  vit_lstm_prediction_confidence.csv")

print("=" * 80)
print("✓ PREDICTION CONFIDENCE ANALYSIS COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# CELL 90: ViT + LSTM CLASS-WISE F1 SCORE
# ============================================================

from sklearn.metrics import precision_recall_fscore_support
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

print("=" * 80)
print("VIT + LSTM CLASS-WISE PERFORMANCE")
print("=" * 80)

# ------------------------------------------------------------
# 1. Calculate class-wise metrics
# ------------------------------------------------------------

labels = np.arange(len(class_names))

precision, recall, f1, support = precision_recall_fscore_support(
    test_true,
    test_pred,
    labels=labels,
    zero_division=0
)

# ------------------------------------------------------------
# 2. Create DataFrame
# ------------------------------------------------------------

vit_lstm_class_metrics = pd.DataFrame({
    "Species": class_names,
    "Precision": precision * 100,
    "Recall": recall * 100,
    "F1 Score": f1 * 100,
    "Support": support
})

# Sort by F1 score
vit_lstm_class_metrics_sorted = (
    vit_lstm_class_metrics
    .sort_values("F1 Score", ascending=False)
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 3. Display complete table
# ------------------------------------------------------------

print("\nCLASS-WISE PERFORMANCE")
print("-" * 80)

print(
    vit_lstm_class_metrics_sorted.to_string(
        index=False,
        formatters={
            "Precision": "{:.2f}%".format,
            "Recall": "{:.2f}%".format,
            "F1 Score": "{:.2f}%".format
        }
    )
)

# ------------------------------------------------------------
# 4. F1 Score Visualization
# ------------------------------------------------------------

plt.figure(figsize=(16, 10))

bars = plt.barh(
    vit_lstm_class_metrics_sorted["Species"],
    vit_lstm_class_metrics_sorted["F1 Score"]
)

plt.xlabel(
    "F1 Score (%)",
    fontsize=12
)

plt.ylabel(
    "Species",
    fontsize=12
)

plt.title(
    "ViT + LSTM Class-wise F1 Score",
    fontsize=16,
    fontweight="bold"
)

plt.xlim(0, 105)

# Add F1 values
for bar, value in zip(
    bars,
    vit_lstm_class_metrics_sorted["F1 Score"]
):
    plt.text(
        value + 1,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.1f}%",
        va="center",
        fontsize=8
    )

plt.grid(
    axis="x",
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "vit_lstm_classwise_f1.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 5. Save class-wise metrics
# ------------------------------------------------------------

vit_lstm_class_metrics.to_csv(
    "vit_lstm_classwise_metrics.csv",
    index=False
)

print("\n✓ Saved:")
print("  vit_lstm_classwise_f1.png")
print("  vit_lstm_classwise_metrics.csv")

print("=" * 80)
print("✓ CLASS-WISE F1 ANALYSIS COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# CELL 91: ViT + LSTM SUPPORT VS F1 ANALYSIS
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print("=" * 80)
print("VIT + LSTM SUPPORT VS F1 ANALYSIS")
print("=" * 80)

# ------------------------------------------------------------
# 1. Prepare data
# ------------------------------------------------------------

support_f1_df = vit_lstm_class_metrics.copy()

# Error count
support_f1_df["Errors"] = (
    support_f1_df["Support"]
    * (1 - support_f1_df["Recall"] / 100)
).round().astype(int)

# Error rate
support_f1_df["Error Rate"] = np.where(
    support_f1_df["Support"] > 0,
    (support_f1_df["Errors"] / support_f1_df["Support"]) * 100,
    0
)

# ------------------------------------------------------------
# 2. Display support and F1
# ------------------------------------------------------------

print("\nSPECIES SUPPORT VS F1")
print("-" * 80)

display_df = (
    support_f1_df[
        ["Species", "Support", "F1 Score", "Recall", "Errors", "Error Rate"]
    ]
    .sort_values("Support")
    .reset_index(drop=True)
)

print(
    display_df.to_string(
        index=False,
        formatters={
            "F1 Score": "{:.2f}%".format,
            "Recall": "{:.2f}%".format,
            "Error Rate": "{:.2f}%".format
        }
    )
)

# ------------------------------------------------------------
# 3. Scatter Plot: Support vs F1
# ------------------------------------------------------------

plt.figure(figsize=(14, 9))

plt.scatter(
    support_f1_df["Support"],
    support_f1_df["F1 Score"],
    s=80,
    alpha=0.75
)

# Add species labels
for _, row in support_f1_df.iterrows():

    plt.annotate(
        row["Species"],
        (
            row["Support"],
            row["F1 Score"]
        ),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8
    )

plt.xlabel(
    "Test Set Support (Number of Images)",
    fontsize=12
)

plt.ylabel(
    "F1 Score (%)",
    fontsize=12
)

plt.title(
    "ViT + LSTM: Test Support vs Class-wise F1 Score",
    fontsize=16,
    fontweight="bold"
)

plt.ylim(-5, 105)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "vit_lstm_support_vs_f1.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 4. Identify low-support classes
# ------------------------------------------------------------

LOW_SUPPORT_THRESHOLD = 3

low_support_df = support_f1_df[
    support_f1_df["Support"] <= LOW_SUPPORT_THRESHOLD
].sort_values("Support")

print("\nLOW-SUPPORT SPECIES")
print("-" * 80)

if len(low_support_df) > 0:

    print(
        low_support_df[
            ["Species", "Support", "F1 Score", "Recall"]
        ].to_string(
            index=False,
            formatters={
                "F1 Score": "{:.2f}%".format,
                "Recall": "{:.2f}%".format
            }
        )
    )

else:

    print("No species found with support <= 3.")

# ------------------------------------------------------------
# 5. Save analysis
# ------------------------------------------------------------

support_f1_df.to_csv(
    "vit_lstm_support_vs_f1.csv",
    index=False
)

print("\n✓ Saved:")
print("  vit_lstm_support_vs_f1.png")
print("  vit_lstm_support_vs_f1.csv")

print("=" * 80)
print("✓ SUPPORT VS F1 ANALYSIS COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# CELL 92: ViT + LSTM FINAL PERFORMANCE SUMMARY
# ============================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("VIT + LSTM FINAL PERFORMANCE SUMMARY")
print("=" * 80)

# ------------------------------------------------------------
# 1. Cross-Validation Results
# ------------------------------------------------------------

cv_mean = 85.17
cv_std = 1.66
cv_min = 82.44
cv_max = 86.89

# ------------------------------------------------------------
# 2. Test Performance
# ------------------------------------------------------------

summary_data = {
    "Metric": [
        "Mean CV Accuracy",
        "CV Standard Deviation",
        "Minimum CV Accuracy",
        "Maximum CV Accuracy",
        "Test Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1 Score",
        "Weighted Precision",
        "Weighted Recall",
        "Weighted F1 Score"
    ],

    "Value": [
        f"{cv_mean:.2f}%",
        f"{cv_std:.2f}%",
        f"{cv_min:.2f}%",
        f"{cv_max:.2f}%",
        f"{vit_lstm_test_accuracy:.2f}%",
        f"{vit_lstm_macro_precision:.2f}%",
        f"{vit_lstm_macro_recall:.2f}%",
        f"{vit_lstm_macro_f1:.2f}%",
        f"{vit_lstm_weighted_precision:.2f}%",
        f"{vit_lstm_weighted_recall:.2f}%",
        f"{vit_lstm_weighted_f1:.2f}%"
    ]
}

vit_lstm_final_summary = pd.DataFrame(summary_data)

# ------------------------------------------------------------
# 3. Display Summary
# ------------------------------------------------------------

print("\nFINAL PERFORMANCE")
print("-" * 80)

print(
    vit_lstm_final_summary.to_string(
        index=False
    )
)

# ------------------------------------------------------------
# 4. Dataset / Model Information
# ------------------------------------------------------------

print("\nMODEL INFORMATION")
print("-" * 80)

print(f"Architecture       : ViT-B/16 + 2-Layer LSTM")
print(f"Number of Classes  : {len(class_names)}")
print(f"Test Samples       : {len(test_true)}")
print(f"CV Folds           : 4")
print(f"Maximum Epochs     : 150")
print(f"Early Stopping     : 10 epochs")
print(f"Optimizer          : AdamW")
print(f"Initial Learning Rate : 0.001")
print(f"Weight Decay       : 0.0001")
print(f"Batch Size         : 32")

# ------------------------------------------------------------
# 5. Best CV Fold
# ------------------------------------------------------------

print("\nBEST CROSS-VALIDATION FOLD")
print("-" * 80)

print("Best Fold          : 3")
print("Best Epoch         : 50")
print("Best Validation Acc: 86.89%")

# ------------------------------------------------------------
# 6. Save Summary
# ------------------------------------------------------------

vit_lstm_final_summary.to_csv(
    "vit_lstm_final_performance_summary.csv",
    index=False
)

print("\n✓ Saved:")
print("  vit_lstm_final_performance_summary.csv")

print("=" * 80)
print("✓ FINAL VIT + LSTM SUMMARY COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# CELL 93: ViT + LSTM MODEL EXPORT
# ============================================================

import torch
import json
import os

print("=" * 80)
print("VIT + LSTM MODEL EXPORT")
print("=" * 80)

# ------------------------------------------------------------
# 1. Save Model Weights
# ------------------------------------------------------------

vit_lstm_model_path = "final_vit_lstm_leaf_classifier.pth"

torch.save(
    vit_lstm_model.state_dict(),
    vit_lstm_model_path
)

print(f"✓ Model weights saved: {vit_lstm_model_path}")

# ------------------------------------------------------------
# 2. Save Class Names
# ------------------------------------------------------------

class_names_path = "vit_lstm_class_names.json"

with open(
    class_names_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        class_names,
        f,
        ensure_ascii=False,
        indent=4
    )

print(f"✓ Class names saved: {class_names_path}")

# ------------------------------------------------------------
# 3. Model Architecture Information
# ------------------------------------------------------------

model_info = {
    "model_name": "ViT + LSTM",
    "backbone": "ViT-B/16",
    "backbone_pretrained": True,
    "backbone_frozen": True,
    "lstm_layers": 2,
    "lstm_hidden_size": 256,
    "dropout": 0.3,
    "num_classes": len(class_names),
    "num_parameters": sum(
        p.numel()
        for p in vit_lstm_model.parameters()
    ),
    "trainable_parameters": sum(
        p.numel()
        for p in vit_lstm_model.parameters()
        if p.requires_grad
    ),
    "device": str(device),
    "cv_folds": 4,
    "max_epochs": 150,
    "early_stopping_patience": 10,
    "optimizer": "AdamW",
    "learning_rate": 0.001,
    "weight_decay": 0.0001,
    "batch_size": 32
}

model_info_path = "vit_lstm_model_info.json"

with open(
    model_info_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        model_info,
        f,
        indent=4
    )

print(f"✓ Model information saved: {model_info_path}")

# ------------------------------------------------------------
# 4. Save Final Performance
# ------------------------------------------------------------

performance_info = {
    "mean_cv_accuracy": cv_mean,
    "cv_std": cv_std,
    "cv_min_accuracy": cv_min,
    "cv_max_accuracy": cv_max,
    "test_accuracy": vit_lstm_test_accuracy,
    "macro_precision": vit_lstm_macro_precision,
    "macro_recall": vit_lstm_macro_recall,
    "macro_f1": vit_lstm_macro_f1,
    "weighted_precision": vit_lstm_weighted_precision,
    "weighted_recall": vit_lstm_weighted_recall,
    "weighted_f1": vit_lstm_weighted_f1
}

performance_path = "vit_lstm_performance.json"

with open(
    performance_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        performance_info,
        f,
        indent=4
    )

print(f"✓ Performance saved: {performance_path}")

# ------------------------------------------------------------
# 5. Export Summary
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("EXPORTED FILES")
print("=" * 80)

exported_files = [
    vit_lstm_model_path,
    class_names_path,
    model_info_path,
    performance_path,
    "vit_lstm_cv_results.csv",
    "vit_lstm_final_performance_summary.csv",
    "vit_lstm_classwise_metrics.csv",
    "vit_lstm_prediction_confidence.csv",
    "vit_lstm_confusion_matrix.png",
    "vit_lstm_misclassified_images.png",
    "vit_lstm_prediction_confidence.png",
    "vit_lstm_classwise_f1.png",
    "vit_lstm_support_vs_f1.png"
]

for file_name in exported_files:

    status = "✓" if os.path.exists(file_name) else "✗"

    print(f"{status} {file_name}")

print("\n" + "=" * 80)
print("✓ VIT + LSTM MODEL EXPORT COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# CELL 94: FINAL 4-MODEL COMPARISON
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("=" * 90)
print("FINAL COMPARISON: FOUR DEEP LEARNING MODELS")
print("=" * 90)

# ------------------------------------------------------------
# 1. Model Performance Data
# ------------------------------------------------------------

comparison_data = [
    {
        "Model": "EfficientNet-B0",
        "Mean CV Accuracy": 92.71,
        "Test Accuracy": 93.22,
        "Macro Precision": 80.10,
        "Macro Recall": 77.98,
        "Macro F1": 78.98,
        "Weighted Precision": 93.12,
        "Weighted Recall": 93.22,
        "Weighted F1": 92.23
    },

    {
        "Model": "ResNet50",
        "Mean CV Accuracy": 88.58,
        "Test Accuracy": 90.96,
        "Macro Precision": 76.22,
        "Macro Recall": 77.63,
        "Macro F1": 75.40,
        "Weighted Precision": 89.92,
        "Weighted Recall": 90.96,
        "Weighted F1": 89.35
    },

    {
        "Model": "DenseNet121",
        "Mean CV Accuracy": 88.58,
        "Test Accuracy": 89.83,
        "Macro Precision": 82.88,
        "Macro Recall": 78.76,
        "Macro F1": 78.48,
        "Weighted Precision": 91.54,
        "Weighted Recall": 89.83,
        "Weighted F1": 89.17
    },

    {
        "Model": "ViT + LSTM",
        "Mean CV Accuracy": cv_mean,
        "Test Accuracy": vit_lstm_test_accuracy,
        "Macro Precision": vit_lstm_macro_precision,
        "Macro Recall": vit_lstm_macro_recall,
        "Macro F1": vit_lstm_macro_f1,
        "Weighted Precision": vit_lstm_weighted_precision,
        "Weighted Recall": vit_lstm_weighted_recall,
        "Weighted F1": vit_lstm_weighted_f1
    }
]

comparison_df = pd.DataFrame(comparison_data)

# ------------------------------------------------------------
# 2. Display Full Comparison
# ------------------------------------------------------------

print("\nFOUR-MODEL PERFORMANCE COMPARISON")
print("-" * 90)

print(
    comparison_df.to_string(
        index=False,
        formatters={
            "Mean CV Accuracy": "{:.2f}%".format,
            "Test Accuracy": "{:.2f}%".format,
            "Macro Precision": "{:.2f}%".format,
            "Macro Recall": "{:.2f}%".format,
            "Macro F1": "{:.2f}%".format,
            "Weighted Precision": "{:.2f}%".format,
            "Weighted Recall": "{:.2f}%".format,
            "Weighted F1": "{:.2f}%".format
        }
    )
)

# ------------------------------------------------------------
# 3. CV Accuracy Comparison
# ------------------------------------------------------------

plt.figure(figsize=(12, 7))

bars = plt.bar(
    comparison_df["Model"],
    comparison_df["Mean CV Accuracy"]
)

plt.ylabel(
    "Mean CV Accuracy (%)",
    fontsize=12
)

plt.xlabel(
    "Model",
    fontsize=12
)

plt.title(
    "Four-Model Comparison: Mean Cross-Validation Accuracy",
    fontsize=16,
    fontweight="bold"
)

plt.ylim(0, 100)

for bar, value in zip(
    bars,
    comparison_df["Mean CV Accuracy"]
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 1,
        f"{value:.2f}%",
        ha="center",
        fontsize=11
    )

plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "four_model_cv_accuracy_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 4. Test Accuracy Comparison
# ------------------------------------------------------------

plt.figure(figsize=(12, 7))

bars = plt.bar(
    comparison_df["Model"],
    comparison_df["Test Accuracy"]
)

plt.ylabel(
    "Test Accuracy (%)",
    fontsize=12
)

plt.xlabel(
    "Model",
    fontsize=12
)

plt.title(
    "Four-Model Comparison: Independent Test Accuracy",
    fontsize=16,
    fontweight="bold"
)

plt.ylim(0, 100)

for bar, value in zip(
    bars,
    comparison_df["Test Accuracy"]
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 1,
        f"{value:.2f}%",
        ha="center",
        fontsize=11
    )

plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "four_model_test_accuracy_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 5. F1 Score Comparison
# ------------------------------------------------------------

metrics_to_plot = [
    "Macro F1",
    "Weighted F1"
]

x = np.arange(len(comparison_df))
width = 0.35

plt.figure(figsize=(13, 7))

bars1 = plt.bar(
    x - width / 2,
    comparison_df["Macro F1"],
    width,
    label="Macro F1"
)

bars2 = plt.bar(
    x + width / 2,
    comparison_df["Weighted F1"],
    width,
    label="Weighted F1"
)

plt.xticks(
    x,
    comparison_df["Model"]
)

plt.ylabel(
    "F1 Score (%)",
    fontsize=12
)

plt.xlabel(
    "Model",
    fontsize=12
)

plt.title(
    "Four-Model Comparison: Macro vs Weighted F1",
    fontsize=16,
    fontweight="bold"
)

plt.ylim(0, 100)

plt.legend()

for bars_group in [bars1, bars2]:

    for bar in bars_group:

        value = bar.get_height()

        plt.text(
            bar.get_x() + bar.get_width() / 2,
            value + 1,
            f"{value:.1f}",
            ha="center",
            fontsize=9
        )

plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "four_model_f1_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 6. Save Final Comparison
# ------------------------------------------------------------

comparison_df.to_csv(
    "four_model_final_comparison.csv",
    index=False
)

print("\n✓ Saved:")
print("  four_model_final_comparison.csv")
print("  four_model_cv_accuracy_comparison.png")
print("  four_model_test_accuracy_comparison.png")
print("  four_model_f1_comparison.png")

print("=" * 90)
print("✓ FOUR-MODEL COMPARISON COMPLETE")
print("=" * 90)

In [ ]:
# ============================================================
# CELL 95: PURE VISION TRANSFORMER (ViT-B/16) SETUP
# ============================================================

import torch
import torch.nn as nn
from torchvision import models

print("=" * 80)
print("PURE VISION TRANSFORMER (ViT-B/16) SETUP")
print("=" * 80)

# ------------------------------------------------------------
# 1. Device
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Device: {device}")

# ------------------------------------------------------------
# 2. Number of Classes
# ------------------------------------------------------------

NUM_CLASSES = len(class_names)

print(f"Number of Classes: {NUM_CLASSES}")

# ------------------------------------------------------------
# 3. Load ImageNet Pretrained ViT-B/16
# ------------------------------------------------------------

vit_weights = models.ViT_B_16_Weights.DEFAULT

vit_model = models.vit_b_16(
    weights=vit_weights
)

# ------------------------------------------------------------
# 4. Freeze ViT Backbone
# ------------------------------------------------------------

for param in vit_model.parameters():
    param.requires_grad = False

# ------------------------------------------------------------
# 5. Replace Classification Head
# ------------------------------------------------------------

in_features = vit_model.heads.head.in_features

vit_model.heads.head = nn.Linear(
    in_features,
    NUM_CLASSES
)

# Make only the new classification head trainable
for param in vit_model.heads.head.parameters():
    param.requires_grad = True

# ------------------------------------------------------------
# 6. Move Model to Device
# ------------------------------------------------------------

vit_model = vit_model.to(device)

# ------------------------------------------------------------
# 7. Parameter Count
# ------------------------------------------------------------

total_params = sum(
    p.numel()
    for p in vit_model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in vit_model.parameters()
    if p.requires_grad
)

frozen_params = total_params - trainable_params

# ------------------------------------------------------------
# 8. Model Summary
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("MODEL ARCHITECTURE")
print("=" * 80)

print(vit_model)

print("\n" + "=" * 80)
print("PARAMETER SUMMARY")
print("=" * 80)

print(f"Total Parameters      : {total_params:,}")
print(f"Trainable Parameters  : {trainable_params:,}")
print(f"Frozen Parameters     : {frozen_params:,}")

print("\n" + "=" * 80)
print("CLASSIFIER INFORMATION")
print("=" * 80)

print(f"Original Head Input   : {in_features}")
print(f"Output Classes        : {NUM_CLASSES}")
print("Architecture          : ViT-B/16 → Classification Head")
print("Pretrained            : ImageNet")
print("Backbone Frozen       : Yes")

print("\n" + "=" * 80)
print("✓ PURE VIT SETUP COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# CELL 96: PURE ViT-B/16 4-FOLD CROSS-VALIDATION TRAINING
# ============================================================

import torch
import torch.nn as nn
import numpy as np
import copy
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

print("=" * 80)
print("PURE ViT-B/16 4-FOLD CROSS-VALIDATION TRAINING")
print("=" * 80)

# ------------------------------------------------------------
# 1. Training Configuration
# ------------------------------------------------------------

NUM_FOLDS = 4
MAX_EPOCHS = 150
EARLY_STOPPING_PATIENCE = 10

INITIAL_LR = 0.001
WEIGHT_DECAY = 0.0001
BATCH_SIZE = 32

print("\nTraining Configuration:")
print(f"Folds                  : {NUM_FOLDS}")
print(f"Maximum Epochs         : {MAX_EPOCHS}")
print(f"Early Stopping         : {EARLY_STOPPING_PATIENCE}")
print(f"Learning Rate          : {INITIAL_LR}")
print(f"Weight Decay           : {WEIGHT_DECAY}")
print(f"Batch Size             : {BATCH_SIZE}")

# ------------------------------------------------------------
# 2. Extract Labels from Training Dataset
# ------------------------------------------------------------

all_dataset_labels = []

for i in range(len(train_dataset)):

    label = train_dataset[i][1]

    if torch.is_tensor(label):
        label = label.item()

    all_dataset_labels.append(int(label))

all_dataset_labels = np.array(
    all_dataset_labels
)

print(f"\nTraining Samples: {len(all_dataset_labels)}")
print(f"Number of Classes: {len(np.unique(all_dataset_labels))}")

# ------------------------------------------------------------
# 3. Stratified K-Fold
# ------------------------------------------------------------

skf = StratifiedKFold(
    n_splits=NUM_FOLDS,
    shuffle=True,
    random_state=42
)

vit_fold_results = []

global_best_acc = 0.0
global_best_state = None
global_best_fold = None
global_best_epoch = None

# ------------------------------------------------------------
# 4. Cross-Validation
# ------------------------------------------------------------

for fold, (train_idx, val_idx) in enumerate(
    skf.split(
        np.zeros(len(all_dataset_labels)),
        all_dataset_labels
    ),
    start=1
):

    print("\n" + "=" * 80)
    print(f"FOLD {fold}/{NUM_FOLDS}")
    print("=" * 80)

    # --------------------------------------------------------
    # Create fold datasets
    # --------------------------------------------------------

    fold_train_dataset = Subset(
        train_dataset,
        train_idx
    )

    fold_val_dataset = Subset(
        train_dataset,
        val_idx
    )

    fold_train_loader = DataLoader(
        fold_train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=torch.cuda.is_available()
    )

    fold_val_loader = DataLoader(
        fold_val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=torch.cuda.is_available()
    )

    # --------------------------------------------------------
    # Fresh ViT model for each fold
    # --------------------------------------------------------

    fold_model = models.vit_b_16(
        weights=vit_weights
    )

    # Freeze backbone
    for param in fold_model.parameters():
        param.requires_grad = False

    # Replace classification head
    fold_model.heads.head = nn.Linear(
        fold_model.heads.head.in_features,
        NUM_CLASSES
    )

    # Train only classifier
    for param in fold_model.heads.head.parameters():
        param.requires_grad = True

    fold_model = fold_model.to(device)

    # --------------------------------------------------------
    # Loss, Optimizer, Scheduler
    # --------------------------------------------------------

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        fold_model.heads.head.parameters(),
        lr=INITIAL_LR,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=3
    )

    # --------------------------------------------------------
    # Tracking
    # --------------------------------------------------------

    best_val_loss = float("inf")
    best_val_acc = 0.0
    best_epoch = 0
    patience_counter = 0

    fold_history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    # --------------------------------------------------------
    # Training Loop
    # --------------------------------------------------------

    for epoch in range(1, MAX_EPOCHS + 1):

        # ====================================================
        # TRAIN
        # ====================================================

        fold_model.train()

        train_running_loss = 0.0
        train_correct = 0
        train_total = 0

        for images, labels in fold_train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = fold_model(images)

            loss = criterion(
                outputs,
                labels
            )

            loss.backward()

            optimizer.step()

            train_running_loss += (
                loss.item() * images.size(0)
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            train_correct += (
                predictions == labels
            ).sum().item()

            train_total += labels.size(0)

        train_loss = (
            train_running_loss / train_total
        )

        train_acc = (
            train_correct / train_total
        ) * 100

        # ====================================================
        # VALIDATION
        # ====================================================

        fold_model.eval()

        val_running_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():

            for images, labels in fold_val_loader:

                images = images.to(device)
                labels = labels.to(device)

                outputs = fold_model(images)

                loss = criterion(
                    outputs,
                    labels
                )

                val_running_loss += (
                    loss.item() * images.size(0)
                )

                predictions = torch.argmax(
                    outputs,
                    dim=1
                )

                val_correct += (
                    predictions == labels
                ).sum().item()

                val_total += labels.size(0)

        val_loss = (
            val_running_loss / val_total
        )

        val_acc = (
            val_correct / val_total
        ) * 100

        scheduler.step(val_loss)

        # ----------------------------------------------------
        # Store History
        # ----------------------------------------------------

        fold_history["train_loss"].append(train_loss)
        fold_history["train_acc"].append(train_acc)
        fold_history["val_loss"].append(val_loss)
        fold_history["val_acc"].append(val_acc)

        # ----------------------------------------------------
        # Best Model
        # ----------------------------------------------------

        if val_acc > best_val_acc:

            best_val_acc = val_acc
            best_val_loss = val_loss
            best_epoch = epoch

            best_fold_state = copy.deepcopy(
                fold_model.state_dict()
            )

            patience_counter = 0

        else:

            patience_counter += 1

        # ----------------------------------------------------
        # Print Progress
        # ----------------------------------------------------

        if (
            epoch == 1
            or epoch % 5 == 0
            or epoch == best_epoch
        ):

            print(
                f"Epoch {epoch:3d}/{MAX_EPOCHS} | "
                f"Train Loss: {train_loss:.4f} | "
                f"Train Acc: {train_acc:6.2f}% | "
                f"Val Loss: {val_loss:.4f} | "
                f"Val Acc: {val_acc:6.2f}%"
            )

        # ----------------------------------------------------
        # Early Stopping
        # ----------------------------------------------------

        if patience_counter >= EARLY_STOPPING_PATIENCE:

            print(
                f"\nEarly stopping at epoch {epoch}"
            )

            break

    # --------------------------------------------------------
    # Save Best Fold Model
    # --------------------------------------------------------

    fold_model.load_state_dict(
        best_fold_state
    )

    fold_model_path = (
        f"vit_fold_{fold}.pth"
    )

    torch.save(
        best_fold_state,
        fold_model_path
    )

    # --------------------------------------------------------
    # Store Fold Result
    # --------------------------------------------------------

    vit_fold_results.append({
        "fold": fold,
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "best_val_acc": best_val_acc,
        "history": fold_history
    })

    # --------------------------------------------------------
    # Global Best
    # --------------------------------------------------------

    if best_val_acc > global_best_acc:

        global_best_acc = best_val_acc
        global_best_state = copy.deepcopy(
            best_fold_state
        )
        global_best_fold = fold
        global_best_epoch = best_epoch

    print(
        f"\nFold {fold} Best Validation Accuracy: "
        f"{best_val_acc:.2f}%"
    )

# ------------------------------------------------------------
# 5. Save Global Best ViT
# ------------------------------------------------------------

torch.save(
    global_best_state,
    "best_vit.pth"
)

# ------------------------------------------------------------
# 6. Cross-Validation Summary
# ------------------------------------------------------------

fold_accuracies = [
    result["best_val_acc"]
    for result in vit_fold_results
]

mean_cv_accuracy = np.mean(
    fold_accuracies
)

std_cv_accuracy = np.std(
    fold_accuracies
)

print("\n" + "=" * 80)
print("PURE ViT CROSS-VALIDATION SUMMARY")
print("=" * 80)

for result in vit_fold_results:

    print(
        f"Fold {result['fold']}: "
        f"Best Epoch = {result['best_epoch']}, "
        f"Best Val Acc = {result['best_val_acc']:.2f}%"
    )

print("\n" + "-" * 80)

print(
    f"Mean CV Accuracy : "
    f"{mean_cv_accuracy:.2f}%"
)

print(
    f"Std CV Accuracy  : "
    f"{std_cv_accuracy:.2f}%"
)

print(
    f"Minimum Accuracy : "
    f"{min(fold_accuracies):.2f}%"
)

print(
    f"Maximum Accuracy : "
    f"{max(fold_accuracies):.2f}%"
)

print("\nGlobal Best Fold:")
print(f"Fold              : {global_best_fold}")
print(f"Best Epoch        : {global_best_epoch}")
print(f"Validation Acc    : {global_best_acc:.2f}%")

print("\n✓ Saved:")
print("  vit_fold_1.pth")
print("  vit_fold_2.pth")
print("  vit_fold_3.pth")
print("  vit_fold_4.pth")
print("  best_vit.pth")

print("=" * 80)
print("✓ PURE ViT CROSS-VALIDATION COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# CELL 97: PURE ViT CV RESULTS & VISUALIZATION
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("=" * 80)
print("PURE ViT CROSS-VALIDATION RESULTS")
print("=" * 80)

# ------------------------------------------------------------
# 1. Create Fold-wise Results Table
# ------------------------------------------------------------

vit_cv_table = pd.DataFrame([
    {
        "fold": result["fold"],
        "best_epoch": result["best_epoch"],
        "best_val_loss": result["best_val_loss"],
        "best_val_acc": result["best_val_acc"]
    }
    for result in vit_fold_results
])

print("\nFold-wise Validation Performance:")
print("-" * 80)

print(
    vit_cv_table.to_string(
        index=False,
        formatters={
            "best_val_loss": "{:.4f}".format,
            "best_val_acc": "{:.2f}%".format
        }
    )
)

# ------------------------------------------------------------
# 2. CV Summary
# ------------------------------------------------------------

fold_accuracies = vit_cv_table["best_val_acc"].values

vit_cv_mean = np.mean(fold_accuracies)
vit_cv_std = np.std(fold_accuracies)
vit_cv_min = np.min(fold_accuracies)
vit_cv_max = np.max(fold_accuracies)

best_fold_row = vit_cv_table.loc[
    vit_cv_table["best_val_acc"].idxmax()
]

print("\n" + "=" * 80)
print("CROSS-VALIDATION SUMMARY")
print("=" * 80)

print(
    f"Mean CV Accuracy : {vit_cv_mean:.2f}%"
)

print(
    f"Std CV Accuracy  : {vit_cv_std:.2f}%"
)

print(
    f"Minimum Accuracy : {vit_cv_min:.2f}%"
)

print(
    f"Maximum Accuracy : {vit_cv_max:.2f}%"
)

print("\nBest Fold:")
print(
    f"Fold           : {int(best_fold_row['fold'])}"
)

print(
    f"Best Epoch     : {int(best_fold_row['best_epoch'])}"
)

print(
    f"Validation Acc : {best_fold_row['best_val_acc']:.2f}%"
)

# ------------------------------------------------------------
# 3. Fold Accuracy Visualization
# ------------------------------------------------------------

plt.figure(figsize=(11, 7))

bars = plt.bar(
    vit_cv_table["fold"].astype(str),
    vit_cv_table["best_val_acc"]
)

plt.axhline(
    vit_cv_mean,
    linestyle="--",
    linewidth=2,
    label=f"Mean CV Accuracy = {vit_cv_mean:.2f}%"
)

plt.xlabel(
    "Fold",
    fontsize=12
)

plt.ylabel(
    "Validation Accuracy (%)",
    fontsize=12
)

plt.title(
    "Pure ViT-B/16: 4-Fold Cross-Validation Accuracy",
    fontsize=16,
    fontweight="bold"
)

plt.ylim(0, 100)

for bar, value in zip(
    bars,
    vit_cv_table["best_val_acc"]
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 1,
        f"{value:.2f}%",
        ha="center",
        fontsize=10
    )

plt.legend()
plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "vit_cv_accuracy.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 4. Best Epoch Visualization
# ------------------------------------------------------------

plt.figure(figsize=(11, 7))

bars = plt.bar(
    vit_cv_table["fold"].astype(str),
    vit_cv_table["best_epoch"]
)

plt.xlabel(
    "Fold",
    fontsize=12
)

plt.ylabel(
    "Best Epoch",
    fontsize=12
)

plt.title(
    "Pure ViT-B/16: Best Epoch per Fold",
    fontsize=16,
    fontweight="bold"
)

for bar, value in zip(
    bars,
    vit_cv_table["best_epoch"]
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 1,
        f"{int(value)}",
        ha="center",
        fontsize=10
    )

plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "vit_best_epoch_per_fold.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 5. Save Results
# ------------------------------------------------------------

vit_cv_table.to_csv(
    "vit_cv_results.csv",
    index=False
)

print("\n✓ Saved:")
print("  vit_cv_results.csv")
print("  vit_cv_accuracy.png")
print("  vit_best_epoch_per_fold.png")

print("\n" + "=" * 80)
print("✓ PURE ViT CV RESULTS & VISUALIZATION COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# CELL 98: PURE ViT TRAINING HISTORY
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

print("=" * 80)
print("PURE ViT-B/16 TRAINING HISTORY")
print("=" * 80)

# ------------------------------------------------------------
# 1. Plot Training & Validation Accuracy
# ------------------------------------------------------------

plt.figure(figsize=(12, 8))

for result in vit_fold_results:

    history = result["history"]

    epochs = np.arange(
        1,
        len(history["train_acc"]) + 1
    )

    plt.plot(
        epochs,
        history["train_acc"],
        linewidth=2,
        label=f"Fold {result['fold']} Train"
    )

    plt.plot(
        epochs,
        history["val_acc"],
        linestyle="--",
        linewidth=2,
        label=f"Fold {result['fold']} Validation"
    )

plt.xlabel(
    "Epoch",
    fontsize=12
)

plt.ylabel(
    "Accuracy (%)",
    fontsize=12
)

plt.title(
    "Pure ViT-B/16: Training vs Validation Accuracy",
    fontsize=16,
    fontweight="bold"
)

plt.legend(
    ncol=2,
    fontsize=9
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "vit_training_validation_accuracy.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 2. Plot Training & Validation Loss
# ------------------------------------------------------------

plt.figure(figsize=(12, 8))

for result in vit_fold_results:

    history = result["history"]

    epochs = np.arange(
        1,
        len(history["train_loss"]) + 1
    )

    plt.plot(
        epochs,
        history["train_loss"],
        linewidth=2,
        label=f"Fold {result['fold']} Train"
    )

    plt.plot(
        epochs,
        history["val_loss"],
        linestyle="--",
        linewidth=2,
        label=f"Fold {result['fold']} Validation"
    )

plt.xlabel(
    "Epoch",
    fontsize=12
)

plt.ylabel(
    "Loss",
    fontsize=12
)

plt.title(
    "Pure ViT-B/16: Training vs Validation Loss",
    fontsize=16,
    fontweight="bold"
)

plt.legend(
    ncol=2,
    fontsize=9
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "vit_training_validation_loss.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 3. Training History Summary
# ------------------------------------------------------------

print("\nTRAINING HISTORY SUMMARY")
print("-" * 80)

for result in vit_fold_results:

    history = result["history"]

    print(
        f"Fold {result['fold']}: "
        f"{len(history['train_acc'])} epochs | "
        f"Best Val Acc = {result['best_val_acc']:.2f}% | "
        f"Best Epoch = {result['best_epoch']}"
    )

# ------------------------------------------------------------
# 4. Save
# ------------------------------------------------------------

print("\n✓ Saved:")
print("  vit_training_validation_accuracy.png")
print("  vit_training_validation_loss.png")

print("=" * 80)
print("✓ PURE ViT TRAINING HISTORY COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# CELL 99: PURE ViT-B/16 FINAL TEST EVALUATION
# ============================================================

import torch
import numpy as np
from sklearn.metrics import accuracy_score

print("=" * 80)
print("PURE ViT-B/16 FINAL TEST EVALUATION")
print("=" * 80)

# ------------------------------------------------------------
# 1. Create Fresh ViT Model
# ------------------------------------------------------------

final_vit_model = models.vit_b_16(
    weights=None
)

# Replace classification head
final_vit_model.heads.head = nn.Linear(
    final_vit_model.heads.head.in_features,
    NUM_CLASSES
)

# ------------------------------------------------------------
# 2. Load Best CV Model
# ------------------------------------------------------------

vit_checkpoint_path = "best_vit.pth"

checkpoint = torch.load(
    vit_checkpoint_path,
    map_location=device
)

# Handle state_dict directly or checkpoint dictionary
if (
    isinstance(checkpoint, dict)
    and "model_state_dict" in checkpoint
):

    final_vit_model.load_state_dict(
        checkpoint["model_state_dict"]
    )

else:

    final_vit_model.load_state_dict(
        checkpoint
    )

final_vit_model = final_vit_model.to(device)
final_vit_model.eval()

print(f"✓ Model loaded: {vit_checkpoint_path}")
print(f"✓ Device: {device}")

# ------------------------------------------------------------
# 3. Test Evaluation
# ------------------------------------------------------------

vit_test_true = []
vit_test_pred = []
vit_test_probs = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        outputs = final_vit_model(
            images
        )

        probabilities = torch.softmax(
            outputs,
            dim=1
        )

        predictions = torch.argmax(
            probabilities,
            dim=1
        )

        vit_test_true.extend(
            labels.cpu().numpy()
        )

        vit_test_pred.extend(
            predictions.cpu().numpy()
        )

        vit_test_probs.extend(
            probabilities.cpu().numpy()
        )

# ------------------------------------------------------------
# 4. Convert to NumPy
# ------------------------------------------------------------

vit_test_true = np.array(
    vit_test_true
)

vit_test_pred = np.array(
    vit_test_pred
)

vit_test_probs = np.array(
    vit_test_probs
)

# ------------------------------------------------------------
# 5. Calculate Accuracy
# ------------------------------------------------------------

vit_test_accuracy = (
    accuracy_score(
        vit_test_true,
        vit_test_pred
    ) * 100
)

correct_predictions = np.sum(
    vit_test_true == vit_test_pred
)

incorrect_predictions = np.sum(
    vit_test_true != vit_test_pred
)

# ------------------------------------------------------------
# 6. Final Result
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PURE ViT FINAL TEST RESULT")
print("=" * 80)

print(
    f"Test Samples       : {len(vit_test_true)}"
)

print(
    f"Correct Predictions: {correct_predictions}"
)

print(
    f"Incorrect Predictions: {incorrect_predictions}"
)

print(
    f"Test Accuracy      : {vit_test_accuracy:.2f}%"
)

print("=" * 80)
print("✓ PURE ViT TEST EVALUATION COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# CELL 100: PURE ViT CLASSIFICATION REPORT
# ============================================================

from sklearn.metrics import (
    classification_report,
    precision_score,
    recall_score,
    f1_score
)

print("=" * 80)
print("PURE ViT-B/16 CLASSIFICATION REPORT")
print("=" * 80)

# ------------------------------------------------------------
# 1. Classification Report
# ------------------------------------------------------------

vit_classification_report = classification_report(
    vit_test_true,
    vit_test_pred,
    labels=np.arange(len(class_names)),
    target_names=class_names,
    zero_division=0
)

print(vit_classification_report)

# ------------------------------------------------------------
# 2. Overall Metrics
# ------------------------------------------------------------

vit_macro_precision = (
    precision_score(
        vit_test_true,
        vit_test_pred,
        labels=np.arange(len(class_names)),
        average="macro",
        zero_division=0
    ) * 100
)

vit_macro_recall = (
    recall_score(
        vit_test_true,
        vit_test_pred,
        labels=np.arange(len(class_names)),
        average="macro",
        zero_division=0
    ) * 100
)

vit_macro_f1 = (
    f1_score(
        vit_test_true,
        vit_test_pred,
        labels=np.arange(len(class_names)),
        average="macro",
        zero_division=0
    ) * 100
)

vit_weighted_precision = (
    precision_score(
        vit_test_true,
        vit_test_pred,
        labels=np.arange(len(class_names)),
        average="weighted",
        zero_division=0
    ) * 100
)

vit_weighted_recall = (
    recall_score(
        vit_test_true,
        vit_test_pred,
        labels=np.arange(len(class_names)),
        average="weighted",
        zero_division=0
    ) * 100
)

vit_weighted_f1 = (
    f1_score(
        vit_test_true,
        vit_test_pred,
        labels=np.arange(len(class_names)),
        average="weighted",
        zero_division=0
    ) * 100
)

# ------------------------------------------------------------
# 3. Overall Performance Summary
# ------------------------------------------------------------

print("=" * 80)
print("PURE ViT OVERALL PERFORMANCE")
print("=" * 80)

print(
    f"Test Accuracy          : {vit_test_accuracy:.2f}%"
)

print(
    f"Macro Precision        : {vit_macro_precision:.2f}%"
)

print(
    f"Macro Recall           : {vit_macro_recall:.2f}%"
)

print(
    f"Macro F1 Score         : {vit_macro_f1:.2f}%"
)

print(
    f"Weighted Precision     : {vit_weighted_precision:.2f}%"
)

print(
    f"Weighted Recall        : {vit_weighted_recall:.2f}%"
)

print(
    f"Weighted F1 Score      : {vit_weighted_f1:.2f}%"
)

print("=" * 80)
print("✓ PURE ViT CLASSIFICATION REPORT COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# CELL 101: PURE ViT-B/16 CONFUSION MATRIX
# ============================================================

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

print("=" * 80)
print("PURE ViT-B/16 CONFUSION MATRIX")
print("=" * 80)

# ------------------------------------------------------------
# 1. Generate Confusion Matrix
# ------------------------------------------------------------

vit_cm = confusion_matrix(
    vit_test_true,
    vit_test_pred,
    labels=np.arange(len(class_names))
)

# ------------------------------------------------------------
# 2. Plot Confusion Matrix
# ------------------------------------------------------------

plt.figure(figsize=(20, 17))

sns.heatmap(
    vit_cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    linewidths=0.3,
    linecolor="white"
)

plt.title(
    "Pure ViT-B/16 Confusion Matrix",
    fontsize=18,
    fontweight="bold",
    pad=15
)

plt.xlabel(
    "Predicted Class",
    fontsize=13,
    fontweight="bold"
)

plt.ylabel(
    "True Class",
    fontsize=13,
    fontweight="bold"
)

plt.xticks(
    rotation=90,
    fontsize=9
)

plt.yticks(
    rotation=0,
    fontsize=9
)

plt.tight_layout()

# ------------------------------------------------------------
# 3. Save Figure
# ------------------------------------------------------------

plt.savefig(
    "vit_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 4. Summary
# ------------------------------------------------------------

print(f"\nConfusion Matrix Shape: {vit_cm.shape}")

print(
    f"Total Test Samples: "
    f"{len(vit_test_true)}"
)

print(
    f"Correct Predictions: "
    f"{np.sum(vit_test_true == vit_test_pred)}"
)

print(
    f"Incorrect Predictions: "
    f"{np.sum(vit_test_true != vit_test_pred)}"
)

print("\n✓ Saved: vit_confusion_matrix.png")

print("=" * 80)
print("✓ PURE ViT CONFUSION MATRIX COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# CELL 102: PURE ViT MISCLASSIFIED IMAGES
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
import torch

print("=" * 80)
print("PURE ViT-B/16 MISCLASSIFIED IMAGES")
print("=" * 80)

# ------------------------------------------------------------
# 1. Find Misclassified Samples
# ------------------------------------------------------------

vit_misclassified_indices = np.where(
    vit_test_true != vit_test_pred
)[0]

total_test = len(vit_test_true)
total_wrong = len(vit_misclassified_indices)
total_correct = total_test - total_wrong

print(f"Total Test Images       : {total_test}")
print(f"Correct Predictions     : {total_correct}")
print(f"Misclassified Images    : {total_wrong}")

# ------------------------------------------------------------
# 2. Display Misclassified Images
# ------------------------------------------------------------

if total_wrong == 0:

    print("\n🎉 No misclassified images found!")

else:

    num_to_show = min(
        20,
        total_wrong
    )

    cols = 4

    rows = int(
        np.ceil(
            num_to_show / cols
        )
    )

    plt.figure(
        figsize=(18, rows * 4)
    )

    for plot_idx, test_idx in enumerate(
        vit_misclassified_indices[:num_to_show]
    ):

        # ----------------------------------------------------
        # Get Image
        # ----------------------------------------------------

        image, true_label = test_dataset[
            test_idx
        ]

        pred_label = vit_test_pred[
            test_idx
        ]

        confidence = (
            vit_test_probs[test_idx][pred_label]
            * 100
        )

        # ----------------------------------------------------
        # Reverse ImageNet Normalization
        # ----------------------------------------------------

        image_display = image.clone()

        mean = torch.tensor(
            [0.485, 0.456, 0.406]
        ).view(3, 1, 1)

        std = torch.tensor(
            [0.229, 0.224, 0.225]
        ).view(3, 1, 1)

        image_display = (
            image_display * std + mean
        )

        image_display = torch.clamp(
            image_display,
            0,
            1
        )

        image_display = (
            image_display
            .permute(1, 2, 0)
            .cpu()
            .numpy()
        )

        # ----------------------------------------------------
        # Plot
        # ----------------------------------------------------

        ax = plt.subplot(
            rows,
            cols,
            plot_idx + 1
        )

        ax.imshow(
            image_display
        )

        ax.set_title(
            f"True: {class_names[true_label]}\n"
            f"Pred: {class_names[pred_label]}\n"
            f"Confidence: {confidence:.1f}%",
            fontsize=10
        )

        ax.axis("off")

    plt.suptitle(
        "Pure ViT-B/16 Misclassified Test Images",
        fontsize=18,
        fontweight="bold"
    )

    plt.tight_layout(
        rect=[0, 0, 1, 0.96]
    )

    plt.savefig(
        "vit_misclassified_images.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    print(
        f"\n✓ Displayed {num_to_show} "
        f"of {total_wrong} misclassified images."
    )

    print(
        "✓ Saved: vit_misclassified_images.png"
    )

print("=" * 80)
print("✓ PURE ViT MISCLASSIFIED IMAGE ANALYSIS COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# CELL 103: PURE ViT-B/16 PREDICTION CONFIDENCE ANALYSIS
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 80)
print("PURE ViT-B/16 PREDICTION CONFIDENCE ANALYSIS")
print("=" * 80)

# ------------------------------------------------------------
# 1. Prediction Confidence
# ------------------------------------------------------------

vit_prediction_confidences = (
    np.max(
        vit_test_probs,
        axis=1
    ) * 100
)

# Correct / Incorrect masks
vit_correct_mask = (
    vit_test_true == vit_test_pred
)

vit_incorrect_mask = (
    vit_test_true != vit_test_pred
)

vit_correct_confidences = (
    vit_prediction_confidences[
        vit_correct_mask
    ]
)

vit_incorrect_confidences = (
    vit_prediction_confidences[
        vit_incorrect_mask
    ]
)

# ------------------------------------------------------------
# 2. Confidence Summary
# ------------------------------------------------------------

print("\nCONFIDENCE SUMMARY")
print("-" * 80)

print(
    f"Average Confidence (All)      : "
    f"{np.mean(vit_prediction_confidences):.2f}%"
)

print(
    f"Average Confidence (Correct)   : "
    f"{np.mean(vit_correct_confidences):.2f}%"
)

if len(vit_incorrect_confidences) > 0:

    print(
        f"Average Confidence (Incorrect) : "
        f"{np.mean(vit_incorrect_confidences):.2f}%"
    )

else:

    print(
        "Average Confidence (Incorrect) : N/A"
    )

print(
    f"Minimum Confidence             : "
    f"{np.min(vit_prediction_confidences):.2f}%"
)

print(
    f"Maximum Confidence             : "
    f"{np.max(vit_prediction_confidences):.2f}%"
)

# ------------------------------------------------------------
# 3. Confidence Distribution
# ------------------------------------------------------------

plt.figure(
    figsize=(12, 6)
)

plt.hist(
    vit_correct_confidences,
    bins=20,
    alpha=0.7,
    label="Correct Predictions"
)

if len(vit_incorrect_confidences) > 0:

    plt.hist(
        vit_incorrect_confidences,
        bins=20,
        alpha=0.7,
        label="Incorrect Predictions"
    )

plt.xlabel(
    "Prediction Confidence (%)",
    fontsize=12
)

plt.ylabel(
    "Number of Images",
    fontsize=12
)

plt.title(
    "Pure ViT-B/16 Prediction Confidence Distribution",
    fontsize=16,
    fontweight="bold"
)

plt.legend()

plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "vit_prediction_confidence.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 4. High-Confidence Wrong Predictions
# ------------------------------------------------------------

if len(vit_incorrect_confidences) > 0:

    print("\nTOP HIGH-CONFIDENCE WRONG PREDICTIONS")
    print("-" * 80)

    wrong_indices = np.where(
        vit_incorrect_mask
    )[0]

    sorted_wrong_indices = wrong_indices[
        np.argsort(
            vit_prediction_confidences[
                wrong_indices
            ]
        )[::-1]
    ]

    top_n = min(
        10,
        len(sorted_wrong_indices)
    )

    for rank, idx in enumerate(
        sorted_wrong_indices[:top_n],
        start=1
    ):

        true_name = class_names[
            vit_test_true[idx]
        ]

        predicted_name = class_names[
            vit_test_pred[idx]
        ]

        confidence = (
            vit_prediction_confidences[idx]
        )

        print(
            f"{rank:2d}. "
            f"True: {true_name:<18} | "
            f"Predicted: {predicted_name:<18} | "
            f"Confidence: {confidence:.2f}%"
        )

# ------------------------------------------------------------
# 5. Save Confidence Data
# ------------------------------------------------------------

vit_confidence_df = pd.DataFrame({
    "index": np.arange(
        len(vit_test_true)
    ),
    "true_class": [
        class_names[x]
        for x in vit_test_true
    ],
    "predicted_class": [
        class_names[x]
        for x in vit_test_pred
    ],
    "confidence": vit_prediction_confidences,
    "correct": vit_correct_mask
})

vit_confidence_df.to_csv(
    "vit_prediction_confidence.csv",
    index=False
)

print("\n✓ Saved:")
print("  vit_prediction_confidence.png")
print("  vit_prediction_confidence.csv")

print("=" * 80)
print("✓ PURE ViT CONFIDENCE ANALYSIS COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# CELL 104: PURE ViT-B/16 CLASS-WISE F1 SCORE
# ============================================================

from sklearn.metrics import precision_recall_fscore_support
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

print("=" * 80)
print("PURE ViT-B/16 CLASS-WISE PERFORMANCE")
print("=" * 80)

# ------------------------------------------------------------
# 1. Calculate Class-wise Metrics
# ------------------------------------------------------------

vit_labels = np.arange(
    len(class_names)
)

vit_precision, vit_recall, vit_f1, vit_support = (
    precision_recall_fscore_support(
        vit_test_true,
        vit_test_pred,
        labels=vit_labels,
        zero_division=0
    )
)

# ------------------------------------------------------------
# 2. Create DataFrame
# ------------------------------------------------------------

vit_class_metrics = pd.DataFrame({
    "Species": class_names,
    "Precision": vit_precision * 100,
    "Recall": vit_recall * 100,
    "F1 Score": vit_f1 * 100,
    "Support": vit_support
})

# Sort by F1
vit_class_metrics_sorted = (
    vit_class_metrics
    .sort_values(
        "F1 Score",
        ascending=False
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 3. Display Complete Table
# ------------------------------------------------------------

print("\nCLASS-WISE PERFORMANCE")
print("-" * 80)

print(
    vit_class_metrics_sorted.to_string(
        index=False,
        formatters={
            "Precision": "{:.2f}%".format,
            "Recall": "{:.2f}%".format,
            "F1 Score": "{:.2f}%".format
        }
    )
)

# ------------------------------------------------------------
# 4. F1 Score Visualization
# ------------------------------------------------------------

plt.figure(
    figsize=(16, 10)
)

bars = plt.barh(
    vit_class_metrics_sorted["Species"],
    vit_class_metrics_sorted["F1 Score"]
)

plt.xlabel(
    "F1 Score (%)",
    fontsize=12
)

plt.ylabel(
    "Species",
    fontsize=12
)

plt.title(
    "Pure ViT-B/16 Class-wise F1 Score",
    fontsize=16,
    fontweight="bold"
)

plt.xlim(
    0,
    105
)

# Add values
for bar, value in zip(
    bars,
    vit_class_metrics_sorted["F1 Score"]
):

    plt.text(
        value + 1,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.1f}%",
        va="center",
        fontsize=8
    )

plt.grid(
    axis="x",
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "vit_classwise_f1.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 5. Save Class-wise Metrics
# ------------------------------------------------------------

vit_class_metrics.to_csv(
    "vit_classwise_metrics.csv",
    index=False
)

print("\n✓ Saved:")
print("  vit_classwise_f1.png")
print("  vit_classwise_metrics.csv")

print("=" * 80)
print("✓ PURE ViT CLASS-WISE F1 ANALYSIS COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# CELL 105: PURE ViT SUPPORT VS F1 ANALYSIS
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print("=" * 80)
print("PURE ViT-B/16 SUPPORT VS F1 ANALYSIS")
print("=" * 80)

# ------------------------------------------------------------
# 1. Prepare Data
# ------------------------------------------------------------

vit_support_f1_df = vit_class_metrics.copy()

# Error count
vit_support_f1_df["Errors"] = (
    vit_support_f1_df["Support"]
    * (1 - vit_support_f1_df["Recall"] / 100)
).round().astype(int)

# Error rate
vit_support_f1_df["Error Rate"] = np.where(
    vit_support_f1_df["Support"] > 0,
    (
        vit_support_f1_df["Errors"]
        / vit_support_f1_df["Support"]
    ) * 100,
    0
)

# ------------------------------------------------------------
# 2. Display Support vs F1 Table
# ------------------------------------------------------------

print("\nSPECIES SUPPORT VS F1")
print("-" * 80)

vit_support_display = (
    vit_support_f1_df[
        [
            "Species",
            "Support",
            "F1 Score",
            "Recall",
            "Errors",
            "Error Rate"
        ]
    ]
    .sort_values(
        "Support"
    )
    .reset_index(drop=True)
)

print(
    vit_support_display.to_string(
        index=False,
        formatters={
            "F1 Score": "{:.2f}%".format,
            "Recall": "{:.2f}%".format,
            "Error Rate": "{:.2f}%".format
        }
    )
)

# ------------------------------------------------------------
# 3. Support vs F1 Scatter Plot
# ------------------------------------------------------------

plt.figure(
    figsize=(14, 9)
)

plt.scatter(
    vit_support_f1_df["Support"],
    vit_support_f1_df["F1 Score"],
    s=80,
    alpha=0.75
)

# Add species names
for _, row in vit_support_f1_df.iterrows():

    plt.annotate(
        row["Species"],
        (
            row["Support"],
            row["F1 Score"]
        ),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8
    )

plt.xlabel(
    "Test Set Support (Number of Images)",
    fontsize=12
)

plt.ylabel(
    "F1 Score (%)",
    fontsize=12
)

plt.title(
    "Pure ViT-B/16: Test Support vs Class-wise F1 Score",
    fontsize=16,
    fontweight="bold"
)

plt.ylim(
    -5,
    105
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "vit_support_vs_f1.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 4. Identify Low-Support Species
# ------------------------------------------------------------

LOW_SUPPORT_THRESHOLD = 3

vit_low_support_df = vit_support_f1_df[
    vit_support_f1_df["Support"]
    <= LOW_SUPPORT_THRESHOLD
].sort_values(
    "Support"
)

print("\nLOW-SUPPORT SPECIES")
print("-" * 80)

if len(vit_low_support_df) > 0:

    print(
        vit_low_support_df[
            [
                "Species",
                "Support",
                "F1 Score",
                "Recall"
            ]
        ].to_string(
            index=False,
            formatters={
                "F1 Score": "{:.2f}%".format,
                "Recall": "{:.2f}%".format
            }
        )
    )

else:

    print(
        "No species found with support <= 3."
    )

# ------------------------------------------------------------
# 5. Save Analysis
# ------------------------------------------------------------

vit_support_f1_df.to_csv(
    "vit_support_vs_f1.csv",
    index=False
)

print("\n✓ Saved:")
print("  vit_support_vs_f1.png")
print("  vit_support_vs_f1.csv")

print("=" * 80)
print("✓ PURE ViT SUPPORT VS F1 ANALYSIS COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# CELL 106: PURE ViT FINAL PERFORMANCE SUMMARY
# ============================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("PURE ViT-B/16 FINAL PERFORMANCE SUMMARY")
print("=" * 80)

# ------------------------------------------------------------
# 1. Cross-Validation Results
# ------------------------------------------------------------

print("\nCROSS-VALIDATION PERFORMANCE")
print("-" * 80)

print(
    f"Mean CV Accuracy : {vit_cv_mean:.2f}%"
)

print(
    f"CV Std           : {vit_cv_std:.2f}%"
)

print(
    f"Minimum CV Acc   : {vit_cv_min:.2f}%"
)

print(
    f"Maximum CV Acc   : {vit_cv_max:.2f}%"
)

print(
    f"Best Fold        : "
    f"{int(best_fold_row['fold'])}"
)

print(
    f"Best Epoch       : "
    f"{int(best_fold_row['best_epoch'])}"
)

# ------------------------------------------------------------
# 2. Test Performance
# ------------------------------------------------------------

print("\nINDEPENDENT TEST PERFORMANCE")
print("-" * 80)

print(
    f"Test Accuracy        : "
    f"{vit_test_accuracy:.2f}%"
)

print(
    f"Macro Precision      : "
    f"{vit_macro_precision:.2f}%"
)

print(
    f"Macro Recall         : "
    f"{vit_macro_recall:.2f}%"
)

print(
    f"Macro F1             : "
    f"{vit_macro_f1:.2f}%"
)

print(
    f"Weighted Precision   : "
    f"{vit_weighted_precision:.2f}%"
)

print(
    f"Weighted Recall      : "
    f"{vit_weighted_recall:.2f}%"
)

print(
    f"Weighted F1         : "
    f"{vit_weighted_f1:.2f}%"
)

# ------------------------------------------------------------
# 3. Create Summary DataFrame
# ------------------------------------------------------------

vit_final_summary = pd.DataFrame({
    "Metric": [
        "Mean CV Accuracy",
        "CV Standard Deviation",
        "Minimum CV Accuracy",
        "Maximum CV Accuracy",
        "Test Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1 Score",
        "Weighted Precision",
        "Weighted Recall",
        "Weighted F1 Score"
    ],

    "Value": [
        vit_cv_mean,
        vit_cv_std,
        vit_cv_min,
        vit_cv_max,
        vit_test_accuracy,
        vit_macro_precision,
        vit_macro_recall,
        vit_macro_f1,
        vit_weighted_precision,
        vit_weighted_recall,
        vit_weighted_f1
    ]
})

# ------------------------------------------------------------
# 4. Display Summary Table
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL PURE ViT PERFORMANCE TABLE")
print("=" * 80)

summary_display = vit_final_summary.copy()

summary_display["Value"] = (
    summary_display["Value"]
    .map(lambda x: f"{x:.2f}%")
)

print(
    summary_display.to_string(
        index=False
    )
)

# ------------------------------------------------------------
# 5. Model Information
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("MODEL INFORMATION")
print("=" * 80)

print("Architecture        : Pure ViT-B/16")
print("Backbone            : ViT-B/16")
print("Pretrained          : ImageNet")
print("Backbone Frozen     : Yes")
print("LSTM                : No")
print(f"Number of Classes   : {NUM_CLASSES}")
print(f"Test Samples        : {len(vit_test_true)}")
print("CV Folds            : 4")
print("Maximum Epochs      : 150")
print("Early Stopping      : 10 epochs")
print("Optimizer           : AdamW")
print("Learning Rate       : 0.001")
print("Weight Decay        : 0.0001")
print("Batch Size          : 32")

# ------------------------------------------------------------
# 6. Save Summary
# ------------------------------------------------------------

vit_final_summary.to_csv(
    "vit_final_performance_summary.csv",
    index=False
)

print("\n✓ Saved:")
print("  vit_final_performance_summary.csv")

print("=" * 80)
print("✓ PURE ViT FINAL SUMMARY COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# CELL 107: PURE ViT-B/16 MODEL EXPORT
# ============================================================

import torch
import json
import os

print("=" * 80)
print("PURE ViT-B/16 MODEL EXPORT")
print("=" * 80)

# ------------------------------------------------------------
# 1. Save Model Weights
# ------------------------------------------------------------

vit_model_path = "final_vit_leaf_classifier.pth"

torch.save(
    final_vit_model.state_dict(),
    vit_model_path
)

print(
    f"✓ Model weights saved: {vit_model_path}"
)

# ------------------------------------------------------------
# 2. Save Class Names
# ------------------------------------------------------------

vit_class_names_path = "vit_class_names.json"

with open(
    vit_class_names_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        class_names,
        f,
        ensure_ascii=False,
        indent=4
    )

print(
    f"✓ Class names saved: {vit_class_names_path}"
)

# ------------------------------------------------------------
# 3. Model Information
# ------------------------------------------------------------

vit_model_info = {
    "model_name": "Pure Vision Transformer",
    "architecture": "ViT-B/16",
    "backbone": "ViT-B/16",
    "pretrained": "ImageNet",
    "backbone_frozen": True,
    "lstm_used": False,
    "num_classes": NUM_CLASSES,

    "total_parameters": sum(
        p.numel()
        for p in final_vit_model.parameters()
    ),

    "trainable_parameters": sum(
        p.numel()
        for p in final_vit_model.parameters()
        if p.requires_grad
    ),

    "device": str(device),

    "cv_folds": 4,
    "max_epochs": 150,
    "early_stopping_patience": 10,

    "optimizer": "AdamW",
    "learning_rate": 0.001,
    "weight_decay": 0.0001,
    "batch_size": 32
}

vit_model_info_path = "vit_model_info.json"

with open(
    vit_model_info_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        vit_model_info,
        f,
        indent=4
    )

print(
    f"✓ Model information saved: {vit_model_info_path}"
)

# ------------------------------------------------------------
# 4. Save Performance Information
# ------------------------------------------------------------

vit_performance_info = {
    "mean_cv_accuracy": float(vit_cv_mean),
    "cv_std": float(vit_cv_std),
    "cv_min_accuracy": float(vit_cv_min),
    "cv_max_accuracy": float(vit_cv_max),

    "test_accuracy": float(vit_test_accuracy),

    "macro_precision": float(vit_macro_precision),
    "macro_recall": float(vit_macro_recall),
    "macro_f1": float(vit_macro_f1),

    "weighted_precision": float(
        vit_weighted_precision
    ),

    "weighted_recall": float(
        vit_weighted_recall
    ),

    "weighted_f1": float(
        vit_weighted_f1
    )
}

vit_performance_path = "vit_performance.json"

with open(
    vit_performance_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        vit_performance_info,
        f,
        indent=4
    )

print(
    f"✓ Performance saved: {vit_performance_path}"
)

# ------------------------------------------------------------
# 5. Check Exported Files
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("EXPORTED FILES")
print("=" * 80)

exported_files = [
    vit_model_path,
    vit_class_names_path,
    vit_model_info_path,
    vit_performance_path,

    "vit_cv_results.csv",
    "vit_final_performance_summary.csv",
    "vit_classwise_metrics.csv",
    "vit_prediction_confidence.csv",

    "vit_confusion_matrix.png",
    "vit_misclassified_images.png",
    "vit_prediction_confidence.png",
    "vit_classwise_f1.png",
    "vit_support_vs_f1.png"
]

for file_name in exported_files:

    status = "✓" if os.path.exists(
        file_name
    ) else "✗"

    print(
        f"{status} {file_name}"
    )

print("\n" + "=" * 80)
print("✓ PURE ViT MODEL EXPORT COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# CELL 108: FINAL 5-MODEL COMPARISON
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("=" * 100)
print("FINAL COMPARISON: FIVE DEEP LEARNING MODELS")
print("=" * 100)

# ------------------------------------------------------------
# 1. Model Performance Data
# ------------------------------------------------------------

final_comparison_data = [

    {
        "Model": "EfficientNet-B0",
        "Mean CV Accuracy": 92.71,
        "Test Accuracy": 93.22,
        "Macro Precision": 80.10,
        "Macro Recall": 77.98,
        "Macro F1": 78.98,
        "Weighted Precision": 93.12,
        "Weighted Recall": 93.22,
        "Weighted F1": 92.23
    },

    {
        "Model": "ResNet50",
        "Mean CV Accuracy": 88.58,
        "Test Accuracy": 90.96,
        "Macro Precision": 76.22,
        "Macro Recall": 77.63,
        "Macro F1": 75.40,
        "Weighted Precision": 89.92,
        "Weighted Recall": 90.96,
        "Weighted F1": 89.35
    },

    {
        "Model": "DenseNet121",
        "Mean CV Accuracy": 88.58,
        "Test Accuracy": 89.83,
        "Macro Precision": 82.88,
        "Macro Recall": 78.76,
        "Macro F1": 78.48,
        "Weighted Precision": 91.54,
        "Weighted Recall": 89.83,
        "Weighted F1": 89.17
    },

    {
        "Model": "ViT + LSTM",
        "Mean CV Accuracy": cv_mean,
        "Test Accuracy": vit_lstm_test_accuracy,
        "Macro Precision": vit_lstm_macro_precision,
        "Macro Recall": vit_lstm_macro_recall,
        "Macro F1": vit_lstm_macro_f1,
        "Weighted Precision": vit_lstm_weighted_precision,
        "Weighted Recall": vit_lstm_weighted_recall,
        "Weighted F1": vit_lstm_weighted_f1
    },

    {
        "Model": "Pure ViT-B/16",
        "Mean CV Accuracy": vit_cv_mean,
        "Test Accuracy": vit_test_accuracy,
        "Macro Precision": vit_macro_precision,
        "Macro Recall": vit_macro_recall,
        "Macro F1": vit_macro_f1,
        "Weighted Precision": vit_weighted_precision,
        "Weighted Recall": vit_weighted_recall,
        "Weighted F1": vit_weighted_f1
    }
]

final_comparison_df = pd.DataFrame(
    final_comparison_data
)

# ------------------------------------------------------------
# 2. Display Complete Comparison
# ------------------------------------------------------------

print("\nFINAL PERFORMANCE COMPARISON")
print("-" * 100)

print(
    final_comparison_df.to_string(
        index=False,
        formatters={
            "Mean CV Accuracy": "{:.2f}%".format,
            "Test Accuracy": "{:.2f}%".format,
            "Macro Precision": "{:.2f}%".format,
            "Macro Recall": "{:.2f}%".format,
            "Macro F1": "{:.2f}%".format,
            "Weighted Precision": "{:.2f}%".format,
            "Weighted Recall": "{:.2f}%".format,
            "Weighted F1": "{:.2f}%".format
        }
    )
)

# ------------------------------------------------------------
# 3. CV Accuracy Comparison
# ------------------------------------------------------------

plt.figure(
    figsize=(13, 7)
)

bars = plt.bar(
    final_comparison_df["Model"],
    final_comparison_df["Mean CV Accuracy"]
)

plt.xlabel(
    "Model",
    fontsize=12
)

plt.ylabel(
    "Mean CV Accuracy (%)",
    fontsize=12
)

plt.title(
    "Five-Model Comparison: Mean Cross-Validation Accuracy",
    fontsize=16,
    fontweight="bold"
)

plt.ylim(
    0,
    100
)

for bar, value in zip(
    bars,
    final_comparison_df["Mean CV Accuracy"]
):

    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 1,
        f"{value:.2f}%",
        ha="center",
        fontsize=10
    )

plt.grid(
    axis="y",
    alpha=0.3
)

plt.xticks(
    rotation=15
)

plt.tight_layout()

plt.savefig(
    "five_model_cv_accuracy_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 4. Test Accuracy Comparison
# ------------------------------------------------------------

plt.figure(
    figsize=(13, 7)
)

bars = plt.bar(
    final_comparison_df["Model"],
    final_comparison_df["Test Accuracy"]
)

plt.xlabel(
    "Model",
    fontsize=12
)

plt.ylabel(
    "Test Accuracy (%)",
    fontsize=12
)

plt.title(
    "Five-Model Comparison: Independent Test Accuracy",
    fontsize=16,
    fontweight="bold"
)

plt.ylim(
    0,
    100
)

for bar, value in zip(
    bars,
    final_comparison_df["Test Accuracy"]
):

    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 1,
        f"{value:.2f}%",
        ha="center",
        fontsize=10
    )

plt.grid(
    axis="y",
    alpha=0.3
)

plt.xticks(
    rotation=15
)

plt.tight_layout()

plt.savefig(
    "five_model_test_accuracy_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 5. Macro F1 Comparison
# ------------------------------------------------------------

plt.figure(
    figsize=(13, 7)
)

bars = plt.bar(
    final_comparison_df["Model"],
    final_comparison_df["Macro F1"]
)

plt.xlabel(
    "Model",
    fontsize=12
)

plt.ylabel(
    "Macro F1 (%)",
    fontsize=12
)

plt.title(
    "Five-Model Comparison: Macro F1 Score",
    fontsize=16,
    fontweight="bold"
)

plt.ylim(
    0,
    100
)

for bar, value in zip(
    bars,
    final_comparison_df["Macro F1"]
):

    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 1,
        f"{value:.2f}%",
        ha="center",
        fontsize=10
    )

plt.grid(
    axis="y",
    alpha=0.3
)

plt.xticks(
    rotation=15
)

plt.tight_layout()

plt.savefig(
    "five_model_macro_f1_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 6. Weighted F1 Comparison
# ------------------------------------------------------------

plt.figure(
    figsize=(13, 7)
)

bars = plt.bar(
    final_comparison_df["Model"],
    final_comparison_df["Weighted F1"]
)

plt.xlabel(
    "Model",
    fontsize=12
)

plt.ylabel(
    "Weighted F1 (%)",
    fontsize=12
)

plt.title(
    "Five-Model Comparison: Weighted F1 Score",
    fontsize=16,
    fontweight="bold"
)

plt.ylim(
    0,
    100
)

for bar, value in zip(
    bars,
    final_comparison_df["Weighted F1"]
):

    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 1,
        f"{value:.2f}%",
        ha="center",
        fontsize=10
    )

plt.grid(
    axis="y",
    alpha=0.3
)

plt.xticks(
    rotation=15
)

plt.tight_layout()

plt.savefig(
    "five_model_weighted_f1_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 7. Save Final Comparison Table
# ------------------------------------------------------------

final_comparison_df.to_csv(
    "five_model_final_comparison.csv",
    index=False
)

print("\n✓ Saved:")
print("  five_model_final_comparison.csv")
print("  five_model_cv_accuracy_comparison.png")
print("  five_model_test_accuracy_comparison.png")
print("  five_model_macro_f1_comparison.png")
print("  five_model_weighted_f1_comparison.png")

print("\n" + "=" * 100)
print("✓ FINAL FIVE-MODEL COMPARISON COMPLETE")
print("=" * 100)